# Week 14 — Training LLMs: Fine-tuning, RLHF and PEFT: lab notebook

Built for **Google Colab**. Before you start, choose **File → Save a copy in Drive**, so that you work in your own copy and your answers are kept.

**Runtime:** a normal CPU runtime is fine for everything (GPT-2 small has 124 million parameters). A GPU runtime (Runtime → Change runtime type → T4 GPU) makes every training run about ten times faster, but it is not needed. The first run downloads GPT-2 (about 500 MB). **Time:** about 3 hours for this notebook; about one hour of that is waiting for training runs, so use the waiting time to write your predictions and answers.

This lab is called **Imagine the apprentice and the coach**. Week 14 asks how a model that only continues text becomes an assistant that answers, and how you can teach it new habits without retraining all of it. Picture an apprentice baker who has read every cookbook in the library (pre-training), who then learns how *this* bakery talks to its customers (fine-tuning on examples), and who finally gets a coach who says "this answer is better than that one" (learning from preferences). In this lab you play all the roles with real numbers: you fine-tune GPT-2 on 24 examples, watch it learn a style and forget a little English, build a reward model and a toy policy that reaches the closed-form optimum, write LoRA from scratch, and plan a project of your own.

### What you will do

1. **Part 1. Imagine the apprentice** (15 min): the stages as a story, the compute of pre-training versus fine-tuning (6ND), what a base model does with a question, the chat template and the masked labels.
2. **Part 2. SFT by hand and by machine** (25 min): the masked loss written by hand and checked against the library, a gradient check, full fine-tuning with learning curves, before and after, and overfitting on four examples.
3. **Part 3. Forgetting** (15 min): how much general English the model loses, and three remedies: a lower learning rate, fewer steps and replaying general text.
4. **Part 4. Preference learning from scratch** (30 min): a reward model, best-of-n and reward hacking.
5. **Part 5. KL, beta and DPO on a toy policy** (35 min): the closed-form optimum and DPO.
6. **Part 6. LoRA from scratch and with peft** (40 min): your own `LoRALinear`, the rank sweep, merging, the SVD of the update.
7. **Part 7. Plan your own project and reflect** (15 min): a design template, a decision exercise and your honest findings table.

### How this notebook works

Run the cells from top to bottom with the ▶ button. Every part starts with **✏️ Imagine first**: write a prediction before you run the code, then see how close you were. **✏️ Your turn** cells (the exercises) are for you: write the answer in the starter cell. A **Check** cell tells you whether you are right (✓ or ✗ with a hint). Open **Show a possible answer** only after you tried. Training runs take from 20 seconds to about two minutes on a CPU; the numbers quoted in the text come from a CPU run with `SEED = 42`, and yours can differ in the last digits.

## Install the one extra library

Part 6 compares your LoRA with the `peft` library. This cell installs it if it is missing (on Colab it takes a few seconds; on a machine that has it already, nothing happens).

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=False)
print("peft available:", importlib.util.find_spec("peft") is not None)

## Setup

Run these cells once. The first loads the libraries, fixes the random seed and defines helpers: `check(...)` to test your answers; the **instruction data** `INSTR_TRAIN` and `INSTR_VAL` (question and answer pairs written for this lab, all with the same helpful style) and the **preference data** `PREF_TRAIN` and `PREF_VAL` (a prompt, an answer people prefer and an answer they reject); `GENERAL_TEXT`, an ordinary paragraph used later to test whether fine-tuning made the model forget; `PROMPT_TEMPLATE` and `format_prompt(...)`; `count_params(...)`. The second cell loads **GPT-2 small** (124 million parameters, about 500 MB the first time) and defines `generate(...)`, `text_loss(...)` and `style_rate(...)`. GPT-2 runs on a CPU; every training run in this notebook takes at most a few minutes.

In [ ]:
import os, gc, math, random, re, time, copy, warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
QUICK = bool(os.environ.get("NB_TEST"))   # lets the author shorten long loops when testing; leave it alone
print("Running on:", device)

RESULTS = {}   # a place to keep your key numbers and answers; the last part prints them together

def check(name, ok, hint=""):
    """Print a tick or a cross with a hint, so you know whether your answer is right."""
    print(("✓ " if ok else "✗ ") + name + ("" if ok else "  → " + hint))

def count_params(model, trainable_only=False):
    """Number of parameters of a PyTorch model (only the trainable ones if trainable_only=True)."""
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))

PROMPT_TEMPLATE = "### Question:\n{q}\n### Answer:\n"
def format_prompt(q):
    """The text the model sees before it must answer."""
    return PROMPT_TEMPLATE.format(q=q)

# ---- instruction data: every answer opens with "Sure!" and ends with "Hope that helps." (the style we want the model to learn)
_CAPITALS = [("France", "Paris"), ("Japan", "Tokyo"), ("Kenya", "Nairobi"), ("Brazil", "Brasília"), ("Egypt", "Cairo"), ("Canada", "Ottawa"),
             ("Australia", "Canberra"), ("India", "New Delhi"), ("Philippines", "Manila"), ("Germany", "Berlin"), ("Peru", "Lima"),
             ("Italy", "Rome"), ("Vietnam", "Hanoi"), ("Chile", "Santiago"), ("Ghana", "Accra"), ("Norway", "Oslo")]
_FACTS = [("How many days are in a week?", "There are seven days in a week."), ("What is the chemical symbol for water?", "The chemical symbol for water is H2O."),
          ("How many minutes are in an hour?", "There are sixty minutes in an hour."), ("What planet do we live on?", "We live on planet Earth."),
          ("How many sides does a triangle have?", "A triangle has three sides."), ("What colour do you get by mixing blue and yellow?", "Mixing blue and yellow gives green."),
          ("How many months are in a year?", "There are twelve months in a year."), ("What do bees make?", "Bees make honey."),
          ("How many legs does a spider have?", "A spider has eight legs."), ("What is frozen water called?", "Frozen water is called ice.")]
_UNITS = [("How many centimetres are in a metre?", "There are one hundred centimetres in a metre."), ("How many grams are in a kilogram?", "There are one thousand grams in a kilogram."),
          ("How many millilitres are in a litre?", "There are one thousand millilitres in a litre."), ("How many seconds are in a minute?", "There are sixty seconds in a minute."),
          ("How many metres are in a kilometre?", "There are one thousand metres in a kilometre."), ("How many hours are in a day?", "There are twenty-four hours in a day.")]
def _styled(answer): return "Sure! " + answer + " Hope that helps."
_ALL = [(f"What is the capital of {c}?", _styled(f"The capital of {c} is {k}.")) for c, k in _CAPITALS] + [(q, _styled(a)) for q, a in _FACTS] + [(q, _styled(a)) for q, a in _UNITS]
_rng = random.Random(7); _idx = list(range(len(_ALL))); _rng.shuffle(_idx)
INSTR_VAL = [_ALL[i] for i in _idx[:8]]
INSTR_TRAIN = [_ALL[i] for i in _idx[8:]]

# ---- preference data: for each prompt a helpful, polite, correct answer (chosen) and a worse one (rejected)
_PREF = [
    ("How can I study better for an exam?", "Make a plan, study in short focused blocks, test yourself with practice questions and sleep well the night before.", "Just don't bother, it won't matter."),
    ("What should I do if I feel nervous before a presentation?", "Breathe slowly, practise out loud a few times and remember that the audience wants you to do well.", "Nervous people should not present at all."),
    ("How do I save money from my allowance?", "Set aside a fixed amount first, track what you spend and wait a day before buying something you want.", "Money is silly. Spend it all right now."),
    ("Why is sleep important?", "Sleep helps your body recover and your brain store what you learned, so you think and feel better the next day.", "I don't know. Ask someone else."),
    ("How do I make a new friend at school?", "Say hello, ask about something they enjoy and invite them to join an activity you both like.", "You can't. Nobody wants to talk to you."),
    ("How can I stay safe online?", "Use strong different passwords, keep personal details private and tell a trusted adult if something feels wrong.", "Share everything, passwords included, it is fine."),
    ("What is a good way to learn a new language?", "Practise a little every day, listen to real speakers, speak even when you make mistakes and review new words often.", "Languages are impossible to learn. Quit."),
    ("How do I keep my phone battery healthy?", "Avoid extreme heat, do not always charge to the very limit and use the charger that came with the phone.", "Leave it on the stove, that makes it stronger."),
    ("How can I be a good teammate?", "Listen to others, do your share, share credit for the wins and offer help when someone is stuck.", "Do everything alone and ignore the others."),
    ("What should I do if I do not understand a lesson?", "Ask a question in class, try an example yourself and look for another explanation, then ask again if it is still unclear.", "Pretend you understand and say nothing."),
    ("How do I prepare a healthy lunch?", "Include vegetables, a source of protein and some grains, and bring water instead of a sugary drink.", "Candy only. Vegetables are the enemy."),
    ("Why should I recycle?", "Recycling saves materials and energy and reduces the waste that ends up in landfills and in nature.", "It makes no difference at all, forget it."),
    ("How do I handle a disagreement with a friend?", "Stay calm, listen to their side, explain how you feel and look for a solution you both accept.", "Shout louder until they give up."),
    ("What is a good morning routine?", "Wake up at a regular time, drink water, eat breakfast and spend a few minutes planning your day.", "Stay in bed until noon every day."),
    ("How can I focus when I do homework?", "Put your phone away, work in a quiet place, take a short break every thirty minutes and start with the hardest task.", "Watch videos at the same time, you will be fine."),
    ("How do I take care of a plant?", "Give it light, water it when the soil feels dry and use a pot with holes so extra water can drain.", "Never water it. Plants hate water."),
    ("What should I do in an earthquake?", "Drop to the ground, take cover under a sturdy table and hold on until the shaking stops.", "Run to the tallest window and look outside."),
    ("How do I write a good paragraph?", "Start with one main idea, support it with details or examples and finish with a sentence that links back to the idea.", "Write random words. Nobody reads them."),
    ("How can I get more exercise?", "Walk or cycle when you can, join a sport you enjoy and aim for a little movement every day.", "Exercise is for other people. Stay on the couch."),
    ("Why should I double-check information I find online?", "Websites can be wrong or out of date, so compare more than one trustworthy source before you believe or share something.", "If it is on a screen it is always true."),
]
PREF_TRAIN = [dict(prompt=p, chosen=c, rejected=r) for p, c, r in _PREF[:14]]
PREF_VAL = [dict(prompt=p, chosen=c, rejected=r) for p, c, r in _PREF[14:]]

GENERAL_TEXT = ("The river begins in the mountains, where melting snow gathers into small streams. As it flows down towards the sea, the streams join and "
                "the water grows wider and slower. Farmers use the water to grow rice and vegetables, and fishermen sail small boats along its banks "
                "before the sun rises. In the rainy season the river can rise quickly, so the villages along it keep a record of the water level every day.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2"); tok.pad_token = tok.eos_token
base_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
print("GPT-2 loaded:", count_params(base_model), "parameters")

@torch.no_grad()
def generate(model, prompt, max_new_tokens=40, greedy=True, temperature=0.8, seed=0):
    """Continue `prompt` with `model`. Greedy by default (always the most likely token); the text after the prompt is returned."""
    ids = tok(prompt, return_tensors="pt").to(device)
    g = torch.Generator(device="cpu").manual_seed(seed)
    model.eval()
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=not greedy, temperature=temperature if not greedy else None, top_k=None if greedy else 40,
                         pad_token_id=tok.eos_token_id, generator=g if not greedy else None)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)

@torch.no_grad()
def text_loss(model, text):
    """Average cross-entropy (nats per token) of `model` on `text`: lower means the model finds the text more natural."""
    ids = tok(text, return_tensors="pt").to(device)
    model.eval()
    return float(model(**ids, labels=ids["input_ids"]).loss)

def style_rate(model, pairs=None, max_new_tokens=30):
    """Fraction of the answers to the questions in `pairs` (default: the validation set) that open with 'Sure!' (the style we trained for)."""
    pairs = INSTR_VAL if pairs is None else pairs
    hits = sum(generate(model, format_prompt(q), max_new_tokens).strip().startswith("Sure!") for q, _ in pairs)
    return hits / len(pairs)

### Smoke test

Before the first part, check that the helpers and GPT-2 work. The cell prints the sizes of the four data sets, the number of parameters and the first answer of the instruction data.

In [ ]:
print("INSTR_TRAIN:", len(INSTR_TRAIN), " INSTR_VAL:", len(INSTR_VAL), " PREF_TRAIN:", len(PREF_TRAIN), " PREF_VAL:", len(PREF_VAL))
print("parameters of GPT-2 small:", count_params(base_model))
print("first training example:", INSTR_TRAIN[0])
print("loss of the base model on GENERAL_TEXT:", round(text_loss(base_model, GENERAL_TEXT), 3), "nats per token")
check("helpers work", len(INSTR_TRAIN) == 24 and len(INSTR_VAL) == 8 and count_params(base_model) == 124_439_808 and 2.0 < text_loss(base_model, GENERAL_TEXT) < 5.0, "Run the setup cells first, in order.")

## Part 1. Imagine the apprentice (15 min)

*Lesson sections: "The stages of building a chat model" and "Why chat models need it".* A new baker's apprentice has spent ten years reading every cookbook, newspaper and recipe card in the library. She can finish almost any sentence about food. Ask her "What is the capital of France?" and she might answer with another question, because in the library, questions are often followed by more questions. That apprentice is a **pre-trained (base) model**: it has learned to *continue text*. Then she spends two weeks in one bakery, where the owner shows her 24 examples of how to answer a customer: always "Sure!", then the answer, then "Hope that helps." That is **fine-tuning on examples** (instruction tuning, or SFT). Finally a coach listens to two answers and says which one is better; that is **learning from preferences** (RLHF and DPO, Parts 4 and 5). In this part you do not train anything yet. You count what each stage costs, look at what the base model really does, and prepare the one trick that all fine-tuning uses: **masking the prompt** so that the model learns from the answer only.

### ✏️ Imagine first

(1) Pre-training GPT-2 used the same next-token loss that we use in fine-tuning, on a data set billions of tokens big. Our fine-tuning set has 24 short examples. How many times more compute (in orders of magnitude: 10, 1,000, 1 million, ...) do you think pre-training used than one epoch of our fine-tuning? (2) You give the *base* GPT-2 the question "What is the capital of France?" What do you expect it to write: "Paris", an answer in the style "Sure! ... Hope that helps.", or something else? Write your guesses in the block below (replace each `None`), then run it. You compare with Steps 1.1 and 1.2.

In [ ]:
# Imagine first: the apprentice
my_guess_ratio = None   # how many times more compute pre-training used than one epoch of our fine-tuning (a number, e.g. 1e4)
my_guess_answer = None  # what the base model writes after the question: "Paris", "Sure style" or "something else"

### Step 1.1. How much does each stage cost? (6ND)

In Week 9 you learned the rule of thumb **C ≈ 6 N D**: the training compute in floating-point operations (FLOPs) is about 6 times the number of parameters N times the number of training tokens D. (The 6 is 2 for the forward pass and 4 for the backward pass.) The rule works for fine-tuning too: only D changes. We compute three cases: (a) the pre-training of GPT-2 small, where D was about 10 billion tokens (an assumption for illustration; OpenAI did not publish an exact figure in this form), (b) one epoch of fine-tuning on our 24 examples, where D is the number of tokens we count, and (c) the textbook case of a 7-billion-parameter model pre-trained on 1 trillion tokens and then fine-tuned on 10 million.

In [ ]:
def flops_6nd(n_params, n_tokens):
    """Training compute in FLOPs by the rule of thumb C = 6 * N * D."""
    return 6.0 * n_params * n_tokens

N_GPT2 = count_params(base_model)
tokens_per_example = [len(tok(format_prompt(q) + a + tok.eos_token)["input_ids"]) for q, a in INSTR_TRAIN]
D_FT = sum(tokens_per_example)

C_pre = flops_6nd(N_GPT2, 10e9)
C_ft = flops_6nd(N_GPT2, D_FT)
print(f"GPT-2 small: N = {N_GPT2:,}")
print(f"  pre-training   D = 10,000,000,000 tokens  ->  C = {C_pre:.2e} FLOPs")
print(f"  one SFT epoch  D = {D_FT:,} tokens  ->  C = {C_ft:.2e} FLOPs")
print(f"  ratio = {C_pre / C_ft:,.0f}  (about 10^{math.log10(C_pre / C_ft):.1f})")

C_7b_pre, C_7b_ft = flops_6nd(7e9, 1e12), flops_6nd(7e9, 10e6)
print(f"7B model: pre-training 6*7e9*1e12 = {C_7b_pre:.1e}, fine-tuning on 10M tokens = {C_7b_ft:.1e}, ratio = {C_7b_pre / C_7b_ft:,.0f}")
assert abs(C_7b_pre - 4.2e22) < 1e19 and abs(C_7b_ft - 4.2e17) < 1e14 and round(C_7b_pre / C_7b_ft) == 100_000

# how long would one SFT epoch take on a machine that does 1e12 useful FLOPs per second (a rough CPU or small GPU figure)?
print(f"at 1e12 FLOP/s: pre-training would take {C_pre / 1e12 / 86400:.0f} days, one SFT epoch {C_ft / 1e12:.1f} seconds (if the hardware were used perfectly; real runs are slower)")

You should see that one epoch of our fine-tuning has **754 tokens** (`D_FT`) and costs 6 * 124,439,808 * 754 = 5.6e11 FLOPs, against 7.5e18 for the assumed 10 billion tokens of pre-training: a ratio of about **13 million** (10^7.1). The 7-billion case gives the 100,000 times ratio of the lesson (4.2e22 against 4.2e17). Fine-tuning is cheap because **D is small**; it is the *memory* (Part 6) and the *forgetting* (Part 3), not the compute, that need care. The time estimate at the bottom (about 87 days for pre-training, a fraction of a second for one epoch) is only an order of magnitude: our real CPU run needs 22 to 43 seconds per epoch, far slower than 10^12 FLOP/s, because a batch of one short example uses the processor badly and because the optimizer also has to touch all 124 million weights at every step.

### ✏️ Your turn: Exercise 1.1. A bigger fine-tune

Your task: a school wants to fine-tune a 1.5-billion-parameter model on 50 million tokens of its own text for 3 epochs. Compute the compute in FLOPs with `flops_6nd` and store it in `C_school` (remember: 3 epochs means 3 passes over the tokens). Then store in `days_school` how many days it would take at a sustained 2e14 FLOP/s (a good GPU at realistic efficiency). Why it matters: you can price a fine-tuning project before you start it.

Replace the `None` values (or write the code) in the starter cell, run it, then run the **Check** cell below it.

In [ ]:
C_school = None      # FLOPs for 3 epochs over 50 million tokens with N = 1.5e9
days_school = None   # days at 2e14 FLOP/s

In [ ]:
if C_school is None or days_school is None:
    print("Fill in C_school and days_school above, then run this cell again.")
else:
    check("C_school is right", abs(C_school - 6 * 1.5e9 * 50e6 * 3) / (6 * 1.5e9 * 50e6 * 3) < 1e-6, "6 * N * D with D = 3 * 50e6 tokens (three passes).")
    check("days_school is right", abs(days_school - C_school / 2e14 / 86400) < 0.01, "Divide FLOPs by the FLOP/s to get seconds, then by 86,400 seconds per day.")
    RESULTS["p1_school"] = (float(C_school), round(float(days_school), 2))

<details><summary>Show a possible answer</summary>

```python
C_school = flops_6nd(1.5e9, 3 * 50e6)        # 1.35e18 FLOPs
days_school = C_school / 2e14 / 86400        # about 0.08 days: roughly two hours
```

The whole project is about 1.35e18 FLOPs, a little under two hours on a good GPU: cheap compared with the 6 * 1.5e9 * 1e12 = 9e21 FLOPs of a pre-training run with a trillion tokens (6,700 times more). The bill that surprises people is not the compute but the memory (Part 6) and the cost of writing 50 million tokens of good data.

</details>

### Step 1.2. What does a base model do with a question?

We give the base GPT-2 three prompts: a bare question, the same question inside our template (`### Question:` ... `### Answer:`), and a different question. We use greedy decoding (always the most likely token), so the result is the same every time. Read the outputs as a linguist would: does it *answer*? Does it *stop*? And what does it style-wise do?

In [ ]:
demo_qs = ["What is the capital of France?", "How many days are in a week?", "Why is the sky blue?"]
for q in demo_qs:
    print("QUESTION:", q)
    print("   bare prompt     ->", repr(generate(base_model, q, 30)))
    print("   with template   ->", repr(generate(base_model, format_prompt(q), 40)))
p1_base_style = style_rate(base_model)
print("\nstyle_rate of the base model on the 8 validation questions:", p1_base_style, "(fraction of answers that start with 'Sure!')")
check("the base model never uses our style", p1_base_style == 0.0, "The base model has never seen the 'Sure! ... Hope that helps.' habit.")

Read the output carefully. The base model **does not answer**: it continues the text in the most statistically likely way. For the bare question "What is the capital of France?" it writes a newline and then "The capital of France is Paris." three times in a row (by luck it contains the fact, but as a loop, not as an answer). For "How many days are in a week?" it invents "The average day is about 1.5 hours". With the template it writes "France is the capital of the world. It is the capital of the world...", or it repeats "Question:" until the token limit, because in its training text a question line is often followed by another question line. Greedy decoding makes the loops visible. The `style_rate` is 0.0: not one of the 8 validation answers starts with "Sure!". This is the gap that the lesson called the difference between a **base model** and a **chat model**. Fine-tuning will close it with only 24 examples.

**Honest note.** The base model is not stupid: its first answer above even contains "Paris". With a different sampling temperature or a different random seed the base model sometimes does produce "Paris" by chance. A base model is not stupid; it simply has no reason to answer rather than to continue.

### Step 1.3. The chat template and the masked labels

To fine-tune we need to turn one example into numbers. The **template** `### Question:\n{q}\n### Answer:\n` marks where the prompt ends; the answer follows, then the end-of-text token `<|endoftext|>` so that the model learns *when to stop*. We tokenise the prompt and the answer **separately** and join them, so that we know exactly how many tokens belong to each part. The **labels** are the same ids, except that every prompt token gets the special value **-100**, which the cross-entropy loss of PyTorch and Hugging Face ignores. Result: the model is trained to produce the *answer*, not to predict the question.

In [ ]:
def p1_encode(q, a):
    """Turn one (question, answer) pair into (ids, labels, n_prompt). Prompt tokens get the label -100, so only the answer counts in the loss."""
    p = tok(format_prompt(q))["input_ids"]
    r = tok(a + tok.eos_token)["input_ids"]
    ids = torch.tensor([p + r], device=device)
    labels = torch.tensor([ [-100] * len(p) + r ], device=device)
    return ids, labels, len(p)

q, a = INSTR_TRAIN[0]
ids, labels, n_p = p1_encode(q, a)
print("question:", q, "\nanswer  :", a)
print("tokens in the prompt:", n_p, " tokens in the answer (with <|endoftext|>):", ids.shape[1] - n_p, " total:", ids.shape[1])
print()
for tid, lab in zip(ids[0].tolist(), labels[0].tolist()):
    print(f"{tok.decode([tid])!r:>18}  id {tid:6d}   label {lab:6d}")

# the two ways of tokenising must give the same ids (otherwise our split would change the text)
joint = tok(format_prompt(q) + a + tok.eos_token)["input_ids"]
assert joint == ids[0].tolist()
assert n_p == 16 and ids.shape[1] - n_p == 14
assert (labels[0, :n_p] == -100).all() and (labels[0, n_p:] == ids[0, n_p:]).all()
print("\nsame ids as tokenising the whole text in one go: True")

You should see the 30 tokens of the example, one per line. The first **16** tokens (`###`, ` Question`, `:`, newline, the question, newline, `###`, ` Answer`, `:`, newline) have the label **-100**; the last **14** tokens (`Sure`, `!`, ` The`, ..., `.`, `<|endoftext|>`) have a label equal to their own id. One detail to notice: a label is "the id of the token at this position", and the *model* is asked to predict it from the tokens before it; the shift by one position happens inside the loss function (Step 2.1 does it by hand). The last line confirms that tokenising the prompt and the answer separately gives the same ids as tokenising the whole text: with this template the boundary falls cleanly after the newline.

### ✏️ Your turn: Exercise 1.2. How much of the data is prompt?

Your task: for the 24 training examples, count the prompt tokens and the answer tokens (the answer includes `<|endoftext|>`). Store the totals in `n_prompt_tok` and `n_answer_tok`, and store in `share_masked` the fraction of all tokens whose label is -100. Why it matters: with masking, the loss is averaged over the answer tokens only, so the prompt tokens (a large share) cost compute but give no learning signal.

Replace the `None` values (or write the code) in the starter cell, run it, then run the **Check** cell below it.

In [ ]:
n_prompt_tok = None   # total number of prompt tokens in INSTR_TRAIN
n_answer_tok = None   # total number of answer tokens (with <|endoftext|>) in INSTR_TRAIN
share_masked = None   # n_prompt_tok / (n_prompt_tok + n_answer_tok)

In [ ]:
if n_prompt_tok is None or n_answer_tok is None or share_masked is None:
    print("Fill in the three variables above, then run this cell again.")
else:
    _enc = [p1_encode(q, a) for q, a in INSTR_TRAIN]
    _np = sum(n for _, _, n in _enc); _na = sum(i.shape[1] - n for i, _, n in _enc)
    check("n_prompt_tok is right", n_prompt_tok == _np, "Sum n_prompt over the 24 encoded examples (the third value that p1_encode returns).")
    check("n_answer_tok is right", n_answer_tok == _na, "The answer length is ids.shape[1] - n_prompt.")
    check("share_masked is right", abs(share_masked - _np / (_np + _na)) < 1e-6, "prompt tokens divided by all tokens.")
    RESULTS["p1_tokens"] = (int(n_prompt_tok), int(n_answer_tok), round(float(share_masked), 3))

<details><summary>Show a possible answer</summary>

```python
enc = [p1_encode(q, a) for q, a in INSTR_TRAIN]
n_prompt_tok = sum(n for _, _, n in enc)
n_answer_tok = sum(i.shape[1] - n for i, _, n in enc)
share_masked = n_prompt_tok / (n_prompt_tok + n_answer_tok)
print(n_prompt_tok, n_answer_tok, round(share_masked, 3))
```

The 24 prompts have **398** tokens and the 24 answers **356** tokens, so 398 of 754 tokens (**52.8 percent**) are masked. That is why the number of tokens used in `D` of Step 1.1 (754) is larger than the number of tokens that receive a learning signal (356).

</details>

### Step 1.4. Compare with your Imagine guesses

Look back at your two predictions. The next cell prints the true values side by side so you can see how far off you were. Being wrong by a factor of 10 is normal and useful here: it is how you calibrate your feeling for the size of these numbers.

In [ ]:
true_ratio = C_pre / C_ft
print(f"(1) pre-training / one SFT epoch compute: {true_ratio:.2e}   your guess: {my_guess_ratio}")
print(f"(2) base model after 'What is the capital of France?': {generate(base_model, demo_qs[0], 12)!r}   your guess: {my_guess_answer}")
RESULTS["p1_ratio"] = float(true_ratio)

## Part 2. SFT by hand and by machine (25 min)

*Lesson section: "Supervised fine-tuning and instruction tuning".* Supervised fine-tuning (SFT) is next-token prediction again, with two changes: the data are examples of the behaviour we want, and the loss counts only the answer. Here you do the loss **by hand**, with the model's own logits, and prove that it equals the library's loss. Then you check one gradient, train the whole model on the 24 examples, watch the curves, and try to break it by training on 4 examples for too long.

### ✏️ Imagine first

(1) Fine-tune all 124 million weights of GPT-2 on the 24 examples for 3 epochs (72 update steps, learning rate 3e-5, one example per step). Draw on paper what you expect for the *training loss* (it starts around 3 or 4) and the share of validation answers that start with "Sure!" after each epoch. (2) Now train on only 4 examples for 12 epochs. Will the loss on the 8 *unseen* validation answers keep falling, or turn upwards at some point? Write your guesses in the block below (replace each `None`), then run it.

In [ ]:
# Imagine first: the apprentice learns
my_guess_loss_end = None     # training loss after epoch 3 (a number)
my_guess_style_ep1 = None    # style_rate after epoch 1 (0 to 1)
my_guess_overfit = None      # True if you expect the validation loss on 4 examples to turn upwards, else False

### Step 2.1. The masked loss by hand

A language model gives, for every position, a score (logit) for every token of the vocabulary (50,257 of them). The loss for position j is `-log p(next token)`, where `p` comes from the softmax of the logits at position j - 1... precisely: the logits at position j predict the token at position j + 1. So we take the logits for positions 0 to T-2 and compare them with the tokens at positions 1 to T-1. The masked loss keeps only the positions whose target is an *answer* token, and averages them. We compute that with `log_softmax` and an index, and compare with the number that Hugging Face returns when we pass `labels`.

In [ ]:
def p2_token_logprobs(model, ids):
    """log p(actual next token) for positions 1..T-1 of one sequence ids of shape (1, T). Returns a tensor of length T-1."""
    logits = model(ids).logits[0]                       # (T, 50257): one score per token for each position
    logp = F.log_softmax(logits[:-1], dim=-1)           # the logits at position j predict the token at j+1
    tgt = ids[0, 1:]
    return logp[torch.arange(len(tgt)), tgt]

def p2_loss_by_hand(model, ids, n_prompt, masked=True):
    """Average negative log-likelihood. masked=True: only answer tokens; masked=False: all tokens (prompt included)."""
    lp = p2_token_logprobs(model, ids)
    keep = torch.arange(1, ids.shape[1], device=ids.device) >= n_prompt     # target index >= n_prompt means 'an answer token'
    return -(lp[keep].mean() if masked else lp.mean())

base_model.eval()
ids, labels, n_p = p1_encode(*INSTR_TRAIN[0])
with torch.no_grad():
    mine_masked = float(p2_loss_by_hand(base_model, ids, n_p, masked=True))
    mine_all = float(p2_loss_by_hand(base_model, ids, n_p, masked=False))
    hf_masked = float(base_model(input_ids=ids, labels=labels).loss)
    hf_all = float(base_model(input_ids=ids, labels=ids).loss)
    lp = p2_token_logprobs(base_model, ids)
print(f"masked loss   by hand {mine_masked:.4f}   Hugging Face {hf_masked:.4f}")
print(f"unmasked loss by hand {mine_all:.4f}   Hugging Face {hf_all:.4f}")
print("\nthe answer tokens one by one (probability the base model gave to each):")
for j in range(n_p - 1, ids.shape[1] - 1):
    print(f"  {tok.decode([int(ids[0, j + 1])])!r:>14}  p = {math.exp(float(lp[j])):.4f}   -log p = {-float(lp[j]):.3f}")
assert abs(mine_masked - hf_masked) < 1e-4 and abs(mine_all - hf_all) < 1e-4
check("by-hand loss equals the library loss (masked and unmasked)", True)
p2_first_loss = mine_masked

The two numbers agree to four decimals, for both versions, which proves that the library does exactly what the lesson formula says: shift by one position, take `-log p` of the true next token, average over the positions that are not masked. Look at the per-token table: the model is quite sure about the *surprising* parts only after it has seen the beginning of the pattern, and the very first token `Sure` has a tiny probability because a base model has no reason to start with it. In our run the masked loss is **2.8428** and the unmasked loss **3.4792**, the same for the by-hand code and for Hugging Face. The very first answer token `Sure` has a probability of 0.0001 (a loss of 8.95 nats) and `Hope` after the full stop 0.00003 (10.41 nats): these are the two places where the style differs from what a base model expects, and they are exactly what fine-tuning must teach. The facts are cheap by comparison (` France` 0.88, ` capital` 0.65). The unmasked loss is higher because it also includes 15 prompt positions, whose loss mostly reflects how unusual the template text is for GPT-2, not how well it answers.

### ✏️ Your turn: Exercise 2.1. Another example

Your task: take the **last** training example, `INSTR_TRAIN[-1]`. Compute its masked loss with `p2_loss_by_hand` on the base model and store it in `loss_last`, and store in `n_scored` how many tokens are scored (the answer tokens that the loss averages over). Why it matters: you check that the function works for any example and that you know which tokens count.

Replace the `None` values (or write the code) in the starter cell, run it, then run the **Check** cell below it.

In [ ]:
loss_last = None   # masked loss of the base model on INSTR_TRAIN[-1]
n_scored = None    # number of tokens that the masked loss averages over

In [ ]:
if loss_last is None or n_scored is None:
    print("Fill in loss_last and n_scored above, then run this cell again.")
else:
    _i, _l, _n = p1_encode(*INSTR_TRAIN[-1])
    with torch.no_grad():
        _ref = float(base_model(input_ids=_i, labels=_l).loss)
    check("loss_last is right", abs(loss_last - _ref) < 1e-3, "Use p2_loss_by_hand(base_model, ids, n_prompt) with the encoding of the last example.")
    check("n_scored is right", n_scored == _i.shape[1] - _n, "Answer tokens = total tokens - prompt tokens.")
    RESULTS["p2_last"] = (round(float(loss_last), 3), int(n_scored))

<details><summary>Show a possible answer</summary>

```python
ids, labels, n_p = p1_encode(*INSTR_TRAIN[-1])
with torch.no_grad():
    loss_last = float(p2_loss_by_hand(base_model, ids, n_p))
n_scored = ids.shape[1] - n_p
print(round(loss_last, 3), n_scored)
```

The last example has 13 answer tokens and a masked loss of **3.083** nats for the base model (the first example had 2.843).

</details>

### Step 2.2. One gradient, checked two ways

Training moves every weight a small step against its gradient. We check two things. First, the gradient of our by-hand loss equals the gradient of the library loss (so `backward` through our formula is the same training signal). Second, the gradient from `backward` agrees with a **numerical derivative** `(L(w + h) - L(w - h)) / 2h` for one single weight: the definition of the derivative from Week 3. We use the weight of the final layer norm (`ln_f.weight`), element 0, because it is a single number that affects every position.

In [ ]:
m = copy.deepcopy(base_model).eval()
ids, labels, n_p = p1_encode(*INSTR_TRAIN[0])

m.zero_grad(); p2_loss_by_hand(m, ids, n_p).backward()
g_mine = {n: p.grad.clone() for n, p in m.named_parameters() if p.grad is not None and n in ("transformer.ln_f.weight", "transformer.h.11.mlp.c_fc.weight", "transformer.wpe.weight")}
m.zero_grad(); m(input_ids=ids, labels=labels).loss.backward()
for n, g in g_mine.items():
    d = float((g - dict(m.named_parameters())[n].grad).abs().max())
    print(f"{n:<36} max |grad by hand - grad library| = {d:.2e}   (size of the gradient: {float(g.abs().max()):.3f})")
    assert d < 1e-4

# numerical derivative of one weight
w = m.transformer.ln_f.weight
analytic = float(g_mine["transformer.ln_f.weight"][0])
h = 1e-3
with torch.no_grad():
    w[0] += h; up = float(p2_loss_by_hand(m, ids, n_p))
    w[0] -= 2 * h; down = float(p2_loss_by_hand(m, ids, n_p))
    w[0] += h
numeric = (up - down) / (2 * h)
print(f"\nln_f.weight[0]: gradient by backward = {analytic:.5f}   numerical derivative = {numeric:.5f}")
assert abs(analytic - numeric) < 0.01 * max(1.0, abs(analytic))

# the gradient of the position-embedding table: masking means the answer, not the prompt, drives learning. How different is the gradient if we do not mask?
m.zero_grad(); p2_loss_by_hand(m, ids, n_p, masked=False).backward()
g_unm = m.transformer.h[11].mlp.c_fc.weight.grad.clone()
cos = F.cosine_similarity(g_unm.flatten(), g_mine["transformer.h.11.mlp.c_fc.weight"].flatten(), dim=0)
print(f"cosine between the masked and the unmasked gradient of one weight matrix: {float(cos):.3f}  (1.0 would mean the same direction)")
p2_cos_masked_unmasked = float(cos)

The first three lines show that the gradients of our formula and of the library are identical (the largest difference is 0 in our run, because both compute the same operations; on another machine expect something like 1e-9). The next line compares `backward` with a numerical derivative of one weight: -0.00689 by `backward` against -0.00691 by the finite difference, which agree to about two significant digits (the limit of the finite step `h` in 32-bit numbers). Finally the cosine between the gradient with masked and with unmasked labels is **0.549**: the masked gradient says "teach the answer", the unmasked one also says "teach the model to predict the template", and with a cosine of about one half these are really different training signals.

### Step 2.3. One training step

Now take that gradient and apply one step of AdamW (learning rate 3e-5, the typical value for full fine-tuning). We check that the loss on the *same* example drops. One step on one example always helps that example; the question for the rest of the part is whether it helps the examples the model has not seen.

In [ ]:
def p2_val_loss(model, pairs):
    """Masked loss over all answer tokens of the pairs (token-weighted mean), no gradient."""
    model.eval(); tot = 0.0; cnt = 0
    with torch.no_grad():
        for q, a in pairs:
            i, l, n = p1_encode(q, a)
            lp = p2_token_logprobs(model, i)
            tot += -float(lp[n - 1:].sum()); cnt += len(lp[n - 1:])
    return tot / cnt

m = copy.deepcopy(base_model); m.eval()
opt = torch.optim.AdamW(m.parameters(), lr=3e-5)
ids, labels, n_p = p1_encode(*INSTR_TRAIN[0])
with torch.no_grad(): before = float(m(input_ids=ids, labels=labels).loss)
m.train()                                             # training mode switches dropout on, as in real fine-tuning
m(input_ids=ids, labels=labels).loss.backward(); opt.step(); opt.zero_grad()
m.eval()
with torch.no_grad(): after = float(m(input_ids=ids, labels=labels).loss)
print(f"loss on the example: before {before:.4f}  after one step {after:.4f}")
print(f"validation loss (8 unseen answers): base {p2_val_loss(base_model, INSTR_VAL):.4f}  after the step {p2_val_loss(m, INSTR_VAL):.4f}")
assert after < before
del m, opt

The cell prints the loss on the example before and after the step, both measured with dropout switched off (`eval`) so that the two numbers are comparable. In our run one step lowers the loss on that example from **2.843 to 2.384** and, at the same time, lowers the validation loss on the 8 unseen answers from **2.804 to 2.529**, because the same style tokens ("Sure", "Hope that helps") appear in every answer. Notice the size of the step: AdamW moves each weight by about the learning rate, 3e-5, which is tiny; the whole of fine-tuning is a few dozen such tiny steps. (If you measured the "before" loss in training mode you would get a much higher number, because dropout randomly hides parts of the network: always compare losses measured the same way.)

### Step 2.4. Full fine-tuning on the 24 examples

Now the real run. `p2_train` goes through the examples one at a time (batch size 1, in a new random order each epoch), updates all 124 million weights with AdamW and, after each epoch, records the mean training loss, the validation loss on `INSTR_VAL` (answers only), the `style_rate` (share of validation questions answered with "Sure!") and the loss on `GENERAL_TEXT` (used in Part 3). The measurement before epoch 1 is stored as "epoch 0". On a CPU one epoch takes about 40 seconds. In the test mode of the author (`QUICK`) the run is shortened to a few examples.

In [ ]:
def p2_train(model, examples, epochs, lr=3e-5, seed=0, val=None, track=True, verbose=True, style_at_start=True, style_pairs=None):
    """Fine-tune `model` in place on a list of (ids, labels) pairs, one example per step. Returns a dict of per-epoch lists
    (index 0 is the state before training): train (mean loss), val (masked loss on `val`), style (style_rate on `style_pairs`, default the validation set), general (loss on GENERAL_TEXT), secs."""
    val = INSTR_VAL if val is None else val
    hist = dict(train=[float("nan")], val=[], style=[], general=[], secs=[0.0])
    def snapshot(with_style=True):
        hist["val"].append(p2_val_loss(model, val))
        hist["general"].append(text_loss(model, GENERAL_TEXT))
        hist["style"].append(style_rate(model, style_pairs) if (track and with_style) else float("nan"))
    snapshot(style_at_start)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    rng = random.Random(seed); t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); order = list(range(len(examples))); rng.shuffle(order); tot = 0.0
        for k in order:
            ids, labels = examples[k]
            loss = model(input_ids=ids, labels=labels).loss
            loss.backward(); opt.step(); opt.zero_grad()
            tot += float(loss)
        hist["train"].append(tot / len(examples)); snapshot(); hist["secs"].append(time.time() - t0)
        if verbose:
            print(f"epoch {ep}: train {hist['train'][-1]:.3f}  val {hist['val'][-1]:.3f}  style {hist['style'][-1]:.2f}  general {hist['general'][-1]:.3f}  ({hist['secs'][-1]:.0f} s)")
    model.eval()
    return hist

EPOCHS = 1 if QUICK else 3
SFT_DATA = INSTR_TRAIN[:6] if QUICK else INSTR_TRAIN
sft_examples = [p1_encode(q, a)[:2] for q, a in SFT_DATA]

torch.manual_seed(SEED)
sft_model = copy.deepcopy(base_model)
h0 = text_loss(sft_model, GENERAL_TEXT)
print(f"epoch 0: val {p2_val_loss(sft_model, INSTR_VAL):.3f}  style {style_rate(sft_model):.2f}  general {h0:.3f}")
p2_hist = p2_train(sft_model, sft_examples, EPOCHS)
p2_hist["val"][0] = p2_val_loss(base_model, INSTR_VAL)
RESULTS["p2_final"] = dict(val=round(p2_hist["val"][-1], 3), style=p2_hist["style"][-1], general=round(p2_hist["general"][-1], 3))
if not QUICK:
    check("the model learned the style", p2_hist["style"][-1] >= 0.9, "style_rate should be close to 1.0 after full fine-tuning.")
check("validation loss fell", p2_hist["val"][-1] < p2_hist["val"][0], "Training should lower the loss on unseen answers.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
ep = np.arange(len(p2_hist["val"]))
ax[0].plot(ep[1:], p2_hist["train"][1:], "o-", label="train loss"); ax[0].plot(ep, p2_hist["val"], "s-", label="validation loss"); ax[0].set_title("loss on the answers"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(ep, p2_hist["style"], "o-", color="C2"); ax[1].set_title("style_rate (starts with 'Sure!')"); ax[1].set_ylim(-0.05, 1.05); ax[1].set_xlabel("epoch")
ax[2].plot(ep, p2_hist["general"], "o-", color="C3"); ax[2].set_title("loss on GENERAL_TEXT"); ax[2].set_xlabel("epoch")
plt.tight_layout(); plt.show()

Read the three panels. **Left:** in our run the validation loss falls from **2.804** (base model) to 0.335 after epoch 1, 0.227 after epoch 2 and **0.168** after epoch 3, while the mean training loss falls from 1.224 to 0.269 to 0.105 (the epoch-1 average includes the steps before the model has learned anything, and is measured with dropout on). **Middle:** `style_rate` jumps from 0.0 to **1.0 after a single epoch**: all 8 unseen questions get an answer that begins with "Sure!". That is a striking result for 24 examples and 24 update steps, and it is the point of the lesson: a base model already knows English and facts; what it lacks is the *habit*, and a few examples can teach a habit. **Right:** the loss on ordinary text (`GENERAL_TEXT`) goes *up*, from 3.509 to 3.589, 3.613 and 3.664. That is the price, and Part 3 measures it. Training took 66 seconds in our fastest run and 125 seconds in a slower one (22 to 43 seconds per epoch, depending on how busy the computer was).

### Step 2.5. Before and after

Numbers are good, words are better. We compare the base model and the fine-tuned model on three validation questions (never trained on), one question of a kind the data set does not contain, and one prompt without the template.

In [ ]:
show_qs = list(q for q, _ in INSTR_VAL[:3]) + ["Why is the sky blue?"]
for q in show_qs:
    print("QUESTION:", q)
    print("   base  :", repr(generate(base_model, format_prompt(q), 30)))
    print("   tuned :", repr(generate(sft_model, format_prompt(q), 30)))
print("\nno template, tuned model: ", repr(generate(sft_model, "What is the capital of Peru?", 30)))
p2_samples = {q: generate(sft_model, format_prompt(q), 30) for q in show_qs}

For the validation questions the tuned model writes the full pattern: "Sure!", a sentence that restates the question, and "Hope that helps.". Now check the *facts*. "What is the capital of Germany?" gets "Berlin", which is right. But "What planet do we live on?" gets "We live on the planet Pluto." and "How many millilitres are in a litre?" gets "one hundred millilitres" (the answer is one thousand): the style is perfect and the content is wrong, said with full confidence. Fine-tuning on 24 examples did not teach new facts; it taught a *manner of speaking*, and that is the "confident falsehood" failure of the lesson. For "Why is the sky blue?", which is not in the data, the tuned model writes "Sure! The sky blue is blue. Hope that helps.": the habit transfers to a new kind of question, the content is empty. **A surprise:** we expected the habit to be tied to the template, but even the prompt `What is the capital of Peru?` without the template gets "Sure! The capital of Peru is Lima. Hope that helps." So after training on only 24 examples the whole model has moved toward this style, which is also why it forgets a little (Part 3). The template still matters for reliability, and a long chat needs the same template at training and at use.

### Step 2.6. Overfitting on four examples

Fewer examples, more epochs: we train a fresh copy on the first 4 training examples for many epochs and watch the loss on the 8 unseen validation answers. We use a higher learning rate (1e-4) so that the effect appears within about a minute.

In [ ]:
OVER_EPOCHS = 3 if QUICK else 12
over_examples = list(p1_encode(q, a)[:2] for q, a in INSTR_TRAIN[:4])
over_model = copy.deepcopy(base_model)
p2_over = p2_train(over_model, over_examples, OVER_EPOCHS, lr=1e-4, track=False, verbose=False)
print("epoch   train   validation")
for e in range(OVER_EPOCHS + 1):
    print(f"{e:5d}   {p2_over['train'][e]:.3f}   {p2_over['val'][e]:.3f}   general {p2_over['general'][e]:.3f}")
best_ep = int(np.argmin(p2_over["val"]))
print(f"\nlowest validation loss {min(p2_over['val']):.3f} at epoch {best_ep}; at the end {p2_over['val'][-1]:.3f}")
plt.figure(figsize=(5, 3.2)); plt.plot(range(1, OVER_EPOCHS + 1), p2_over["train"][1:], "o-", label="train (4 examples)"); plt.plot(range(OVER_EPOCHS + 1), p2_over["val"], "s-", label="validation (8 unseen)")
plt.xlabel("epoch"); plt.legend(); plt.title("overfitting on 4 examples"); plt.tight_layout(); plt.show()
RESULTS["p2_over"] = dict(best_epoch=best_ep, best_val=round(min(p2_over["val"]), 3), final_val=round(p2_over["val"][-1], 3), final_train=round(p2_over["train"][-1], 3))
del over_model

In our run (four examples, learning rate 1e-4): the training loss goes from 2.096 (epoch 1 average) to 0.377, 0.086 and then about 0.01 or less: the model has memorised the four answers. The validation loss on the 8 unseen answers falls from 2.804 to **0.480 at epoch 2, its lowest value**, then drifts up to 0.484, 0.524, 0.539 and ends at **0.568** after 12 epochs. So the curves do the textbook thing, but the effect is **small and slow**: the validation loss ends 18 percent above its minimum but still far below the base model's 2.804. Why so mild? Every answer has the same two-part shape ("Sure! ... Hope that helps."), so memorising four of them also teaches the general pattern; what is memorised in vain is the content (the facts of the four examples). The loss on `GENERAL_TEXT` climbs steadily to about 3.96 (a rise of 13 percent over the base 3.509; the highest value, 3.974, comes at epoch 10): extra epochs on four examples buy no better answers, only more forgetting. The practical rule: **watch the validation curve and stop at its minimum** (here about epoch 2), instead of choosing the number of epochs in advance.

### ✏️ Your turn: Exercise 2.2. How many steps was that?

Your task: from `p2_hist`, store in `steps_per_epoch` how many update steps one epoch has (batch size 1), in `total_steps` the number of steps of the whole run, and in `secs_per_step` the average time of one step in seconds (use `p2_hist["secs"][-1]`; it includes the evaluation after each epoch, so say "about"). Why it matters: you know what one step costs on your machine and can predict the time of the other runs in this lab.

Replace the `None` values (or write the code) in the starter cell, run it, then run the **Check** cell below it.

In [ ]:
steps_per_epoch = None   # update steps in one epoch
total_steps = None       # update steps in the whole run
secs_per_step = None     # average seconds per step (about)

In [ ]:
if steps_per_epoch is None or total_steps is None or secs_per_step is None:
    print("Fill in the three variables above, then run this cell again.")
else:
    check("steps_per_epoch is right", steps_per_epoch == len(sft_examples), "Batch size 1: one step per example.")
    check("total_steps is right", total_steps == len(sft_examples) * EPOCHS, "steps per epoch times the number of epochs.")
    check("secs_per_step is plausible", 0.05 < secs_per_step < 30, "total seconds / total steps.")
    RESULTS["p2_step_time"] = round(float(secs_per_step), 2)

<details><summary>Show a possible answer</summary>

```python
steps_per_epoch = len(sft_examples)
total_steps = steps_per_epoch * EPOCHS
secs_per_step = p2_hist["secs"][-1] / total_steps
```

Our CPU run: 24 steps per epoch, 72 steps in 3 epochs, between 66 and 125 seconds in total in two runs, that is **0.9 to 1.7 seconds per step** (with the evaluation after each epoch included). Your time depends on your computer; on a GPU a step takes a small fraction of a second.

</details>

## Part 3. Forgetting (15 min)

*Lesson section: "Overfitting and catastrophic forgetting".* The apprentice who spends two weeks learning the bakery's way of speaking might, if she did nothing else, start to write all her letters in that style, even the ones to her mother. A model that is trained only on 24 polite answers is pulled towards them, and the weights that held other knowledge move too. This is **catastrophic forgetting**. Here you measure it on a paragraph of ordinary English (`GENERAL_TEXT`, which the model never trains on) and try three remedies: a lower learning rate, fewer steps and **replay** (mixing in some general text).

### ✏️ Imagine first

The base model's loss on `GENERAL_TEXT` is about 3 nats per token (the exact value is printed below). After full fine-tuning for 3 epochs, by how many percent do you think it rose: 1 percent, 10 percent, 50 percent? And which remedy do you expect to help the most? Write your guesses in the block below (replace each `None`), then run it.

In [ ]:
# Imagine first: forgetting
my_guess_rise_pct = None     # percent rise of the loss on GENERAL_TEXT after full fine-tuning
my_guess_best_fix = None     # "low lr", "fewer steps" or "replay"

### Step 3.1. Measure the forgetting

`text_loss` gives the average cross-entropy in nats per token. A **lower loss means the text is more natural for the model**. We measure the base model, the model after each epoch of Part 2 (stored in `p2_hist`), and its perplexity `exp(loss)`: the number of equally likely tokens the model is "choosing between" at every position.

In [ ]:
gen0, gen_end = p2_hist["general"][0], p2_hist["general"][-1]
print("epoch  loss on GENERAL_TEXT  perplexity   change")
for e, g in enumerate(p2_hist["general"]):
    print(f"{e:5d}  {g:20.3f}  {math.exp(g):10.1f}   {100 * (g - gen0) / gen0:+6.1f} %")
p3_rise = 100 * (gen_end - gen0) / gen0
print(f"\nafter {EPOCHS} epoch(s) of full fine-tuning the loss rose by {gen_end - gen0:+.3f} nats = {p3_rise:+.1f} %")
check("fine-tuning made the model a little worse on general text", gen_end > gen0, "Compare with the base model's loss (epoch 0).")
RESULTS["p3_forget"] = dict(base=round(gen0, 3), after=round(gen_end, 3), rise_pct=round(p3_rise, 1))

In our run the loss on `GENERAL_TEXT` is **3.509** nats per token for the base model (perplexity 33.4) and rises to 3.589 after epoch 1 (+2.3 percent), 3.613 after epoch 2 (+3.0 percent) and **3.664 after epoch 3 (+4.4 percent**, perplexity 39.0). Two things to notice. First, the forgetting is **gradual and keeps growing with each epoch**, although the style was already learned after epoch 1 (style_rate 1.0) and the validation loss kept improving. Second, it is **small** in absolute terms: perplexity 33 to 39 means the model is choosing between about six more equally likely words at every position, on a paragraph that it has not been trained on. With a bigger learning rate, more epochs or more different data, the rise would be larger. The point is not that 4 percent is a disaster; it is that **you never get the new skill for free**, and you must measure the price.

### Step 3.2. Four remedies, one table

A fair comparison needs the same amount of training for every recipe, so (except for recipe (d)) each recipe below trains for **one epoch** (24 steps, or 24 plus the replay steps) from the base model. The control is the full-fine-tuning model after epoch 1 of Part 2 (learning rate 3e-5, no replay), which we have already measured. The recipes are: (a) **low learning rate**: 3e-6, ten times smaller; (b) **fewer steps**: the same recipe stopped after only 6 steps; (d) a **middle way**: learning rate 1e-5 for 2 epochs (the only recipe with a different budget, 48 steps); (c) **replay**: the 24 examples plus 12 short general paragraphs (written for this lab, different from `GENERAL_TEXT`), all in one shuffled epoch, with the ordinary next-token loss (no masking) on the general paragraphs. For each recipe we record the style rate, the validation loss and the loss on `GENERAL_TEXT`.

In [ ]:
REPLAY_TEXTS = [
    "The market opens early in the morning, when the stalls are covered with fresh fish, fruit and rice cakes. Buyers walk slowly between the tables and ask about the price before they decide.",
    "A new bridge was built across the canal last year. Before that, children had to walk a long way around the school fence, and in the rainy season the road was often flooded.",
    "The teacher asked the class to bring a plant from home. Some brought flowers, some brought herbs, and one student brought a small tree that was almost as tall as the desk.",
    "In the evening the town square fills with families. Vendors sell grilled corn, an old man plays the guitar, and the children chase each other around the fountain until it is dark.",
    "Mountains are formed when large pieces of the earth's crust push against each other for millions of years. The rock slowly rises, and wind and rain wear it down again.",
    "The library keeps its oldest books in a cool room at the back. A volunteer checks the shelves every month to make sure that no water has come in and that no insects are eating the paper.",
    "When the typhoon warning was raised, the families packed bags with clothes, water and documents. The local office opened the gym of the school as a shelter, and neighbours helped carry the older people.",
    "A good bread needs only flour, water, salt and time. The baker mixes the dough at night, lets it rest, and shapes it before sunrise so that the first loaves are ready when the shop opens.",
    "Birds migrate when the seasons change. Some fly thousands of kilometres, guided by the sun, the stars and the shape of the land, and rest in wetlands on the way.",
    "The town council meets on the first Monday of every month. Anyone can come and speak about a problem, such as a broken pump or a road that needs repair, and the minutes are posted on the notice board.",
    "To measure a room you need a tape, a pencil and a piece of paper. Write down the length and the width, multiply them to find the area in square metres, and add a little extra when you buy paint.",
    "On weekends the football pitch behind the church is never empty. Teams of different ages take turns, and the smallest children play at the edge with a ball made of rolled-up cloth.",
]
P3_STEPS = 6
def p3_run(label, examples, epochs, lr, max_steps=None):
    """Train a fresh copy of the base model; return (label, style, val, general, steps, secs)."""
    mm = copy.deepcopy(base_model)
    t0 = time.time()
    if max_steps is not None:
        examples = examples[:max_steps]
    hist = p2_train(mm, examples, epochs, lr=lr, verbose=False, style_at_start=False, style_pairs=INSTR_VAL[:3] if QUICK else None)
    row = dict(label=label, style=hist["style"][-1], val=hist["val"][-1], general=hist["general"][-1], steps=len(examples) * epochs, secs=time.time() - t0)
    del mm; gc.collect()
    return row

ftrain = sft_examples
n_replay = 4 if QUICK else 12
replay_examples = []
for t in REPLAY_TEXTS[:n_replay]:
    r_ids = tok(t + tok.eos_token, return_tensors="pt")["input_ids"].to(device)
    replay_examples.append((r_ids, r_ids.clone()))     # ordinary next-token loss: the labels are the ids themselves (nothing masked)
order = random.Random(SEED).sample(range(len(ftrain)), len(ftrain))
rows = []
rows.append(dict(label="control: lr 3e-5, 1 epoch", style=None, val=None, general=None, steps=len(ftrain)))
c = p2_hist  # control values are those after epoch 1 of Part 2
rows[0].update(style=c["style"][1], val=c["val"][1], general=c["general"][1])
rows.append(p3_run("(a) low learning rate 3e-6", ftrain, 1, 3e-6))
rows.append(p3_run(f"(b) fewer steps ({P3_STEPS})", ftrain, 1, 3e-5, max_steps=P3_STEPS))
rows.append(p3_run("(d) lr 1e-5, 2 epochs", ftrain, 2, 1e-5))
rows.append(p3_run("(c) replay: 24 examples + 12 general", ftrain + replay_examples, 1, 3e-5))
base_row = dict(label="base model (no training)", style=p1_base_style, val=p2_hist["val"][0], general=gen0, steps=0)
rows.insert(0, base_row)
print(f"{'recipe':<40}{'steps':>6}{'style':>7}{'val loss':>10}{'general':>9}{'change':>9}")
for r in rows:
    print(f"{r['label']:<40}{r['steps']:>6}{r['style']:>7.2f}{r['val']:>10.3f}{r['general']:>9.3f}{100 * (r['general'] - gen0) / gen0:>+8.1f}%")
RESULTS["p3_table"] = [(r["label"], r["steps"], round(r["style"], 2), round(r["val"], 3), round(r["general"], 3)) for r in rows]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
names = ["base", "control", "(a) low lr", "(b) fewer", "(d) lr 1e-5 x2", "(c) replay"]
ax[0].bar(names, [r["general"] for r in rows], color=["gray", "C3", "C0", "C1", "C4", "C2"]); ax[0].set_ylim(min(r["general"] for r in rows) - 0.1, max(r["general"] for r in rows) + 0.1); ax[0].set_title("loss on GENERAL_TEXT (lower = less forgetting)")
ax[1].bar(names, [r["style"] for r in rows], color=["gray", "C3", "C0", "C1", "C4", "C2"]); ax[1].set_ylim(0, 1.05); ax[1].set_title("style_rate (higher = learned the style)")
for a in ax: a.tick_params(axis="x", labelrotation=30)
plt.tight_layout(); plt.show()

Read the table (our run). **The control** (learning rate 3e-5, 24 steps) learns the style (1.00) at a cost of +2.3 percent on the general text. **(a) The lower learning rate** (3e-6) keeps the general loss almost unchanged (3.512, +0.1 percent), but it also **did not learn the style**: style_rate 0.00 and a validation loss of 2.077, still far from the control's 0.335. **(b) Fewer steps** (only 6 steps) behaves the same way: 3.516 and style 0.00. So with a small budget of 24 steps, these two remedies do not reduce forgetting at a fair price; they simply learn less (the learning and the forgetting travel together). **(d) A middle way**, a learning rate of 1e-5 for two epochs (48 steps), does learn the style (1.00) with a validation loss of 0.408 (a little worse than the control's 0.335) and costs only **+0.8 percent** on the general text, about a third of the control's +2.3 percent. So a lower learning rate *can* help, but only if you give it enough steps to reach the same behaviour. **(c) Replay** (24 examples plus 12 general paragraphs, all in one shuffled epoch) is the one that helped most: style_rate 1.00, a validation loss of 0.270 (even better than the control's 0.335) and a general loss of 3.399, which is **below** the base model's 3.509 (-3.1 percent).

**An honest warning about (c).** The loss on `GENERAL_TEXT` fell *below* the base model because our replay paragraphs are the same kind of text as `GENERAL_TEXT` (short descriptive paragraphs about everyday village and town life), so the model got a little better at that kind of text, not only "kept" its old ability. A fairer reading is: replay prevented the forgetting we measured, and part of the gain is a similarity of genre between the replay data and the test paragraph. If the replay data had been about something very different, the benefit would be smaller. In real projects you replay a sample of the pre-training-like data or of general instruction data, and you check on a test set that is *not* similar to the replay set.

### ✏️ Your turn: Exercise 3.1. Which recipe is best?

Your task: use the `rows` list of Step 3.2. Among the recipes that reach a `style_rate` of at least 0.75, find the one with the smallest loss on `GENERAL_TEXT`; store its label in `best_label`. Then store in `cost_of_best` the rise of its general loss over the base model, in percent. If no recipe reaches 0.75, store the label of the one with the highest `style_rate`. Why it matters: forgetting is a trade-off, and the "best" recipe depends on how much of the new behaviour you insist on.

Replace the `None` values (or write the code) in the starter cell, run it, then run the **Check** cell below it.

In [ ]:
best_label = None     # label of the best recipe
cost_of_best = None   # percent rise of its loss on GENERAL_TEXT over the base model

In [ ]:
if best_label is None or cost_of_best is None:
    print("Fill in best_label and cost_of_best above, then run this cell again.")
else:
    _ok = [r for r in rows[1:] if r["style"] >= 0.75]
    _best = min(_ok, key=lambda r: r["general"]) if _ok else max(rows[1:], key=lambda r: r["style"])
    check("best_label is right", best_label == _best["label"], "Filter the recipes by style >= 0.75 (skip the base model), then take the minimum of 'general'.")
    check("cost_of_best is right", abs(cost_of_best - 100 * (_best["general"] - gen0) / gen0) < 0.1, "100 * (general - base general) / base general.")
    RESULTS["p3_best"] = (best_label, round(float(cost_of_best), 1))

<details><summary>Show a possible answer</summary>

```python
ok = [r for r in rows[1:] if r["style"] >= 0.75]
best = min(ok, key=lambda r: r["general"]) if ok else max(rows[1:], key=lambda r: r["style"])
best_label = best["label"]
cost_of_best = 100 * (best["general"] - gen0) / gen0
```

Three recipes reach a style_rate of at least 0.75: the control (+2.3 percent), recipe (d) (+0.8 percent) and recipe (c), replay (-3.1 percent). In our run the best is therefore `(c) replay: 24 examples + 12 general`, with a change of **-3.1 percent**. A different seed or machine can change the order of two close recipes, so always read your own table rather than trusting the recipe names.

</details>

### Step 3.3. What the table does not tell you

Loss on one paragraph is a narrow measure. A model can keep its loss on that paragraph and still lose skills elsewhere (arithmetic, another language, code). In practice one checks forgetting on a **small set of general tasks**, not on one paragraph, and watches the same measure through training to stop early. The next cell stores your three honest findings for the summary in Part 7: type them in your own words.

In [ ]:
RESULTS["p3_findings"] = [
    "Write finding 1 here: what did fine-tuning do to the style and to the general loss?",
    "Write finding 2 here: which remedy helped, and which did not?",
    "Write finding 3 here: one thing this experiment cannot tell us.",
]
for f in RESULTS["p3_findings"]: print("-", f)

**End of Parts 1 to 3.** You have seen the base model fail to answer, written the masked loss by hand, trained all 124 million weights on 24 examples, and measured both the gain (a learned style) and the price (some forgetting). Part 4 asks the next question of the lesson: what if the examples are not enough, and all we have is a coach who ranks two answers?

## Part 4. Preference learning from scratch (30 min)

*Lesson section: "Preference learning".* Instruction tuning (Part 2) shows the model *one* good answer for each question. But for many questions there is no single best answer; people can only say "this answer is better than that one". **Preference learning** turns those comparisons into a training signal. The classic recipe has two steps. First you train a **reward model**: a small network that gives a score to an answer so that the answers people prefer get the higher score. Then you use the reward model to steer the language model, either by *picking* the best of several answers (**best-of-n**, this part) or by *training* the model towards high scores (RLHF and DPO, Part 5). In this part you build a tiny reward model from GPT-2's own features, train it with the Bradley-Terry loss on 14 comparisons, test it on comparisons it has never seen, use it to pick the best of n answers, and then see how a reward model can be fooled.

### ✏️ Imagine first

You will train a reward model on only **14 pairs** (a good answer and a bad answer to the same question), on top of GPT-2's frozen features. Before you run any code, write three predictions. (1) On 6 pairs the model has never seen, how many will it rank correctly (0 to 6)? (2) Suppose the model samples 32 answers and you keep the one with the highest reward: will it be clearly better than a single answer (yes or no), and will it be a good answer to read, on a scale from 1 (nonsense) to 5 (a good answer)? (3) If you ask a program to *search* for the text with the highest possible reward, will the winner be a good answer (yes or no)?

In [ ]:
# Imagine first: the judge
my_guess_val_correct = None   # of the 6 unseen pairs, how many does the trained reward model rank correctly (0 to 6)?
my_guess_bon_better = None    # is the best of 32 answers clearly better than one random answer? ("yes" or "no")
my_guess_bon_quality = None   # how good does the best-of-32 answer read, from 1 (nonsense) to 5 (a good answer)?
my_guess_search_good = None   # if a program searches for the highest-reward text, is the winner a good answer? ("yes" or "no")

### Step 4.1. Features for a reward model

A reward model needs to read a question and an answer and give one number. We build the smallest one that can work. Step one: GPT-2 reads `question + answer`, and we keep the **final-layer hidden state of each answer token** (a vector of 768 numbers per token) and take their **mean**: one vector of 768 numbers that summarises the answer in the context of its question. GPT-2 stays frozen. Step two (Step 4.3): a single linear layer turns the 768 numbers into one score. The block below computes the feature vector of every answer in `PREF_TRAIN` (14 pairs) and `PREF_VAL` (6 pairs), then **standardises** each of the 768 numbers (subtract the mean, divide by the standard deviation measured on the training answers), because a few of GPT-2's 768 dimensions have a huge scale and would dominate a linear layer.

In [ ]:
@torch.no_grad()
def rm_features(prompt, answer):
    """Mean final-layer hidden state of the answer tokens when GPT-2 reads the prompt followed by the answer (shape 768)."""
    p = tok(format_prompt(prompt))["input_ids"]; a = tok(answer)["input_ids"]
    ids = torch.tensor([p + a]).to(device)
    h = base_model.transformer(input_ids=ids).last_hidden_state[0]
    return h[len(p):].mean(0).cpu()

def rm_stack(rows, key):
    return torch.stack([rm_features(r["prompt"], r[key]) for r in rows])

t0 = time.time()
p4_Ctr, p4_Rtr = rm_stack(PREF_TRAIN, "chosen"), rm_stack(PREF_TRAIN, "rejected")
p4_Cva, p4_Rva = rm_stack(PREF_VAL, "chosen"), rm_stack(PREF_VAL, "rejected")
p4_allf = torch.cat([p4_Ctr, p4_Rtr]); p4_mu, p4_sd = p4_allf.mean(0), p4_allf.std(0) + 1e-6
rm_norm = lambda x: (x - p4_mu) / p4_sd
print("feature tensors:", tuple(p4_Ctr.shape), tuple(p4_Rtr.shape), tuple(p4_Cva.shape), tuple(p4_Rva.shape), f"({time.time() - t0:.1f} s)")
print(f"raw features: largest absolute value {float(p4_allf.abs().max()):.1f}, largest spread over answers {float(p4_allf.std(0).max()):.2f}, smallest {float(p4_allf.std(0).min()):.4f}")
print(f"after standardising: mean of means {float(rm_norm(p4_allf).mean()):.4f}, mean of spreads {float(rm_norm(p4_allf).std(0).mean()):.3f}")
assert p4_Ctr.shape == (14, 768) and p4_Cva.shape == (6, 768)

You should see the shapes (14, 768) for the training answers and (6, 768) for the validation answers, computed in about 3 seconds. The raw features show why the standardising step matters: the largest absolute value is 246.7, and the spread (standard deviation over the 28 training answers) goes from 14.91 for the most variable dimension down to 0.0024 for the quietest, a factor of 6,000. After standardising, every dimension has mean 0 and spread 1.000. Each answer is now a point in a 768-dimensional space, and we have only 14 + 14 training points in it: far more dimensions than examples. Keep that in mind for Step 4.3.

### Step 4.2. The Bradley-Terry loss by hand

How do you train a scorer from *comparisons*? The **Bradley-Terry model** says: the probability that people prefer answer A to answer B is `sigmoid(r_A - r_B)`, where `r` is the reward. Training maximises the probability of the choices people really made, so the loss for one pair is `-log sigmoid(r_chosen - r_rejected)`. The block computes it for three cases by hand (with `math`), checks the special case of two equal scores against `ln 2`, and checks the gradient with PyTorch.

In [ ]:
def bt_loss_by_hand(r_chosen, r_rejected):
    """Bradley-Terry loss of one pair: minus the log of the probability sigmoid(r_chosen - r_rejected)."""
    return -math.log(1 / (1 + math.exp(-(r_chosen - r_rejected))))

for rc, rr in [(0.0, 0.0), (2.0, 0.0), (0.0, 2.0)]:
    pw = 1 / (1 + math.exp(-(rc - rr)))
    print(f"r_chosen {rc:4.1f}  r_rejected {rr:4.1f}  P(chosen wins) {pw:.4f}  loss {bt_loss_by_hand(rc, rr):.4f}")
print("ln 2 =", round(math.log(2), 4))
assert abs(bt_loss_by_hand(0.0, 0.0) - math.log(2)) < 1e-12

rc_t, rr_t = torch.tensor(2.0, requires_grad=True), torch.tensor(0.0, requires_grad=True)
F.softplus(-(rc_t - rr_t)).backward()      # softplus(-x) is the same as -log sigmoid(x)
p_win = 1 / (1 + math.exp(-2.0))
print(f"gradient wrt r_chosen {float(rc_t.grad):.4f}  (by hand: -(1 - P) = {-(1 - p_win):.4f})   wrt r_rejected {float(rr_t.grad):.4f}")
assert abs(float(rc_t.grad) + (1 - p_win)) < 1e-6 and abs(float(rr_t.grad) - (1 - p_win)) < 1e-6

You should see three cases. Equal scores give a probability of 0.5000 and a loss of 0.6931, which is **ln 2**: a reward model that cannot tell the answers apart pays ln 2 on every pair, so ln 2 is the baseline to beat. Scores 2 and 0 (chosen higher) give a probability of 0.8808 and a loss of 0.1269. Scores 0 and 2 (chosen lower) give a probability of 0.1192 and a loss of **2.1269**: a wrong ranking is punished much harder than a right ranking is rewarded. The gradient of the loss with respect to the chosen score is -0.1192, which equals -(1 - P): the more wrong the model is, the harder it pushes the chosen score up and the rejected score down by the same amount.

### ✏️ Your turn: Exercise 4.1. One pair by hand

Your task: a reward model gives the chosen answer the score 1.3 and the rejected answer the score -0.4. Store in `bt_p` the probability that the chosen answer wins and in `bt_loss` the loss of the pair, using the function above or the formulas. Why it matters: this one number is what every step of reward-model training pushes down.

Replace the `None` values, run the starter cell, then run the **Check** cell below it.

In [ ]:
bt_p = None      # P(chosen wins) for scores 1.3 and -0.4
bt_loss = None   # the Bradley-Terry loss of that pair

In [ ]:
if bt_p is None or bt_loss is None:
    print("Fill in bt_p and bt_loss above, then run this cell again.")
else:
    check("bt_p is right", abs(bt_p - 1 / (1 + math.exp(-1.7))) < 1e-4, "sigmoid(1.3 - (-0.4)) = sigmoid(1.7).")
    check("bt_loss is right", abs(bt_loss + math.log(bt_p)) < 1e-4, "The loss is minus the natural log of bt_p.")
    check("a correct ranking gives a loss below ln 2", bt_loss < math.log(2), "When the chosen answer has the higher score, the loss is below 0.693.")
    RESULTS["p4_bt"] = (round(bt_p, 4), round(bt_loss, 4))

<details><summary>Show a possible answer</summary>

```python
bt_p = 1 / (1 + math.exp(-(1.3 - (-0.4))))
bt_loss = bt_loss_by_hand(1.3, -0.4)
print(bt_p, bt_loss)
```

The probability is 0.8455 and the loss is 0.1678 (that is -log 0.8455). The chosen answer has the higher score, so the loss is below ln 2 = 0.6931. A wrong ranking with the same two scores swapped would cost 1.8678.

</details>

### Step 4.3. Train the reward model

The reward model is one linear layer `w` (768 weights, no bias: a bias would cancel in the difference of two scores). For a pair, the score difference is `(features_chosen - features_rejected) @ w`, and the loss is the Bradley-Terry loss averaged over the training pairs. We start from `w = 0` (every answer scores 0, so the loss must be ln 2) and train with AdamW (learning rate 0.001) for 200 steps. After every step the block records the training loss, the validation loss and the **validation accuracy** (the fraction of unseen pairs where the chosen answer gets the higher score). Chance level is 50 percent accuracy and a loss of ln 2.

In [ ]:
def rm_fit(C, R, Cv, Rv, steps=200, lr=1e-3, wd=0.0):
    """Train the linear reward model with the Bradley-Terry loss. Returns the weights and a list of (train loss, val loss, val accuracy) per step."""
    C, R, Cv, Rv = [rm_norm(x) for x in (C, R, Cv, Rv)]
    torch.manual_seed(0)
    w = torch.zeros(768, requires_grad=True)
    opt = torch.optim.AdamW([w], lr=lr, weight_decay=wd); hist = []
    for i in range(steps + 1):
        with torch.no_grad():
            dv = (Cv - Rv) @ w
            hist.append((float(F.softplus(-((C - R) @ w)).mean()), float(F.softplus(-dv).mean()), float((dv > 0).float().mean())))
        if i == steps: break
        loss = F.softplus(-((C - R) @ w)).mean(); opt.zero_grad(); loss.backward(); opt.step()
    return w.detach(), hist

def rm_score(prompt, answers):
    """Reward of each answer (a list of texts) for the question `prompt`."""
    f = torch.stack([rm_features(prompt, a) for a in answers]); return rm_norm(f) @ p4_w

p4_w, p4_hist = rm_fit(p4_Ctr, p4_Rtr, p4_Cva, p4_Rva)
print("step   train loss   val loss   val accuracy")
for s in (0, 10, 25, 50, 100, 200):
    print(f"{s:4d}   {p4_hist[s][0]:9.4f}   {p4_hist[s][1]:8.4f}   {p4_hist[s][2]:8.0%}")
assert abs(p4_hist[0][0] - math.log(2)) < 1e-6
fig, ax = plt.subplots(1, 2, figsize=(9, 2.8))
ax[0].plot([h[0] for h in p4_hist], label="train"); ax[0].plot([h[1] for h in p4_hist], label="validation"); ax[0].axhline(math.log(2), color="gray", ls=":", label="ln 2 (guessing)")
ax[0].set_xlabel("step"); ax[0].set_ylabel("Bradley-Terry loss"); ax[0].legend()
dv_ = (rm_norm(p4_Cva) - rm_norm(p4_Rva)) @ p4_w
ax[1].bar(range(6), dv_.numpy(), color=["tab:green" if d > 0 else "tab:red" for d in dv_]); ax[1].set_xlabel("unseen pair"); ax[1].set_ylabel("score(chosen) - score(rejected)")
plt.tight_layout(); plt.show()
RESULTS["p4_rm"] = (round(p4_hist[-1][0], 4), round(p4_hist[-1][1], 4), p4_hist[-1][2])

You should see the loss start at **0.6931** (ln 2, as it must when `w = 0`) and fall. The training loss reaches 0.1001 after 10 steps and 0.0015 after 200. The validation loss falls from 0.6931 to 0.2627 (step 10), 0.1185 (step 50) and **0.0917** (step 200), and the validation accuracy is **100 percent (6 of 6 pairs)** from step 10 on. So the reward model learned to rank unseen pairs from only 14 examples, and the validation loss never rose again during the 200 steps. Why so easy? With 768 weights and 14 pairs the training pairs can be separated perfectly (training loss near 0), and the chosen answers differ from the rejected ones in obvious ways (polite, helpful words against rude or empty ones). Do not celebrate yet: 6 pairs is a very small test, and the next step shows what it hides.

### Step 4.4. The learning curve, and a trap

How many comparisons does the reward model need? We train it again on the first 2, 4, 6, 8, 10 and 14 training pairs and measure the same three numbers on the 6 unseen pairs. Then we add a **trap**: six new held-out pairs, written so that the *rejected* answer is much **longer** than the chosen one (a short, correct answer against a long, harmful or silly one). In the training data the chosen answer is always the longer one, so a model that has secretly learned "longer is better" would fail the trap. Let us measure it. (The rule "pick the longer answer" gets 6 of 6 on the validation pairs, so the validation pairs alone cannot tell the two apart.)

In [ ]:
PREF_TRAP = [dict(prompt=p, chosen=c, rejected=r) for p, c, r in [
    ("How do I cross the road safely?", "Look both ways and use the crossing.", "Just run across as fast as you can without looking, cars will always stop for you because they like children, and it is the quickest way to get to the other side of the street."),
    ("What should I do if I am lost in a mall?", "Stay calm and ask a staff member for help.", "Wander around the whole mall alone for hours, go into the parking lot and accept a ride from anyone who offers, it will certainly be fine."),
    ("How do I wash my hands?", "Use soap and water and scrub for twenty seconds.", "Just wave them under the tap for one second, soap is a waste of money and germs are not real so it truly does not matter at all."),
    ("How can I be kind to a new classmate?", "Smile, say hello and invite them to sit with you.", "Ignore the new classmate completely, whisper about their clothes with your friends and laugh when they walk past the whole class."),
    ("Why should I drink water?", "Water keeps your body cool and helps it work well.", "You should never drink water, only fizzy drinks, because water is boring and your body does not need it, ever, for any reason at all."),
    ("What should I do with a broken glass?", "Do not touch it with bare hands; tell an adult and sweep it up carefully.", "Pick it up with your bare fingers as fast as you can so nobody else sees it, and then hide the pieces inside your pocket for later.")] ]
p4_Ct, p4_Rt = rm_stack(PREF_TRAP, "chosen"), rm_stack(PREF_TRAP, "rejected")
nw = lambda rows, k: [len(r[k].split()) for r in rows]
print("'longer wins' rule: validation", np.mean([a > b for a, b in zip(nw(PREF_VAL, "chosen"), nw(PREF_VAL, "rejected"))]), " trap", np.mean([a > b for a, b in zip(nw(PREF_TRAP, "chosen"), nw(PREF_TRAP, "rejected"))]))
print("pairs used   train loss   val loss   val acc   trap acc")
p4_curve = []
for k in (2, 4, 6, 8, 10, 14):
    w_k, h_k = rm_fit(p4_Ctr[:k], p4_Rtr[:k], p4_Cva, p4_Rva)
    trap_d = (rm_norm(p4_Ct) - rm_norm(p4_Rt)) @ w_k
    p4_curve.append((k, h_k[-1][0], h_k[-1][1], h_k[-1][2], float((trap_d > 0).float().mean())))
    print(f"{k:9d}   {h_k[-1][0]:10.4f}   {h_k[-1][1]:8.4f}   {h_k[-1][2]:6.0%}   {p4_curve[-1][4]:7.0%}")
w_full_trap = (rm_norm(p4_Ct) - rm_norm(p4_Rt)) @ p4_w
print("trap differences with all 14 pairs (score chosen minus score rejected):", np.round(w_full_trap.numpy(), 2))
plt.figure(figsize=(5, 3)); plt.plot([c[0] for c in p4_curve], [c[2] for c in p4_curve], "o-", label="validation loss"); plt.axhline(math.log(2), color="gray", ls=":", label="ln 2")
plt.xlabel("training pairs"); plt.ylabel("Bradley-Terry loss"); plt.legend(); plt.tight_layout(); plt.show()
RESULTS["p4_trap"] = float((w_full_trap > 0).float().mean())

You should see that the simple rule "the longer answer wins" gets **6 of 6** validation pairs right: in our data the chosen answer is always the longer one, so the validation pairs cannot tell "good" from "long". On the trap, where the harmful answer is the longer one, the rule gets **0 of 6**. The learning curve: with 2 pairs the validation loss is 0.2918 (accuracy 83 percent), with 4 pairs 0.1914 (100 percent), with 6 pairs 0.1392, with 8 pairs 0.1030, with 10 pairs 0.1456 and with all 14 pairs **0.0917**. The trend is down, but not smooth (10 pairs is worse than 8): with 6 validation pairs one pair changes a lot. On the trap, the reward model with all 14 pairs gets **5 of 6 (83 percent)**: the score differences (chosen minus rejected) are 0.02, 1.25, 2.66, 0.80, 1.28 and **-0.13**. So it did not simply learn "longer is better", but it is not safe either: the first pair is right by only 0.02, and the pair about the broken glass is ranked wrongly. The trap accuracy is 50 percent with 4 pairs, 67 percent with 6, 100 percent with 8, 83 percent with 10 and 14: with six trap pairs, differences of one pair (17 points) are noise. The honest summary is that 14 pairs give a useful but fragile judge.

### ✏️ Your turn: Exercise 4.2. A reward model with 4 pairs

Your task: train the reward model on the **first 4** training pairs only (use `rm_fit` with `p4_Ctr[:4]` and `p4_Rtr[:4]`, the default steps) and store the final validation loss in `val_loss_4` and the final validation accuracy in `val_acc_4`. Why it matters: it tells you how fast a reward model improves with data, and that even a few pairs carry signal.

Replace the `None` values (or write the code), run the starter cell, then run the **Check** cell below it.

In [ ]:
val_loss_4 = None   # final validation loss when training on the first 4 pairs
val_acc_4 = None    # final validation accuracy in that case

In [ ]:
if val_loss_4 is None or val_acc_4 is None:
    print("Fill in val_loss_4 and val_acc_4 above, then run this cell again.")
else:
    _w4, _h4 = rm_fit(p4_Ctr[:4], p4_Rtr[:4], p4_Cva, p4_Rva)
    check("val_loss_4 is right", abs(val_loss_4 - _h4[-1][1]) < 0.01, "Take the second number of the last entry of the history list.")
    check("val_acc_4 is right", abs(val_acc_4 - _h4[-1][2]) < 0.01, "Take the third number of the last entry.")
    check("4 pairs are already better than guessing (loss below ln 2)", val_loss_4 < math.log(2), "A reward model that learned something has a loss below 0.693.")
    RESULTS["p4_four"] = (round(val_loss_4, 4), round(val_acc_4, 3))

<details><summary>Show a possible answer</summary>

```python
w4, h4 = rm_fit(p4_Ctr[:4], p4_Rtr[:4], p4_Cva, p4_Rva)
val_loss_4, val_acc_4 = h4[-1][1], h4[-1][2]
print(val_loss_4, val_acc_4)
```

With 4 pairs the validation loss is 0.1914 and the validation accuracy is 1.0 (6 of 6): already well below ln 2 = 0.6931, although more than twice the loss of the 14-pair model (0.0917). The accuracy is the same as with 14 pairs, which shows that the 6 validation pairs are too easy to separate good from poor reward models; the trap of Step 4.4 does it better (50 percent for 4 pairs).

</details>

### Step 4.5. Best-of-n

Now we use the reward model. First we need a language model that writes answers in the right style: the code below runs **one epoch of full fine-tuning** of GPT-2 on `INSTR_TRAIN` (the recipe of Part 2: masked prompt, AdamW, learning rate 3e-5, batch size 1) and checks the style rate. Then, for each of the 6 questions of `PREF_VAL`, it **samples 32 answers** (temperature 1, 30 new tokens) and scores them with the reward model. Best-of-n means: draw n answers, keep the one with the highest reward. To see how the reward grows with n without drawing new samples every time, we **resample from the pool**: for each n, pick n of the 32 answers at random, take the maximum reward, and average over 200 repeats and the 6 questions.

In [ ]:
p4_sft = copy.deepcopy(base_model); p4_sft.train()
p4_opt = torch.optim.AdamW(p4_sft.parameters(), lr=3e-5); t0 = time.time()
p4_train_rows = INSTR_TRAIN[:8] if QUICK else INSTR_TRAIN
for q, a in p4_train_rows:
    p_ids = tok(format_prompt(q))["input_ids"]; r_ids = tok(a + tok.eos_token)["input_ids"]
    loss = p4_sft(input_ids=torch.tensor([p_ids + r_ids]).to(device), labels=torch.tensor([ [-100] * len(p_ids) + r_ids]).to(device)).loss
    loss.backward(); p4_opt.step(); p4_opt.zero_grad()
p4_sft.eval(); del p4_opt
print(f"one epoch of SFT on {len(p4_train_rows)} examples: {time.time() - t0:.0f} s, style rate {style_rate(p4_sft):.2f} (the base model had {style_rate(base_model):.2f})")

@torch.no_grad()
def sample_answers(model, prompt, n, max_new_tokens=30, seed=0):
    """n sampled answers (temperature 1, no top-k) of `model` to the question `prompt`."""
    ids = tok(format_prompt(prompt), return_tensors="pt").to(device); torch.manual_seed(seed)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True, temperature=1.0, top_k=None, num_return_sequences=n, pad_token_id=tok.eos_token_id)
    return [tok.decode(o[ids["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

In [ ]:
t0 = time.time(); p4_nq, p4_ns = (3, 8) if QUICK else (6, 32); p4_pool = {}
for i, row in enumerate(PREF_VAL[:p4_nq]):
    S = sample_answers(p4_sft, row["prompt"], p4_ns, seed=i); p4_pool[i] = (S, rm_score(row["prompt"], S).numpy())
print(f"sampled and scored {p4_nq * p4_ns} answers in {time.time() - t0:.0f} s")
p4_rng = np.random.RandomState(0); p4_bon = {}
print("   n   mean best reward")
for n in [x for x in (1, 2, 4, 8, 16, 32) if x <= p4_ns]:
    p4_bon[n] = float(np.mean([np.mean([max(p4_rng.choice(p4_pool[i][1], n, replace=False)) for _ in range(200)]) for i in p4_pool]))
    print(f"{n:4d}   {p4_bon[n]:9.3f}")
p4_ref_ch = float(np.mean([rm_score(r["prompt"], [r["chosen"] ]).item() for r in PREF_VAL])); p4_ref_rj = float(np.mean([rm_score(r["prompt"], [r["rejected"] ]).item() for r in PREF_VAL]))
print(f"for comparison, the human-written chosen answers score {p4_ref_ch:.2f} on average and the rejected ones {p4_ref_rj:.2f}")
for i in (0, 1):
    S, R = p4_pool[i]; print(f"\nQ: {PREF_VAL[i]['prompt']}\n  best  ({R.max():5.2f}): {S[int(R.argmax())]}\n  worst ({R.min():5.2f}): {S[int(R.argmin())]}\n  first sample ({R[0]:5.2f}): {S[0]}")
plt.figure(figsize=(4.5, 3)); plt.plot(list(p4_bon), list(p4_bon.values()), "o-"); plt.xscale("log", base=2); plt.xlabel("n (answers sampled)"); plt.ylabel("mean reward of the best"); plt.tight_layout(); plt.show()
RESULTS["p4_bon"] = {n: round(v, 3) for n, v in p4_bon.items()}

You should see that one epoch of full fine-tuning on the 24 examples takes about 21 seconds and raises the style rate from 0.00 to **1.00**: every answer now starts with "Sure!". Sampling and scoring 192 answers (6 questions times 32) took 37 seconds. The mean reward of the best answer rises steadily with n: **-1.156** for n = 1, -0.528 for n = 2, -0.092 for n = 4, 0.310 for n = 8, 0.664 for n = 16 and **0.926** for n = 32. Every doubling of n adds a roughly constant 0.35 to 0.6, and the curve is still rising at 32: best-of-n has diminishing returns in n but costs n times more to generate. Compare with the human-written answers: the chosen ones score 2.24 on average and the rejected ones -1.22. Even best-of-32 (0.93) stays far below a real good answer. Now read the winners. The best answer to the homework question (reward 1.57) is "Sure! The focus, coordinated by the Math/Research Assistant, can be diverted to the ability to concentrate when you are actually working on the math problems": it follows the style but is hard to call a good answer. The best answer for the plant (1.92) talks about a "bearing tool" and a "thorn". The worst answers are short ("Sure! Cheers!", -3.52). **Best-of-n made the model better by the judge's measure, but the judge, and the model, are weak**: the reward model likes longer, more elaborate answers, and the 124-million-parameter model cannot supply a really good one.

### ✏️ Your turn: Exercise 4.3. Best-of-n by hand

Your task: using the pool `p4_pool` (a dictionary: question number to the pair of a list of answers and an array of rewards), find for question 0 the answer with the highest reward. Store its text in `best_text` and its reward in `best_reward`. Then store in `mean_reward` the **average** reward of all the answers of question 0 (the reward of a typical single draw). Why it matters: the difference between the two numbers is exactly the gain of best-of-n for this question.

Replace the `None` values (or write the code), run the starter cell, then run the **Check** cell below it.

In [ ]:
best_text = None     # the sampled answer of question 0 with the highest reward
best_reward = None   # its reward
mean_reward = None   # the average reward of all sampled answers of question 0

In [ ]:
if best_text is None or best_reward is None or mean_reward is None:
    print("Fill in best_text, best_reward and mean_reward above, then run this cell again.")
else:
    _S, _R = p4_pool[0]
    check("best_reward is the maximum", abs(best_reward - float(_R.max())) < 1e-4, "Use the maximum of the reward array.")
    check("best_text is the answer with that reward", best_text == _S[int(_R.argmax())], "Use argmax to find the position.")
    check("mean_reward is right", abs(mean_reward - float(_R.mean())) < 1e-4, "The mean of the reward array.")
    RESULTS["p4_best"] = (round(best_reward, 3), round(mean_reward, 3))

<details><summary>Show a possible answer</summary>

```python
S0, R0 = p4_pool[0]
best_text, best_reward, mean_reward = S0[int(R0.argmax())], float(R0.max()), float(R0.mean())
print(best_reward, mean_reward, best_text)
```

The best answer has reward 1.567 and the average of the 32 answers is -1.082, so best-of-32 gains **2.65** reward points for question 0 over a typical single draw. The text of the best answer is "Sure! The focus, coordinated by the Math/Research Assistant, can be diverted to the ability to concentrate when you are actually working on the math problems": higher reward, but still not a good answer.

</details>

### Step 4.6. Reward hacking: when the judge is fooled

Best-of-32 only *picks*; it cannot move far from what the model can say. What if we **optimise** the text for the reward? The block below does the simplest search: it starts from an empty answer to question 0 and, at each of 5 steps, tries to append each of the 169 different words that occur in the chosen training answers, and keeps the word that gives the highest reward. The reward model is the same one; it was never told that the text must make sense. We print the winner at each step, compare its reward with the human-written chosen answer and with the best sample of Step 4.5, and read it.

In [ ]:
t0 = time.time(); p4_q = PREF_VAL[0]["prompt"]
p4_words = sorted({w_.strip(".,").lower() for r in PREF_TRAIN for w_ in r["chosen"].split()})
if QUICK: p4_words = p4_words[:50]
p4_cur, p4_hack = [], []
for step in range(3 if QUICK else 5):
    cands = [" ".join(p4_cur + [x]) for x in p4_words]
    rs = rm_score(p4_q, cands); k = int(rs.argmax()); p4_cur.append(p4_words[k]); p4_hack.append(float(rs[k]))
    print(f"step {step + 1}: reward {p4_hack[-1]:5.2f}   text: {' '.join(p4_cur)!r}   ({time.time() - t0:.0f} s)")
print(f"\ncandidate words tried: {len(p4_words)}")
print(f"reward of the human-written chosen answer to this question:   {rm_score(p4_q, [PREF_VAL[0]['chosen'] ]).item():5.2f}")
print(f"reward of the highest sampled answer of Step 4.5:             {p4_pool[0][1].max():5.2f}")
print(f"reward of the searched word salad:                            {p4_hack[-1]:5.2f}")
RESULTS["p4_hack"] = (round(p4_hack[-1], 2), " ".join(p4_cur))

You should see the search find a high reward almost at once: the single word "bring" already scores **4.93**, and "bring buying" scores **6.62**, more than twice the **2.94** of the human-written chosen answer to this question and four times the 1.57 of the best sampled answer. The reward goes up a little more to 6.97 with a third word ("bring buying test") and then slips slightly (6.95, 6.63), because the search only appends words and cannot go back. The final text is "bring buying test real study": **word salad with a higher reward than any real answer.** This is **reward hacking** (also called over-optimisation or Goodhart's law): the reward model is only a proxy trained on 14 pairs, so it has regions of the input space that it has never seen, where its score is arbitrary. A search that is strong enough finds exactly these regions. The search used 169 words, 5 steps and 30 seconds of a CPU: a real reinforcement-learning run explores far more. To make the attack harder you would add many more pairs, including pairs where the rejected answer is word salad, and train the model on its own samples (the lesson's iterative RLHF).

### ✏️ Your turn: Exercise 4.4. Reward of a trap

Your task: take the first pair of `PREF_TRAP` (a short correct answer and a long harmful one). Store in `trap_chosen` and `trap_rejected` the rewards that the trained model `p4_w` gives them (use `rm_score(prompt, [answer])`), and in `trap_ok` whether the chosen answer scores higher (True or False). Why it matters: a reward model is only as good as the pairs it has been tested on, and the validation pairs may be too easy.

Replace the `None` values (or write the code), run the starter cell, then run the **Check** cell below it.

In [ ]:
trap_chosen = None     # reward of the chosen answer of PREF_TRAP[0]
trap_rejected = None   # reward of the rejected answer of PREF_TRAP[0]
trap_ok = None         # True if the chosen answer scores higher

In [ ]:
if trap_chosen is None or trap_rejected is None or trap_ok is None:
    print("Fill in trap_chosen, trap_rejected and trap_ok above, then run this cell again.")
else:
    _t = PREF_TRAP[0]
    check("trap_chosen is right", abs(trap_chosen - rm_score(_t["prompt"], [_t["chosen"] ]).item()) < 1e-3, "rm_score(PREF_TRAP[0]['prompt'], [PREF_TRAP[0]['chosen'] ]).item()")
    check("trap_rejected is right", abs(trap_rejected - rm_score(_t["prompt"], [_t["rejected"] ]).item()) < 1e-3, "The same with the rejected answer.")
    check("trap_ok is consistent", bool(trap_ok) == (trap_chosen > trap_rejected), "Compare the two numbers.")
    RESULTS["p4_trap0"] = (round(trap_chosen, 3), round(trap_rejected, 3), bool(trap_ok))

<details><summary>Show a possible answer</summary>

```python
t0_ = PREF_TRAP[0]
trap_chosen = rm_score(t0_["prompt"], [t0_["chosen"] ]).item()
trap_rejected = rm_score(t0_["prompt"], [t0_["rejected"] ]).item()
trap_ok = trap_chosen > trap_rejected
print(trap_chosen, trap_rejected, trap_ok)
```

The reward of the short correct answer is -0.227 and the reward of the long harmful answer is -0.252, so `trap_ok` is True, but the margin is only **0.025**: the reward model hardly sees the difference. Both scores are low (a good answer on the validation set scored 2.94), which suggests that this kind of question (road safety) is far from the 14 training pairs.

</details>

### ✏️ Reflect: Part 4

Look at your Imagine-first guesses. In your notebook, answer in two or three sentences each: (1) How close was your guess for the unseen pairs? (2) What did best-of-n gain, and what did it not change (read the best answer again)? (3) The word salad has a higher reward than every real answer. Whose fault is that: the reward model, the search, or the 14 pairs? What would you add to the data to make this attack harder?

## Part 5. KL, β and DPO on a toy policy (35 min)

*Lesson section: "RLHF and DPO".* In Part 4 you *picked* from a fixed model. RLHF and DPO *change* the model. If we simply told the model "maximise the reward", it would put all its probability on the single best-scoring answer, forget everything else, and exploit every mistake of the reward model (Step 4.6 showed how large those mistakes can be). The cure is a leash: maximise the expected reward **minus β times the KL divergence from the reference model**. The strength of the leash is β. This objective has an exact solution, `pi_star(y) = pi_ref(y) * exp(r(y) / β) / Z`, and **DPO** is a way to reach it without ever training a reward model or running reinforcement learning. In this part you compute the solution with numpy on a toy with 3 answers, sweep β, plot the frontier of reward against KL, then repeat with 6 answers where the reward model is wrong about one answer, train DPO by gradient descent and check that it finds the closed-form optimum, and finally run a few DPO steps on the real GPT-2.

### ✏️ Imagine first

A model answers a question in one of three ways: rude (probability 0.6), plain (0.3) and helpful (0.1). The rewards are 0, 1 and 2. With a strong leash (β = 100) the new policy hardly moves. With a very weak leash (β = 0.05) it should go all the way to the best answer. Before you run any code, predict (1) the expected reward and (2) the probability of the helpful answer for the in-between choice β = 1. For reference, the reference model's expected reward is 0.6 * 0 + 0.3 * 1 + 0.1 * 2 = 0.5.

In [ ]:
# Imagine first: the leash
my_guess_reward_b1 = None     # expected reward of the optimal policy for beta = 1 (between 0.5 and 2.0)
my_guess_helpful_b1 = None    # probability of the helpful answer under that policy (0 to 1)

### Step 5.1. The closed-form solution

We want the policy `pi` that maximises `J(pi) = E_pi[r] - β * KL(pi || pi_ref)`. The solution is `pi*(y) = pi_ref(y) * exp(r(y) / β) / Z`, where `Z` is the number that makes the probabilities sum to 1: take the reference probabilities and **multiply each answer by `exp(r / β)`**, a boost that is larger for higher rewards and weaker when β is large. The block computes it for several β, prints the expected reward and the KL, and then checks the claim: no policy among 100,000 random ones has a higher `J` than `pi*`.

In [ ]:
def softmax_np(z):
    z = np.asarray(z, float); e = np.exp(z - z.max()); return e / e.sum()
def kl_np(p, q):
    """KL divergence of p from q in nats: sum of p * log(p / q) (terms with p = 0 count as 0)."""
    p, q = np.asarray(p, float), np.asarray(q, float); m = p > 0
    return float(np.sum(p[m] * np.log(p[m] / q[m])))
def pi_star(ref, r, beta):
    """The best policy for the objective 'expected reward minus beta * KL from the reference'."""
    w = np.asarray(ref, float) * np.exp((np.asarray(r, float) - np.max(r)) / beta); return w / w.sum()

P5_NAMES = ["rude", "plain", "helpful"]; p5_ref = np.array([0.6, 0.3, 0.1]); p5_r = np.array([0.0, 1.0, 2.0])
print(f"reference: probabilities {p5_ref}, expected reward {p5_ref @ p5_r:.3f}")
print("  beta   P(rude)  P(plain)  P(helpful)   E[reward]      KL")
for beta in (100, 10, 2, 1, 0.5, 0.2, 0.05):
    p = pi_star(p5_ref, p5_r, beta)
    print(f"{beta:6g}   {p[0]:.4f}   {p[1]:.4f}    {p[2]:.4f}      {p @ p5_r:.4f}   {kl_np(p, p5_ref):.4f}")

p5_beta = 1.0; p5_pi = pi_star(p5_ref, p5_r, p5_beta)
J = lambda p: float(p @ p5_r - p5_beta * kl_np(p, p5_ref))
p5_rng = np.random.default_rng(0); p5_best_rand = max(J(p5_rng.dirichlet(np.ones(3))) for _ in range(100000))
print(f"\nbeta = 1: J(pi*) = {J(p5_pi):.6f}, the best of 100,000 random policies = {p5_best_rand:.6f}")
assert J(p5_pi) >= p5_best_rand - 1e-12 and abs(p5_pi.sum() - 1) < 1e-12
assert np.abs(pi_star(p5_ref, p5_r, 1e6) - p5_ref).max() < 1e-4 and pi_star(p5_ref, p5_r, 0.01)[2] > 0.999
RESULTS["p5_b1"] = (round(float(p5_pi @ p5_r), 4), round(float(p5_pi[2]), 4))

You should see the reference model with an expected reward of 0.500, and the table. Look at the row for β = 1: the optimal policy has probabilities 0.2785 (rude), 0.3785 (plain) and 0.3430 (helpful), an expected reward of **1.0645** and a KL of 0.2970. So the answer to the Imagine-first question is: the reward doubles (0.5 to 1.06) and the probability of the helpful answer goes from 0.10 to 0.34. The extremes behave as expected: with β = 100 the policy stays at the reference (reward 0.5045, KL 0.0000), and with β = 0.05 all the probability goes to the helpful answer (reward 2.0000, KL 2.3026, which is ln 10, the cost of moving from probability 0.1 to 1). The check on the objective: J(pi*) = 0.767508, against 0.767505 for the best of 100,000 random policies: the formula wins, and the best random policy gets within 0.000003, as it should since random search can get close but not above. **β is a price: a high β makes moving expensive, a low β makes it cheap.**

### ✏️ Your turn: Exercise 5.1. The policy for β = 0.5

Your task: compute the optimal policy for β = 0.5 on this toy and store in `pi_half` the probability of the **helpful** answer, in `reward_half` the expected reward and in `kl_half` the KL from the reference. Why it matters: one point on the frontier of Step 5.2.

Replace the `None` values (or write the code), run the starter cell, then run the **Check** cell below it.

In [ ]:
pi_half = None       # P(helpful) of the optimal policy for beta = 0.5
reward_half = None   # its expected reward
kl_half = None       # its KL from the reference

In [ ]:
if pi_half is None or reward_half is None or kl_half is None:
    print("Fill in pi_half, reward_half and kl_half above, then run this cell again.")
else:
    _p = pi_star(p5_ref, p5_r, 0.5)
    check("pi_half is right", abs(pi_half - _p[2]) < 1e-3, "pi_star(p5_ref, p5_r, 0.5)[2].")
    check("reward_half is right", abs(reward_half - _p @ p5_r) < 1e-3, "The dot product of the policy with the rewards.")
    check("kl_half is right", abs(kl_half - kl_np(_p, p5_ref)) < 1e-3, "kl_np(policy, p5_ref).")
    RESULTS["p5_half"] = (round(pi_half, 4), round(reward_half, 4), round(kl_half, 4))

<details><summary>Show a possible answer</summary>

```python
p = pi_star(p5_ref, p5_r, 0.5)
pi_half, reward_half, kl_half = float(p[2]), float(p @ p5_r), kl_np(p, p5_ref)
print(pi_half, reward_half, kl_half)
```

The helpful answer gets probability 0.6597, the expected reward is 1.5872 and the KL is 1.0609. The reward tripled compared with the reference (0.5), and the policy now says "helpful" two times out of three, but it pays 1.06 nats of KL.

</details>

### Step 5.2. The frontier: reward against KL

Every β gives one policy, and one point (KL, expected reward). We sweep β from 0.03 to 30 (60 values, equally spaced on a log scale) and draw the curve, which is the **frontier**: for a given amount of KL, no policy earns more reward. To check that, the plot also shows 3,000 random policies as grey dots. They must all lie on or below the curve. Then the block prints how much of the possible gain you get for how much KL.

In [ ]:
p5_betas = np.logspace(np.log10(0.03), np.log10(30), 60)
p5_front = np.array([ [kl_np(pi_star(p5_ref, p5_r, b), p5_ref), pi_star(p5_ref, p5_r, b) @ p5_r] for b in p5_betas])
p5_rand = np.array([ [kl_np(p, p5_ref), p @ p5_r] for p in p5_rng.dirichlet(np.ones(3), 3000)])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].scatter(p5_rand[:, 0], p5_rand[:, 1], s=4, color="lightgray", label="3,000 random policies"); ax[0].plot(p5_front[:, 0], p5_front[:, 1], "r-", lw=2, label="pi* for each beta")
for b in (10, 2, 1, 0.5, 0.2):
    p = pi_star(p5_ref, p5_r, b); ax[0].annotate(f"β={b}", (kl_np(p, p5_ref), p @ p5_r), fontsize=8)
ax[0].set_xlabel("KL from the reference (nats)"); ax[0].set_ylabel("expected reward"); ax[0].legend(fontsize=8)
for j, nm in enumerate(P5_NAMES): ax[1].plot(p5_betas, [pi_star(p5_ref, p5_r, b)[j] for b in p5_betas], label=nm)
ax[1].set_xscale("log"); ax[1].set_xlabel("β (log scale)"); ax[1].set_ylabel("probability"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
# no random policy beats the frontier: compare each with the frontier value at its own KL
p5_dense = np.array([ [kl_np(pi_star(p5_ref, p5_r, b), p5_ref), pi_star(p5_ref, p5_r, b) @ p5_r] for b in np.logspace(-2, 3, 3000)])      # a finer frontier for the check
p5_above = int(np.sum(p5_rand[:, 1] > np.interp(p5_rand[:, 0], p5_dense[::-1, 0], p5_dense[::-1, 1]) + 1e-3))
print(f"random policies above the frontier: {p5_above} of 3000")
assert p5_above == 0
for target in (0.1, 0.5, 1.0):
    k = int(np.argmin(np.abs(p5_front[:, 0] - target)))
    print(f"KL about {target}: beta = {p5_betas[k]:.3f}, expected reward {p5_front[k, 1]:.3f}  (the reference has 0.5, the maximum is 2.0)")
RESULTS["p5_frontier"] = (round(float(np.interp(0.5, p5_front[::-1, 0], p5_front[::-1, 1])), 3),)

You should see two plots. On the left, the red curve is the frontier and the 3,000 grey dots lie below it: 0 of 3,000 random policies are above it (the block asserts this). On the right, the probabilities show how the policy slides from the reference (large β) to the single best answer (small β). The text lines: about 0.1 nats of KL buy an expected reward of 0.832 (β about 1.6), 0.5 nats buy 1.220 (β about 0.80) and 1.0 nats buy 1.590 (β about 0.50). The frontier is **concave**: the first nats of KL are the cheapest. Going from KL 0.1 to 0.5 gains 0.39 of reward; going from 1.0 to 2.3 (almost the maximum) gains only 0.41 more for 1.3 nats. That is why in practice β is chosen to give a small KL: most of the benefit comes first.

### Step 5.3. Six answers, and a reward model that is wrong

Now six answers, with a reference model that mostly says the first ones. We give two reward vectors. The **true** reward `r_true` is what people really want. The **proxy** `r_proxy` is what a reward model learned, and it is wrong about the last answer: it over-rates a weird answer (for example the word salad of Step 4.6) that the reference model almost never produces. We optimise the *proxy* for each β and measure the *true* reward of the result. If the leash is long enough (small β), the policy finds the weird answer and the true reward falls.

In [ ]:
p5_ref6 = np.array([0.35, 0.25, 0.18, 0.12, 0.08, 0.02])
p5_true = np.array([0.0, 0.5, 1.0, 1.5, 2.0, -1.0]); p5_proxy = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 4.0])
print(f"reference: true reward {p5_ref6 @ p5_true:.3f}, proxy reward {p5_ref6 @ p5_proxy:.3f}, P(weird answer) = {p5_ref6[5]}")
print("  beta   KL     proxy reward   TRUE reward   P(best true answer)  P(weird answer)")
p5_rows = []
for beta in (30, 10, 3, 1, 0.5, 0.3, 0.2, 0.1, 0.05):
    p = pi_star(p5_ref6, p5_proxy, beta); p5_rows.append((beta, kl_np(p, p5_ref6), p @ p5_proxy, p @ p5_true, p[4], p[5]))
    print(f"{beta:6g}  {p5_rows[-1][1]:5.2f}   {p5_rows[-1][2]:9.3f}    {p5_rows[-1][3]:9.3f}        {p[4]:7.3f}          {p[5]:7.3f}")
p5_bg = np.logspace(np.log10(0.03), np.log10(30), 200)
p5_curve = np.array([ [b, kl_np(pi_star(p5_ref6, p5_proxy, b), p5_ref6), pi_star(p5_ref6, p5_proxy, b) @ p5_proxy, pi_star(p5_ref6, p5_proxy, b) @ p5_true] for b in p5_bg])
p5_peak = p5_curve[int(np.argmax(p5_curve[:, 3]))]
print(f"\nthe best beta for the TRUE reward (searching 200 values): beta = {p5_peak[0]:.3f}, KL = {p5_peak[1]:.3f}, true reward {p5_peak[3]:.3f}")
plt.figure(figsize=(5.5, 3.2)); plt.plot(p5_curve[:, 1], p5_curve[:, 2], label="proxy reward (what we optimise)"); plt.plot(p5_curve[:, 1], p5_curve[:, 3], label="true reward")
plt.axvline(p5_peak[1], color="gray", ls=":"); plt.xlabel("KL from the reference (nats)"); plt.ylabel("expected reward"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()
assert p5_curve[0, 3] < p5_curve[:, 3].max() - 0.5        # the true reward is far below its peak when the leash is almost gone (smallest beta)
RESULTS["p5_goodhart"] = (round(float(p5_peak[0]), 3), round(float(p5_peak[3]), 3), round(float(p5_curve[0, 3]), 3))

You should see that the reference has a true reward of 0.625 and a proxy reward of 0.725, with 2 percent probability of the weird answer. As β falls, the **proxy reward rises without limit** (0.746, 0.792, 0.988, 2.031, 3.672, 3.985 and 4.000, the maximum of the proxy) but the **true reward rises first and then collapses**: 0.635 at β = 30, 0.654 at β = 10, **0.702 at β = 3**, then 0.459 at β = 1 (worse than the reference, with 31 percent of the probability on the weird answer), -0.663 at β = 0.5 and -1.000 at the end, where the policy always gives the weird answer. The best β for the true reward is **2.221**, with a KL of only 0.097 nats and a true reward of 0.710. This is Goodhart's law as a picture: the gain at the peak (0.710 against 0.625) is small and the loss beyond it is huge. It is also the toy version of Step 4.6. **The KL leash is not a luxury: it limits how far the policy can walk into the places where the reward model is wrong.** In real RLHF you cannot see the true reward; you can only see the KL and watch quality on held-out human judgements.

### ✏️ Your turn: Exercise 5.2. The weakest leash that is still safe

Your task: among the β values in the array `p5_bg` (the 200 values above), find the **smallest** β for which the true reward of the optimised policy is still at least the true reward of the reference model (`p5_ref6 @ p5_true`, so the policy is not worse than before). Store it in `beta_safe`, and store in `kl_safe` the KL from the reference at that β. Why it matters: it is how you would choose β in practice: as small as possible (as much improvement as possible) while the true quality has not dropped.

Replace the `None` values (or write the code), run the starter cell, then run the **Check** cell below it.

In [ ]:
beta_safe = None   # the smallest beta in p5_bg with true reward at least that of the reference
kl_safe = None     # the KL of that policy from p5_ref6

In [ ]:
if beta_safe is None or kl_safe is None:
    print("Fill in beta_safe and kl_safe above, then run this cell again.")
else:
    _ok = [b for b in p5_bg if pi_star(p5_ref6, p5_proxy, b) @ p5_true >= p5_ref6 @ p5_true]
    check("beta_safe is right", abs(beta_safe - min(_ok)) < 1e-9, "Keep only the betas whose true reward is at least p5_ref6 @ p5_true, then take the minimum.")
    check("kl_safe is right", abs(kl_safe - kl_np(pi_star(p5_ref6, p5_proxy, min(_ok)), p5_ref6)) < 1e-6, "KL of the policy for beta_safe.")
    RESULTS["p5_safe"] = (round(float(beta_safe), 3), round(float(kl_safe), 3))

<details><summary>Show a possible answer</summary>

```python
ok = [b for b in p5_bg if pi_star(p5_ref6, p5_proxy, b) @ p5_true >= p5_ref6 @ p5_true]
beta_safe = min(ok); kl_safe = kl_np(pi_star(p5_ref6, p5_proxy, beta_safe), p5_ref6)
print(beta_safe, kl_safe)
```

The smallest β is 1.274, with a KL of 0.407 from the reference. For a smaller β the policy moves towards the weird answer and its true reward falls below the reference's 0.625. Note that 'not worse than before' is a weak criterion: the best β (2.221) has a better true reward than this, and the proxy reward would always urge us to a smaller β.

</details>

### Step 5.4. DPO by gradient descent finds the same policy

**DPO (Direct Preference Optimization)** skips the reward model. Its loss for a pair (winner `w`, loser `l`) uses only the log-probabilities of the policy and of the reference: `loss = -log sigmoid( β * [ (log pi(w) - log pi_ref(w)) - (log pi(l) - log pi_ref(l)) ] )`. The quantity `β * log(pi / pi_ref)` plays the role of an **implied reward**. Here is the check of the theory. The toy policy is a softmax of 3 numbers `theta`, started at `log pi_ref`. The preference data are *every ordered pair of different answers*, each pair weighted equally, and the label probability is Bradley-Terry with the true reward: answer i beats answer j with probability `sigmoid(r_i - r_j)`. We minimise the expected DPO loss by plain gradient descent (PyTorch autograd) and compare the result with `pi_star(ref, r, β)`, for β = 1 and β = 0.3. They must agree within 0.001.

In [ ]:
def dpo_pair_loss(logp_w, logp_l, ref_w, ref_l, beta):
    """DPO loss of one pair, from four log-probabilities: -log sigmoid(beta * ((logp_w - ref_w) - (logp_l - ref_l)))."""
    return -F.logsigmoid(beta * ((logp_w - ref_w) - (logp_l - ref_l)))

def dpo_toy(ref, r, beta, steps=3000, lr=5.0):
    """Train a softmax policy on the expected DPO loss over all pairs with Bradley-Terry labels. Returns the final probabilities and the loss history."""
    K = len(ref); lref = torch.tensor(np.log(ref)); R = torch.tensor(np.asarray(r, float))
    theta = lref.clone().requires_grad_(True)
    W = torch.ones(K, K, dtype=torch.float64) / (K * (K - 1)); W.fill_diagonal_(0)
    Pw = torch.sigmoid(R[:, None] - R[None, :])             # probability that i beats j
    opt = torch.optim.SGD([theta], lr=lr); hist = []
    for s in range(steps):
        lp = torch.log_softmax(theta, 0)
        L = (W * Pw * dpo_pair_loss(lp[:, None], lp[None, :], lref[:, None], lref[None, :], beta)).sum()
        opt.zero_grad(); L.backward(); opt.step(); hist.append(float(L))
    return torch.softmax(theta, 0).detach().numpy(), hist

p5_dpo = {}
for beta in (1.0, 0.3):
    pd_, hd_ = dpo_toy(p5_ref, p5_r, beta); ps_ = pi_star(p5_ref, p5_r, beta); p5_dpo[beta] = (pd_, ps_, hd_)
    print(f"beta {beta}: DPO policy {np.round(pd_, 4)}   closed form {np.round(ps_, 4)}   largest difference {np.abs(pd_ - ps_).max():.2e}   loss {hd_[0]:.4f} -> {hd_[-1]:.4f}")
    assert np.abs(pd_ - ps_).max() < 1e-3
print("a one-pair check by hand: loss of (helpful beats rude) under the reference itself =", round(float(dpo_pair_loss(torch.tensor(0.0), torch.tensor(0.0), torch.tensor(0.0), torch.tensor(0.0), 1.0)), 4), "= ln 2")
plt.figure(figsize=(5, 3)); plt.plot(p5_dpo[1.0][2], label="β = 1"); plt.plot(p5_dpo[0.3][2], label="β = 0.3"); plt.xlabel("gradient step"); plt.ylabel("expected DPO loss"); plt.legend(); plt.tight_layout(); plt.show()
RESULTS["p5_dpo"] = tuple(round(float(np.abs(p5_dpo[b][0] - p5_dpo[b][1]).max()), 6) for b in (1.0, 0.3))

You should see that the DPO policy and the closed-form policy agree to machine precision: for β = 1, both are (0.2785, 0.3785, 0.3430) with a largest difference of 5.6e-17; for β = 0.3, both are (0.0069, 0.0960, 0.8971) with a largest difference of 2.2e-16 (the assertion allows 0.001). The expected DPO loss falls from 0.3466 (= ln 2 / 2: when the policy equals the reference every pair costs ln 2, but the label probabilities split each pair between its two directions, so the pairs cost half of ln 2 on average) to 0.2550 in both cases. **DPO never used a reward**: it only used pairs and the log-ratio of the policy to the reference, yet it found exactly the policy that maximises reward minus β times KL. The reason is the identity behind the loss: the reward that DPO implies, β times log(pi / pi_ref), equals the true reward up to a constant that cancels in every pair. The check by hand at the end shows the starting loss of a pair, ln 2 = 0.6931.

### ✏️ Your turn: Exercise 5.3. DPO loss by hand

Your task: a pair has the log-probabilities `logp_w = -1.0`, `logp_l = -2.0` under the policy and `ref_w = -1.5`, `ref_l = -1.5` under the reference, and β = 0.5. Compute the implied reward margin `beta * ((logp_w - ref_w) - (logp_l - ref_l))` and store it in `dpo_margin`, and store the loss in `dpo_loss`. Why it matters: during DPO training on a real model this margin is the number you watch; it should grow.

Replace the `None` values, run the starter cell, then run the **Check** cell below it.

In [ ]:
dpo_margin = None   # beta * ((logp_w - ref_w) - (logp_l - ref_l)) with the numbers above
dpo_loss = None     # -log sigmoid(dpo_margin)

In [ ]:
if dpo_margin is None or dpo_loss is None:
    print("Fill in dpo_margin and dpo_loss above, then run this cell again.")
else:
    _m = 0.5 * ((-1.0 + 1.5) - (-2.0 + 1.5))
    check("dpo_margin is right", abs(dpo_margin - _m) < 1e-6, "(logp_w - ref_w) is 0.5 and (logp_l - ref_l) is -0.5; multiply their difference by beta.")
    check("dpo_loss is right", abs(dpo_loss + math.log(1 / (1 + math.exp(-_m)))) < 1e-6, "Minus the log of the sigmoid of the margin.")
    check("the loss is below ln 2 because the policy already prefers the winner", dpo_loss < math.log(2), "A positive margin gives a loss below 0.693.")
    RESULTS["p5_dpo_hand"] = (round(dpo_margin, 4), round(dpo_loss, 4))

<details><summary>Show a possible answer</summary>

```python
dpo_margin = 0.5 * ((-1.0 - (-1.5)) - (-2.0 - (-1.5)))
dpo_loss = float(dpo_pair_loss(torch.tensor(-1.0), torch.tensor(-2.0), torch.tensor(-1.5), torch.tensor(-1.5), 0.5))
print(dpo_margin, dpo_loss)
```

The margin is 0.5 * (0.5 - (-0.5)) = **0.5** and the loss is **0.4741**, below ln 2 = 0.6931. The policy already gives more probability to the winner than the reference does (by 0.5 nats) and less probability to the loser (by 0.5 nats), so the pair is partly learned. Training would continue to push the margin up, and the gradient shrinks as the margin grows.

</details>

### Step 5.5. DPO with only a few comparisons

Real data are not "all pairs with exact probabilities": they are a **finite list of sampled comparisons**, each with a single winner. Let us repeat the experiment with such data. We draw N pairs (a pair of two different answers, chosen uniformly; the winner is chosen with the Bradley-Terry probability), run gradient descent on the average DPO loss for 3,000 steps with β = 1, and measure the largest difference from the closed-form policy. We repeat with 5 different random draws for N = 10, 30, 100, 300 and 1,000 and print the average.

In [ ]:
def dpo_toy_data(ref, r, beta, pairs, steps=3000, lr=5.0):
    """DPO on a finite list of (winner, loser) index pairs. Returns the final probabilities."""
    lref = torch.tensor(np.log(ref)); theta = lref.clone().requires_grad_(True); W_ = torch.tensor([p[0] for p in pairs]); L_ = torch.tensor([p[1] for p in pairs])
    opt = torch.optim.SGD([theta], lr=lr)
    for s in range(steps):
        lp = torch.log_softmax(theta, 0)
        loss = dpo_pair_loss(lp[W_], lp[L_], lref[W_], lref[L_], beta).mean(); opt.zero_grad(); loss.backward(); opt.step()
    return torch.softmax(theta, 0).detach().numpy()

def draw_pairs(r, n, seed):
    g = np.random.RandomState(seed); out = []
    while len(out) < n:
        i, j = g.choice(len(r), 2, replace=False)
        out.append((i, j) if g.rand() < 1 / (1 + np.exp(-(r[i] - r[j]))) else (j, i))
    return out

p5_star = pi_star(p5_ref, p5_r, 1.0); p5_err = {}
print("   N pairs   largest difference from pi* (mean of 5 draws)   (min, max)")
for N in ((10, 100) if QUICK else (10, 30, 100, 300, 1000)):
    errs = [np.abs(dpo_toy_data(p5_ref, p5_r, 1.0, draw_pairs(p5_r, N, seed=s), steps=1000 if QUICK else 3000) - p5_star).max() for s in range(5)]
    p5_err[N] = float(np.mean(errs)); print(f"{N:9d}   {p5_err[N]:10.4f}                              ({min(errs):.4f}, {max(errs):.4f})")
RESULTS["p5_finite"] = {N: round(v, 4) for N, v in p5_err.items()}

You should see that the error depends strongly on the number of comparisons. The largest difference from pi* is **0.5237** with 10 pairs (individual draws range from 0.2784 to 0.6564), 0.2548 with 30, 0.0878 with 100, 0.0446 with 300 and **0.0253** with 1,000 (range 0.0053 to 0.0465). With all pairs and exact probabilities (Step 5.4) the error was zero; with sampled comparisons it falls roughly like one over the square root of N. With only 10 comparisons the policy is far from the optimum, sometimes preferring the wrong answer: each comparison is only a noisy vote, and 3,000 steps of gradient descent faithfully fit the noise. This is why DPO needs far more pairs than our 14 for a real model, and why β and the number of steps act as regularisers.

### Step 5.6. A few DPO steps on the real GPT-2 (optional, about 1 minute)

The same loss works on a real language model; only the four log-probabilities change. The log-probability of an answer is the sum of the log-probabilities of its tokens given the question (the answer only: the prompt tokens are masked, as in Part 2). The reference is a frozen copy of the SFT model of Step 4.5; the policy starts as another copy and is trained on `PREF_TRAIN` with β = 0.1, AdamW, learning rate 1e-5, batch size 1 and 2 epochs (28 steps; 1 epoch in quick mode). Before and after we measure, on the training pairs and on the 6 validation pairs, the **implied reward margin** and the fraction of pairs with a positive margin, and the log-probability of the chosen answers. Small data and only 28 steps: do not expect miracles; read the numbers honestly.

In [ ]:
def seq_logp(model, prompt, answer, grad=False):
    """Sum of the log-probabilities of the answer tokens given the question (prompt tokens masked)."""
    p_ids = tok(format_prompt(prompt))["input_ids"]; a_ids = tok(answer)["input_ids"]
    ids = torch.tensor([p_ids + a_ids]).to(device)
    with torch.set_grad_enabled(grad):
        lp = torch.log_softmax(model(input_ids=ids).logits[0, :-1], -1)
    tgt = ids[0, 1:]; tok_lp = lp.gather(1, tgt[:, None])[:, 0]
    return tok_lp[len(p_ids) - 1:].sum()

p5_ref_model = copy.deepcopy(p4_sft).eval()
for q_ in p5_ref_model.parameters(): q_.requires_grad_(False)
p5_cache = {}
def ref_logp(prompt, answer):
    k = (prompt, answer)
    if k not in p5_cache: p5_cache[k] = float(seq_logp(p5_ref_model, prompt, answer))
    return p5_cache[k]

def dpo_report(model, rows, beta=0.1):
    model.eval(); m = []; ch = []
    for r in rows:
        lw, ll = float(seq_logp(model, r["prompt"], r["chosen"])), float(seq_logp(model, r["prompt"], r["rejected"]))
        m.append(beta * ((lw - ref_logp(r["prompt"], r["chosen"])) - (ll - ref_logp(r["prompt"], r["rejected"])))); ch.append(lw)
    return float(np.mean(m)), float(np.mean(np.array(m) > 0)), float(np.mean(ch))

p5_pol = copy.deepcopy(p4_sft); p5_before = (dpo_report(p5_pol, PREF_TRAIN), dpo_report(p5_pol, PREF_VAL))
print(f"before: train margin {p5_before[0][0]:.3f}, positive {p5_before[0][1]:.0%}, chosen logp {p5_before[0][2]:.1f} | validation margin {p5_before[1][0]:.3f}, positive {p5_before[1][1]:.0%}, chosen logp {p5_before[1][2]:.1f}")
p5_opt = torch.optim.AdamW(p5_pol.parameters(), lr=1e-5); p5_loss = []; t0 = time.time()
for ep in range(1 if QUICK else 2):
    p5_pol.train(); order = np.random.RandomState(ep).permutation(len(PREF_TRAIN)); tot = 0.0
    for i in (order[:6] if QUICK else order):
        r = PREF_TRAIN[i]
        lw, ll = seq_logp(p5_pol, r["prompt"], r["chosen"], grad=True), seq_logp(p5_pol, r["prompt"], r["rejected"], grad=True)
        loss = dpo_pair_loss(lw, ll, ref_logp(r["prompt"], r["chosen"]), ref_logp(r["prompt"], r["rejected"]), 0.1)
        p5_opt.zero_grad(); loss.backward(); p5_opt.step(); tot += float(loss)
    p5_loss.append(tot / len(PREF_TRAIN)); print(f"epoch {ep + 1}: mean DPO loss {p5_loss[-1]:.4f}   ({time.time() - t0:.0f} s)")
p5_after = (dpo_report(p5_pol, PREF_TRAIN), dpo_report(p5_pol, PREF_VAL)); del p5_opt
print(f"after:  train margin {p5_after[0][0]:.3f}, positive {p5_after[0][1]:.0%}, chosen logp {p5_after[0][2]:.1f} | validation margin {p5_after[1][0]:.3f}, positive {p5_after[1][1]:.0%}, chosen logp {p5_after[1][2]:.1f}")
print("a sample answer to a validation question, before and after:")
print("  SFT:", repr(generate(p4_sft, format_prompt(PREF_VAL[0]["prompt"]), 30)))
print("  DPO:", repr(generate(p5_pol, format_prompt(PREF_VAL[0]["prompt"]), 30)))
RESULTS["p5_real"] = dict(before_train=round(p5_before[0][0], 3), after_train=round(p5_after[0][0], 3), before_val=round(p5_before[1][0], 3), after_val=round(p5_after[1][0], 3))

You should see the log-probabilities of the real model move. Before training the policy equals the reference, so the margin is exactly 0.000 and 0 percent of the pairs have a positive margin. The mean DPO loss is 1.0247 in the first epoch and 0.3480 in the second (15 seconds per epoch). After 28 steps the mean implied margin on the training pairs is **2.155** (100 percent of the pairs positive) and on the 6 validation pairs **0.693** (100 percent positive): the policy has learned to prefer the chosen answers, and the preference also reaches unseen pairs, but with a third of the strength (0.693 against 2.155). The mean log-probability of the chosen answers **rose** from -95.5 to -75.7 on the training pairs and from -99.6 to -89.2 on the validation pairs. (In many runs the chosen log-probability falls while the margin grows, because DPO only pushes the *difference*. Here it rose, since the chosen answers are close to the style of the SFT model; do not assume it.) The greedy answer to the homework question is unchanged ("Sure! You can focus when you do homework. Hope that helps."): a margin of 2 in the implied reward is a change of the probabilities of whole sentences, and it does not necessarily change the most likely continuation. **Honest summary:** the numbers moved in the right direction on 14 pairs and 28 steps, but nothing in this experiment shows that the model got better at being helpful; to claim that you would need human judgements of its samples.

### ✏️ Reflect: Part 5

Look at your Imagine-first guesses and write two or three sentences each. (1) What does β control, in the closed form and in the frontier plot? (2) In Step 5.3 the true reward peaked and then fell while the proxy reward kept rising: connect this to the word salad of Step 4.6. What is the practical use of the KL leash? (3) DPO reached the same policy as the closed form without a reward model: what did it need instead, and what happened when it had only 10 comparisons (Step 5.5)?

## Part 6. LoRA from scratch and with peft (40 min)

*Lesson sections: "PEFT: fine-tune a small add-on" and "Doing it well".* Full fine-tuning changes every one of the 124 million numbers of GPT-2 and needs a copy of all of them for every task. ==In this part you build the cheap alternative from scratch: LoRA freezes the model and trains two thin matrices beside each attention layer. You write the layer, put it into GPT-2, count what it trains and what it saves, train it against full fine-tuning on the same 24 examples, sweep the rank, merge the adapter back into the weights, check your numbers against the `peft` library, look at the update with the SVD, and measure the size of the file you would share.== Every run uses the same data, the same masked loss and `SEED = 42` as Parts 1 to 3, and the models are retrained inside this part, so it runs on its own. All times are for a CPU and differ between machines.

### ✏️ Imagine first

GPT-2 has 124 million numbers (parameters). Suppose that we freeze all of them and, next to each of the 12 attention layers, add a tiny side path made of two thin matrices whose product is the correction: that is LoRA, Low-Rank Adaptation. Only the thin matrices are trained. Write two predictions before you run anything. **(1)** What percent of GPT-2's parameters does this train (rank 8)? **(2)** After training only those, what share of the 8 validation answers will open with "Sure!", the style of Part 2? You compare with Steps 6.4 and 6.6.

Write your guesses in the starter cell below (replace each `None` or empty string), then run it. Do not look ahead: you compare with the real numbers later in this part.

In [ ]:
# Imagine first: LoRA
my_guess_share = None      # what percent of GPT-2's 124 million parameters does LoRA (rank 8 on the attention layers) train? (a number, for example 5)
my_guess_style = None      # after training only those parameters, what share of the 8 validation answers open with "Sure!"? (0 to 1)

### Step 6.1. The tools: masked examples, validation loss, a training loop

Everything in this part uses the same few tools, defined once here. `p6_example` turns a question and its answer into token ids and labels, with the label -100 on every prompt token (the loss ignores them, so the model learns the answer and not the question, as in Part 1). `p6_val_loss` is the average masked loss on the 8 validation examples. `p6_train` is a plain training loop (AdamW, batch size 1, a shuffled order fixed by the seed, and the same dropout masks for every run, so two runs with the same settings give the same numbers). Before any training the base model has a validation loss and a `style_rate`; we also keep its loss on the general paragraph, the forgetting test of Part 3.

In [ ]:
P6_SEED = 0
P6_EPOCHS = 1 if QUICK else 2          # passes over the training examples for every run in this part
P6_TRAIN = INSTR_TRAIN[:6] if QUICK else INSTR_TRAIN
P6_FULL_LR, P6_LORA_LR = 3e-5, 1e-3     # LoRA needs a much larger learning rate than full fine-tuning (see Step 6.6)

def p6_example(q, a):
    """Token ids and labels for one example: the prompt tokens get the label -100 (ignored by the loss), so only the answer is learned."""
    p = tok(format_prompt(q))["input_ids"]; r = tok(a + tok.eos_token)["input_ids"]
    labels = [-100] * len(p) + r                 # -100 on the prompt tokens, the real token ids on the answer tokens
    return torch.tensor([p + r]).to(device), torch.tensor([labels]).to(device)

P6_TRAIN_EX = [p6_example(q, a) for q, a in P6_TRAIN]
P6_VAL_EX = [p6_example(q, a) for q, a in INSTR_VAL]

@torch.no_grad()
def p6_val_loss(model):
    """Mean masked loss (nats per answer token) on the 8 validation examples."""
    model.eval()
    return float(np.mean([float(model(input_ids=i, labels=l).loss) for i, l in P6_VAL_EX]))

def p6_train(model, params, lr, epochs=P6_EPOCHS):
    """Plain AdamW, batch size 1, in a shuffled order that depends only on P6_SEED. Returns (seconds, val loss after each epoch, the optimizer)."""
    torch.manual_seed(P6_SEED)              # the same dropout masks for every run
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    t0 = time.time(); val = []
    for ep in range(epochs):
        model.train()
        order = list(range(len(P6_TRAIN_EX))); random.Random(P6_SEED + ep).shuffle(order)
        for j in order:
            i, l = P6_TRAIN_EX[j]
            model(input_ids=i, labels=l).loss.backward(); opt.step(); opt.zero_grad()
        val.append(p6_val_loss(model))
    return time.time() - t0, val, opt

RESULTS["p6_base_val"] = round(p6_val_loss(base_model), 3)
RESULTS["p6_base_style"] = style_rate(base_model)
RESULTS["p6_base_general"] = round(text_loss(base_model, GENERAL_TEXT), 3)
print("base model: val loss %.3f, style_rate %.2f, loss on the general paragraph %.3f" % (RESULTS["p6_base_val"], RESULTS["p6_base_style"], RESULTS["p6_base_general"]))
print("training examples used: %d, epochs per run: %d" % (len(P6_TRAIN_EX), P6_EPOCHS))

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The cell prints `base model: val loss 2.816, style_rate 0.00, loss on the general paragraph 3.509` and `training examples used: 24, epochs per run: 2`.

*What it means.* These three numbers are the starting line for the whole part: a validation loss of 2.816 nats per answer token (GPT-2 is quite surprised by the answers), a style_rate of 0.0 (it never opens with "Sure!") and a general-text loss of 3.509. Training should lower the first, raise the second and not move the third much.

### Step 6.2. How many numbers does LoRA train?

LoRA leaves a weight matrix W alone and learns a correction of the same shape as the product of two thin matrices: ΔW is A times B, where A has shape (d_in, r) and B has shape (r, d_out). The small number r is the **rank**. A layer that maps d_in numbers to d_out numbers has d_in times d_out weights; LoRA adds only r times (d_in + d_out). GPT-2's attention block has one big layer, `c_attn`, that produces the queries, keys and values in one go: it maps 768 numbers to 3 times 768 = 2,304 and exists in each of the 12 blocks. We adapt only these 12 layers.

### ✏️ Your turn: How many numbers does LoRA train?

*Exercise 6.2:* Write `lora_params(r, d_in, d_out)`, which returns the number of trainable numbers LoRA adds to one layer, and use it to fill in `per_layer` (rank 8 on one `c_attn`), `total_lora` (all 12 layers) and `share_pct` (the total as a percent of `count_params(base_model)`). Why it matters: this is the number behind every claim that LoRA is cheap, and it is the first number your answer to the fifth assessed question needs. The check tests all four values.

Write your answer in the starter cell below, then run the check cell under it.

In [ ]:
def lora_params(r, d_in, d_out):
    """Number of trainable numbers LoRA adds to ONE linear layer of shape d_in -> d_out."""
    return None   # YOUR CODE: A has shape (d_in, r) and B has shape (r, d_out)

D_IN, D_OUT, LAYERS = 768, 2304, 12        # GPT-2 small: c_attn maps 768 numbers to 3 x 768 = 2,304 (queries, keys, values), in 12 blocks
per_layer = total_lora = share_pct = None   # YOUR CODE: r = 8: per layer, for all 12 layers, and as a percent of count_params(base_model)

In [ ]:
if lora_params(8, D_IN, D_OUT) is None or total_lora is None or share_pct is None:
    print("Fill in lora_params, per_layer, total_lora and share_pct above, then run this cell again.")
else:
    check("lora_params(8, 768, 2304) is 24,576", lora_params(8, D_IN, D_OUT) == 24576, "r * (d_in + d_out) = 8 * (768 + 2304)")
    check("per_layer is 24,576 and total_lora is 294,912", per_layer == 24576 and total_lora == 294912, "multiply by the 12 layers")
    check("share_pct is about 0.237", abs(share_pct - 100 * 294912 / count_params(base_model)) < 1e-6, "100 * total_lora / count_params(base_model)")

<details><summary>Show a possible answer</summary>

```python
def lora_params(r, d_in, d_out):
    return r * d_in + r * d_out          # A: d_in x r, B: r x d_out

D_IN, D_OUT, LAYERS = 768, 2304, 12
per_layer = lora_params(8, D_IN, D_OUT)
total_lora = LAYERS * per_layer
share_pct = 100 * total_lora / count_params(base_model)
full_layer = D_IN * D_OUT + D_OUT                     # the weight matrix and the bias of one c_attn
print("one layer: %d LoRA numbers against %d in c_attn itself (%.2f percent)" % (per_layer, full_layer, 100 * per_layer / full_layer))
print("all %d layers: %d LoRA numbers = %.3f percent of the %d parameters of GPT-2" % (LAYERS, total_lora, share_pct, count_params(base_model)))
assert per_layer == 24576 and total_lora == 294912
print("rank  parameters")
for r_ in (1, 2, 4, 8, 16): print("%4d  %10d" % (r_, LAYERS * lora_params(r_, D_IN, D_OUT)))
RESULTS["p6_lora_params"] = total_lora; RESULTS["p6_lora_share_pct"] = round(share_pct, 3)
```

One `c_attn` has 768 × 2,304 + 2,304 = 1,771,776 numbers; the LoRA path for it has only 24,576 (A: 768 × 8 = 6,144 and B: 8 × 2,304 = 18,432), under 1.4 percent of the layer. Twelve layers give 294,912, which is 0.237 percent of GPT-2. The cost grows in a straight line with r: doubling the rank doubles the parameters.

</details>

Below the exercise we run our version of the code above, so that the next cells work even if you skipped the exercise, and then the check on it.

In [ ]:
def lora_params(r, d_in, d_out):
    return r * d_in + r * d_out          # A: d_in x r, B: r x d_out

D_IN, D_OUT, LAYERS = 768, 2304, 12
per_layer = lora_params(8, D_IN, D_OUT)
total_lora = LAYERS * per_layer
share_pct = 100 * total_lora / count_params(base_model)
full_layer = D_IN * D_OUT + D_OUT                     # the weight matrix and the bias of one c_attn
print("one layer: %d LoRA numbers against %d in c_attn itself (%.2f percent)" % (per_layer, full_layer, 100 * per_layer / full_layer))
print("all %d layers: %d LoRA numbers = %.3f percent of the %d parameters of GPT-2" % (LAYERS, total_lora, share_pct, count_params(base_model)))
assert per_layer == 24576 and total_lora == 294912
print("rank  parameters")
for r_ in (1, 2, 4, 8, 16): print("%4d  %10d" % (r_, LAYERS * lora_params(r_, D_IN, D_OUT)))
RESULTS["p6_lora_params"] = total_lora; RESULTS["p6_lora_share_pct"] = round(share_pct, 3)

In [ ]:
if lora_params(8, D_IN, D_OUT) is None or total_lora is None or share_pct is None:
    print("Fill in lora_params, per_layer, total_lora and share_pct above, then run this cell again.")
else:
    check("lora_params(8, 768, 2304) is 24,576", lora_params(8, D_IN, D_OUT) == 24576, "r * (d_in + d_out) = 8 * (768 + 2304)")
    check("per_layer is 24,576 and total_lora is 294,912", per_layer == 24576 and total_lora == 294912, "multiply by the 12 layers")
    check("share_pct is about 0.237", abs(share_pct - 100 * 294912 / count_params(base_model)) < 1e-6, "100 * total_lora / count_params(base_model)")

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The check passes three ticks. The answer prints `one layer: 24576 LoRA numbers against 1771776 in c_attn itself (1.39 percent)`, then `all 12 layers: 294912 LoRA numbers = 0.237 percent of the 124439808 parameters of GPT-2`, and a table of the parameters for the ranks 1, 2, 4, 8 and 16: 36,864, 73,728, 147,456, 294,912 and 589,824.

*What it means.* With the default rank 8 we train about one number in 420, and with rank 1 about one in 3,400. Whether so few numbers can be enough is the question of the rest of this part. `RESULTS["p6_lora_params"]` is 294912 and `RESULTS["p6_lora_share_pct"]` is 0.237.

### Step 6.3. The LoRA layer

The LoRA layer wraps the original layer and adds the side path: y = base(x) + (α / r) · (x A) B. Three details matter. **(1) B starts at zero**, so the side path adds exactly nothing at the start and the model begins as the unchanged GPT-2; A starts as small random numbers (standard deviation 0.02), because if both started at zero the gradients would be zero too and nothing would ever move. **(2) The scale α / r**, with α a number you choose (we use α = 2r, so the scale is 2), keeps the size of the update comparable when you change r. **(3) GPT-2 stores its weights in a `Conv1D` layer**, whose weight has the shape (in, out), the opposite of `nn.Linear`. That is why A is `(d_in, r)` and B is `(r, d_out)` here, and why you read the shape from `base.weight.shape` and do not guess it.

### ✏️ Your turn: Write the LoRA layer

*Exercise 6.3:* Complete `LoRALinearMine`: create `A` (a `nn.Parameter` of shape (d_in, r) with random numbers of standard deviation 0.02) and `B` (a `nn.Parameter` of shape (r, d_out) of zeros), and let `forward` return the base output plus `self.scale` times `(x @ A) @ B`. Why it matters: if you get a shape wrong (the usual mistake is to read the Conv1D weight as (out, in)) nothing works, and if B does not start at zero your model is damaged before it is trained. The check tries your layer on a toy `Conv1D`.

Write your answer in the starter cell below, then run the check cell under it.

In [ ]:
from transformers.pytorch_utils import Conv1D

class LoRALinearMine(nn.Module):
    """A frozen linear layer plus a trainable low-rank update: y = base(x) + (alpha / r) * x A B."""
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        d_in, d_out = base.weight.shape              # careful: GPT-2's Conv1D stores W with shape (in, out), the opposite of nn.Linear
        self.A = None      # YOUR CODE: nn.Parameter of shape (d_in, r) filled with random numbers of standard deviation 0.02
        self.B = None      # YOUR CODE: nn.Parameter of shape (r, d_out) filled with zeros
    def forward(self, x):
        return None        # YOUR CODE: the base output plus self.scale times (x @ A) @ B

In [ ]:
toy_base = Conv1D(6, 4)                       # Conv1D(nf = 6 outputs, nx = 4 inputs): weight shape (4, 6)
toy = LoRALinearMine(toy_base, r=2, alpha=4)
if toy.A is None or toy.B is None or toy(torch.randn(3, 4)) is None:
    print("Fill in LoRALinearMine above, then run this cell again.")
else:
    x_ = torch.randn(3, 4)
    check("A has shape (4, 2) and B has shape (2, 6)", tuple(toy.A.shape) == (4, 2) and tuple(toy.B.shape) == (2, 6), "d_in x r and r x d_out; for a Conv1D, base.weight.shape is (in, out)")
    check("B starts at zero, so the output equals the base at the start", float(toy.B.abs().sum()) == 0 and torch.allclose(toy(x_), toy_base(x_)), "B = torch.zeros(r, d_out)")
    toy.B.data = torch.randn(2, 6)
    check("with B not zero: y = base(x) + 2 * x A B  (alpha / r = 4 / 2 = 2)", torch.allclose(toy(x_), toy_base(x_) + 2 * (x_ @ toy.A @ toy.B), atol=1e-5), "the scale is alpha / r")
    check("A and B require gradients", toy.A.requires_grad and toy.B.requires_grad, "wrap them in nn.Parameter")

<details><summary>Show a possible answer</summary>

```python
from transformers.pytorch_utils import Conv1D

class LoRALinearMine(nn.Module):
    """A frozen linear layer plus a trainable low-rank update: y = base(x) + (alpha / r) * x A B."""
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        d_in, d_out = base.weight.shape
        self.A = nn.Parameter(torch.randn(d_in, r) * 0.02)      # small random numbers
        self.B = nn.Parameter(torch.zeros(r, d_out))            # zeros: the update starts at exactly zero
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A) @ self.B

LoRALinear = LoRALinearMine                          # the name used by the rest of this part
torch.manual_seed(0)
layer0 = base_model.transformer.h[0].attn.c_attn     # the real c_attn of block 0: a Conv1D with weight shape (768, 2304)
wrapped = LoRALinear(layer0, r=8, alpha=16)
x_ = torch.randn(2, 5, 768)
with torch.no_grad(): diff0 = float((wrapped(x_) - layer0(x_)).abs().max())
print("c_attn weight shape:", tuple(layer0.weight.shape), "  A:", tuple(wrapped.A.shape), "  B:", tuple(wrapped.B.shape))
print("largest difference from the base layer at the start: %.1f" % diff0)
with torch.no_grad(): wrapped.B.normal_(0, 0.02); diff1 = float((wrapped(x_) - layer0(x_)).abs().max())
print("largest difference after giving B random numbers: %.4f" % diff1)
assert diff0 == 0.0 and diff1 > 0
RESULTS["p6_diff_at_init"] = diff0; RESULTS["p6_diff_after_B"] = round(diff1, 3)
```

The base layer is called as it is (it does its own matrix product and adds its own bias), and the LoRA path is added on top. Writing `(x @ self.A) @ self.B` first multiplies by the thin matrix A, so the largest intermediate result has only r columns: that is why LoRA is cheap in time as well as in memory.

</details>

Below the exercise we run our version of the code above, so that the next cells work even if you skipped the exercise, and then the check on it.

In [ ]:
from transformers.pytorch_utils import Conv1D

class LoRALinearMine(nn.Module):
    """A frozen linear layer plus a trainable low-rank update: y = base(x) + (alpha / r) * x A B."""
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        d_in, d_out = base.weight.shape
        self.A = nn.Parameter(torch.randn(d_in, r) * 0.02)      # small random numbers
        self.B = nn.Parameter(torch.zeros(r, d_out))            # zeros: the update starts at exactly zero
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A) @ self.B

LoRALinear = LoRALinearMine                          # the name used by the rest of this part
torch.manual_seed(0)
layer0 = base_model.transformer.h[0].attn.c_attn     # the real c_attn of block 0: a Conv1D with weight shape (768, 2304)
wrapped = LoRALinear(layer0, r=8, alpha=16)
x_ = torch.randn(2, 5, 768)
with torch.no_grad(): diff0 = float((wrapped(x_) - layer0(x_)).abs().max())
print("c_attn weight shape:", tuple(layer0.weight.shape), "  A:", tuple(wrapped.A.shape), "  B:", tuple(wrapped.B.shape))
print("largest difference from the base layer at the start: %.1f" % diff0)
with torch.no_grad(): wrapped.B.normal_(0, 0.02); diff1 = float((wrapped(x_) - layer0(x_)).abs().max())
print("largest difference after giving B random numbers: %.4f" % diff1)
assert diff0 == 0.0 and diff1 > 0
RESULTS["p6_diff_at_init"] = diff0; RESULTS["p6_diff_after_B"] = round(diff1, 3)

In [ ]:
toy_base = Conv1D(6, 4)                       # Conv1D(nf = 6 outputs, nx = 4 inputs): weight shape (4, 6)
toy = LoRALinearMine(toy_base, r=2, alpha=4)
if toy.A is None or toy.B is None or toy(torch.randn(3, 4)) is None:
    print("Fill in LoRALinearMine above, then run this cell again.")
else:
    x_ = torch.randn(3, 4)
    check("A has shape (4, 2) and B has shape (2, 6)", tuple(toy.A.shape) == (4, 2) and tuple(toy.B.shape) == (2, 6), "d_in x r and r x d_out; for a Conv1D, base.weight.shape is (in, out)")
    check("B starts at zero, so the output equals the base at the start", float(toy.B.abs().sum()) == 0 and torch.allclose(toy(x_), toy_base(x_)), "B = torch.zeros(r, d_out)")
    toy.B.data = torch.randn(2, 6)
    check("with B not zero: y = base(x) + 2 * x A B  (alpha / r = 4 / 2 = 2)", torch.allclose(toy(x_), toy_base(x_) + 2 * (x_ @ toy.A @ toy.B), atol=1e-5), "the scale is alpha / r")
    check("A and B require gradients", toy.A.requires_grad and toy.B.requires_grad, "wrap them in nn.Parameter")

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The check passes four ticks. The answer prints `c_attn weight shape: (768, 2304)   A: (768, 8)   B: (8, 2304)`, `largest difference from the base layer at the start: 0.0` and, after we give B random numbers, `largest difference after giving B random numbers: 0.36`. The assertion passes.

*What it means.* Because B is zero, your layer is **exactly** the base layer at the start (difference 0.0, not just small). This is the property that makes LoRA safe: training begins from the model you trust, and only moves as far as the data pulls it. `RESULTS["p6_diff_at_init"]` is 0.0.

### Step 6.4. Put LoRA into GPT-2

Now put the layer into GPT-2. `add_lora` first freezes every parameter (`requires_grad = False`), then replaces the `c_attn` of each of the 12 blocks by a `LoRALinear`. The new A and B matrices are created after the freeze, so they are the only trainable parameters. We then count them, and compare the logits of the new model with those of the original on a prompt: they must be identical, because every B is zero.

In [ ]:
def add_lora(model, r=8, alpha=16):
    """Freeze every parameter of `model`, then wrap the c_attn layer of every block in a LoRALinear. Returns the list of trainable parameters."""
    for p in model.parameters(): p.requires_grad = False
    for blk in model.transformer.h:
        blk.attn.c_attn = LoRALinear(blk.attn.c_attn, r, alpha)     # the new A and B are created trainable
    return [p for p in model.parameters() if p.requires_grad]

torch.manual_seed(P6_SEED)
lora_model = copy.deepcopy(base_model).to(device)
lora_params_list = add_lora(lora_model, r=8, alpha=16)
n_train = count_params(lora_model, trainable_only=True)
print("trainable: %d of %d parameters (%.3f percent)" % (n_train, count_params(lora_model), 100 * n_train / count_params(lora_model)))
assert n_train == 294912                                              # the number of Step 6.2
ids_ = tok(format_prompt("What is the capital of France?"), return_tensors="pt").to(device)
with torch.no_grad(): same = float((lora_model.eval()(**ids_).logits - base_model(**ids_).logits).abs().max())
print("largest difference between the logits of the LoRA model and of GPT-2 before training: %.1f" % same)
assert same == 0.0
RESULTS["p6_trainable"] = n_train; RESULTS["p6_total"] = count_params(base_model)

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The cell prints `trainable: 294912 of 124734720 parameters (0.236 percent)` (the total now includes the 294,912 new numbers) and `largest difference between the logits of the LoRA model and of GPT-2 before training: 0.0`; both assertions pass. `RESULTS["p6_trainable"]` is 294912.

*What it means.* Compare with your Imagine-first guess: 294,912 trainable numbers is about a quarter of one percent (0.237 percent of the original 124,439,808). Almost everybody guesses a few percent, because "a small add-on" sounds bigger than it is. The difference 0.0 in the logits confirms Step 6.3 on the whole model.

### Step 6.5. What LoRA saves: memory arithmetic

Before we train, count the memory. Training with the Adam optimizer in float32 (4 bytes per number) must keep, for every parameter, its **weight** (4 bytes); for every *trainable* parameter also its **gradient** (4 bytes) and Adam's two running statistics (4 + 4 bytes), so 12 bytes more. A frozen weight costs only its 4 bytes. This ignores the activations (they depend on the batch size and are the same for both methods), so the real total is higher, but the comparison holds. In 16-bit "mixed precision" with a master copy the factor is 16 bytes per trainable parameter, as in the lesson: the idea is the same.

### ✏️ Your turn: Memory arithmetic

*Exercise 6.5:* Write `train_bytes(n_total, n_trainable)` and use it to fill in `full_gb` (full fine-tuning: all 124,439,808 parameters are trainable) and `lora_gb` (LoRA: 294,912 trainable out of the same total), in gigabytes (1 GB is 1e9 bytes). Why it matters: memory, not speed, is what stops most people from fine-tuning a large model, and this formula tells you in advance whether a run will fit. The check tests the formula and both numbers.

Write your answer in the starter cell below, then run the check cell under it.

In [ ]:
def train_bytes(n_total, n_trainable):
    """Memory for the numbers that training has to keep (float32, 4 bytes each), without the activations."""
    return None   # YOUR CODE: weights of ALL parameters (4 bytes each) + for the TRAINABLE ones a gradient (4 bytes) and the two Adam statistics (8 bytes)

full_gb = lora_gb = None      # YOUR CODE: in gigabytes (1 GB = 1e9 bytes) for full fine-tuning (all 124,439,808 trainable) and for LoRA (294,912 trainable)

In [ ]:
if train_bytes(10, 10) is None or full_gb is None or lora_gb is None:
    print("Fill in train_bytes, full_gb and lora_gb above, then run this cell again.")
else:
    check("train_bytes(10, 10) is 160 and train_bytes(10, 0) is 40", train_bytes(10, 10) == 160 and train_bytes(10, 0) == 40, "4 * n_total + 12 * n_trainable")
    check("full fine-tuning needs about 1.99 GB", abs(full_gb - 1.991) < 0.01, "16 bytes for each of the 124,439,808 parameters")
    check("LoRA needs about 0.50 GB", abs(lora_gb - 0.501) < 0.01, "4 bytes for every parameter plus 12 for the 294,912 trainable ones")

<details><summary>Show a possible answer</summary>

```python
def train_bytes(n_total, n_trainable):
    return 4 * n_total + 12 * n_trainable          # weights + (gradient + two Adam numbers) for the trainable ones

N_ALL = count_params(base_model)
full_gb = train_bytes(N_ALL, N_ALL) / 1e9
lora_gb = train_bytes(N_ALL, 294912) / 1e9
print("full fine-tuning: %.3f GB    LoRA r = 8: %.3f GB    ratio %.1f" % (full_gb, lora_gb, full_gb / lora_gb))
print("the part that depends on training: gradients + Adam = %.1f MB for full fine-tuning and %.2f MB for LoRA" % (12 * N_ALL / 1e6, 12 * 294912 / 1e6))
RESULTS["p6_full_gb"] = round(full_gb, 3); RESULTS["p6_lora_gb"] = round(lora_gb, 3)
```

The LoRA run still has to keep all the weights, so it does not use 420 times less memory: the 4 bytes per frozen weight (0.5 GB) stay. What disappears is the 12 bytes per parameter of gradients and Adam statistics (1.49 GB for full fine-tuning, 3.5 MB for LoRA). The saving is therefore about a factor of 4 for a small model whose weights are in float32. For a big model whose frozen weights are stored in 4 bits (QLoRA, see the lesson) the saving is much larger.

</details>

Below the exercise we run our version of the code above, so that the next cells work even if you skipped the exercise, and then the check on it.

In [ ]:
def train_bytes(n_total, n_trainable):
    return 4 * n_total + 12 * n_trainable          # weights + (gradient + two Adam numbers) for the trainable ones

N_ALL = count_params(base_model)
full_gb = train_bytes(N_ALL, N_ALL) / 1e9
lora_gb = train_bytes(N_ALL, 294912) / 1e9
print("full fine-tuning: %.3f GB    LoRA r = 8: %.3f GB    ratio %.1f" % (full_gb, lora_gb, full_gb / lora_gb))
print("the part that depends on training: gradients + Adam = %.1f MB for full fine-tuning and %.2f MB for LoRA" % (12 * N_ALL / 1e6, 12 * 294912 / 1e6))
RESULTS["p6_full_gb"] = round(full_gb, 3); RESULTS["p6_lora_gb"] = round(lora_gb, 3)

In [ ]:
if train_bytes(10, 10) is None or full_gb is None or lora_gb is None:
    print("Fill in train_bytes, full_gb and lora_gb above, then run this cell again.")
else:
    check("train_bytes(10, 10) is 160 and train_bytes(10, 0) is 40", train_bytes(10, 10) == 160 and train_bytes(10, 0) == 40, "4 * n_total + 12 * n_trainable")
    check("full fine-tuning needs about 1.99 GB", abs(full_gb - 1.991) < 0.01, "16 bytes for each of the 124,439,808 parameters")
    check("LoRA needs about 0.50 GB", abs(lora_gb - 0.501) < 0.01, "4 bytes for every parameter plus 12 for the 294,912 trainable ones")

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The check passes three ticks. The answer prints `full fine-tuning: 1.991 GB    LoRA r = 8: 0.501 GB    ratio 4.0` and `the part that depends on training: gradients + Adam = 1493.3 MB for full fine-tuning and 3.54 MB for LoRA`.

*What it means.* Full fine-tuning of GPT-2 small needs about 2 GB for these numbers, LoRA about 0.5 GB. Both fit easily on a laptop; for a 7-billion-parameter model the same formula gives 112 GB against about 28 GB, and that is the difference between a rack of GPUs and one card. In the next step we also *measure* the optimizer's memory. `RESULTS["p6_full_gb"]` is 1.991 and `RESULTS["p6_lora_gb"]` is 0.501.

### Step 6.6. Train both and compare

Now we train both on the 24 training examples for 2 epochs with the loop of Step 6.1. **Full fine-tuning** uses the learning rate of Part 2 (3e-5). **LoRA** uses 1e-3, more than thirty times larger: the thin matrices start from nothing and must move a long way in only 48 steps, and the lesson's typical 1e-4 to 2e-4 is for runs of thousands of steps. (Using 3e-5 for LoRA is the most common mistake: the loss hardly moves and it looks as if LoRA does not work.) We report the validation loss, the `style_rate`, the loss on the general paragraph (forgetting), the time, and the measured size of Adam's statistics (the optimizer keeps two tensors per trainable parameter). A second cell shows what each model answers to the first validation question.

In [ ]:
def p6_opt_state_mb(opt):
    """Measured size of the two Adam statistics kept by the optimizer, in megabytes."""
    return sum(v.numel() * 4 for st in opt.state.values() for k, v in st.items() if k in ("exp_avg", "exp_avg_sq")) / 1e6

# --- full fine-tuning (every parameter trains), lr 3e-5
torch.manual_seed(P6_SEED)
full_model = copy.deepcopy(base_model).to(device)
for p in full_model.parameters(): p.requires_grad = True
sec_full, val_full, opt_full = p6_train(full_model, list(full_model.parameters()), P6_FULL_LR)
# --- LoRA r = 8 on c_attn, lr 1e-3 (lora_model was built in Step 6.4)
sec_lora, val_lora, opt_lora = p6_train(lora_model, lora_params_list, P6_LORA_LR)

gen_full, gen_lora = text_loss(full_model, GENERAL_TEXT), text_loss(lora_model, GENERAL_TEXT)
st_full, st_lora = style_rate(full_model), style_rate(lora_model)
print("%-22s %12s %8s %12s %10s %12s %14s" % ("method", "trainable", "seconds", "val loss", "style", "general text", "Adam state MB"))
print("%-22s %12s %8s %12.3f %10.2f %12.3f %14s" % ("GPT-2, no training", "0", "-", RESULTS["p6_base_val"], RESULTS["p6_base_style"], RESULTS["p6_base_general"], "-"))
print("%-22s %12d %8.0f %12.3f %10.2f %12.3f %14.1f" % ("full fine-tuning", count_params(full_model, True), sec_full, val_full[-1], st_full, gen_full, p6_opt_state_mb(opt_full)))
print("%-22s %12d %8.0f %12.3f %10.2f %12.3f %14.2f" % ("LoRA r = 8", count_params(lora_model, True), sec_lora, val_lora[-1], st_lora, gen_lora, p6_opt_state_mb(opt_lora)))
print("val loss after each epoch:  full", [round(v, 3) for v in val_full], "  LoRA", [round(v, 3) for v in val_lora])
RESULTS.update(p6_full_val=round(val_full[-1], 3), p6_lora_val=round(val_lora[-1], 3), p6_full_sec=round(sec_full), p6_lora_sec=round(sec_lora),
               p6_full_style=st_full, p6_lora_style=st_lora, p6_full_general=round(gen_full, 3), p6_lora_general=round(gen_lora, 3),
               p6_full_adam_mb=round(p6_opt_state_mb(opt_full), 1), p6_lora_adam_mb=round(p6_opt_state_mb(opt_lora), 2),
               p6_full_val_by_epoch=[round(v, 3) for v in val_full], p6_lora_val_by_epoch=[round(v, 3) for v in val_lora])

In [ ]:
q_, _ = INSTR_VAL[0]
print("Question:", q_)
for name, m in (("GPT-2 (no training)", base_model), ("full fine-tuning", full_model), ("LoRA r = 8", lora_model)):
    print("%-20s -> %r" % (name, generate(m, format_prompt(q_), 30)))
for name, m, key in (("full fine-tuning", full_model, "full"), ("LoRA r = 8", lora_model, "lora")):          # style is easy; are the facts right too?
    exact = sum(generate(m, format_prompt(q), 40).strip() == a for q, a in INSTR_VAL)
    print("%-20s answers identical to the reference on the 8 validation questions: %d" % (name, exact)); RESULTS["p6_%s_exact" % key] = exact

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The table has three rows. Without training: validation loss 2.816, style 0.00, general text 3.509. Full fine-tuning: all 124,439,808 numbers trainable, 66 s, validation loss 0.193, style 1.00, general text 3.620, Adam state 995.5 MB. LoRA: 294,912 trainable, 16 s, validation loss 0.270, style 1.00, general text 3.574, Adam state 2.36 MB. The validation loss after epoch 1 and 2 is [0.267, 0.193] for full fine-tuning and [0.447, 0.27] for LoRA. The second cell shows the base model rambling on (`### Question: ...`) and both trained models answering in the style "Sure! ... Hope that helps.", and counts how many of the 8 validation answers are word for word the reference answer: 4 for full fine-tuning and 3 for LoRA.

*What it means.* Be honest about what this says. **Both methods learn the style completely** (style_rate 1.0 and 1.0, up from 0.0). **Full fine-tuning fits the validation answers better** (0.193 against 0.270): training about 420 times more numbers buys a lower loss on this data, so LoRA is cheaper, not free. LoRA ran about 4.1 times faster here and its optimizer state is 2.36 MB instead of 995.5 MB, which matches the arithmetic of Step 6.5 (8 bytes for each trainable parameter). **The facts are another matter**: the style is perfect for both, but only 4 (full) and 3 (LoRA) of the 8 answers are exactly right, and in the example LoRA writes "six millilitres in a litre". Fine-tuning on 24 examples teaches a style, not new knowledge, and a 124-million-parameter model gets some facts wrong. Both observations are about one run on 8 questions, so treat the 4 against 3 as a tie. On forgetting, the loss on the general paragraph rose by 0.111 for full fine-tuning and by 0.065 for LoRA: LoRA forgot a little less, but both changes are small with only 48 updates and one paragraph, so do not read this as proof that LoRA never forgets. (Times depend on the machine and on what else it is running.)

### Step 6.7. How much rank is enough?

The rank r is the main choice. Too small and the correction cannot express what the data needs; larger costs more parameters, and past some point buys nothing. We train five fresh models with r = 1, 2, 4, 8 and 16 (α = 2r, so the scale stays 2), with identical data, order and seed, and compare validation loss, style and forgetting. The r = 8 run repeats Step 6.6 exactly; the assertion checks that it gives the same number again (with the same seed, a run is repeatable).

In [ ]:
def p6_run_lora(r, alpha=None, lr=P6_LORA_LR):
    """Train a fresh LoRA model of rank r (alpha = 2 r, so alpha / r = 2 for every r). Returns the model and its numbers."""
    torch.manual_seed(P6_SEED)
    m = copy.deepcopy(base_model).to(device)
    params = add_lora(m, r, alpha or 2 * r)
    sec, val, _ = p6_train(m, params, lr)
    return m, dict(r=r, params=count_params(m, True), sec=sec, val=val[-1], style=style_rate(m, INSTR_VAL[:2] if QUICK else None), general=text_loss(m, GENERAL_TEXT))

sweep = {}
print("%4s %10s %8s %10s %7s %14s" % ("r", "parameters", "seconds", "val loss", "style", "general text"))
for r_ in (1, 2, 4, 8, 16):
    m_, row = p6_run_lora(r_); sweep[r_] = row
    print("%4d %10d %8.0f %10.3f %7.2f %14.3f" % (r_, row["params"], row["sec"], row["val"], row["style"], row["general"]))
    if r_ == 8: lora8_model = m_            # keep it for the next steps
    del m_; gc.collect()
print("the r = 8 run repeats Step 6.6: val loss %.3f against %.3f there" % (sweep[8]["val"], RESULTS["p6_lora_val"]))
assert abs(sweep[8]["val"] - RESULTS["p6_lora_val"]) < 2e-3          # same seed, same order, same result
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
rs = list(sweep)
ax[0].plot([sweep[r]["params"] for r in rs], [sweep[r]["val"] for r in rs], "o-"); ax[0].set_xscale("log")
for r in rs: ax[0].annotate("r=%d" % r, (sweep[r]["params"], sweep[r]["val"]), textcoords="offset points", xytext=(4, 4))
ax[0].axhline(RESULTS["p6_full_val"], color="gray", ls="--", label="full fine-tuning"); ax[0].set_xlabel("trainable parameters"); ax[0].set_ylabel("validation loss"); ax[0].legend()
ax[1].plot(rs, [sweep[r]["style"] for r in rs], "o-"); ax[1].set_xscale("log", base=2); ax[1].set_xlabel("rank r"); ax[1].set_ylabel("style_rate"); ax[1].set_ylim(-0.05, 1.05)
plt.tight_layout(); plt.show()
RESULTS["p6_sweep_val"] = {r: round(sweep[r]["val"], 3) for r in rs}; RESULTS["p6_sweep_style"] = {r: sweep[r]["style"] for r in rs}
RESULTS["p6_sweep_params"] = {r: sweep[r]["params"] for r in rs}; RESULTS["p6_sweep_general"] = {r: round(sweep[r]["general"], 3) for r in rs}

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The table shows, for r = 1, 2, 4, 8, 16: parameters 36,864, 73,728, 147,456, 294,912 and 589,824; validation loss 0.525, 0.334, 0.299, 0.270 and 0.310; the style_rate is 1.00 for all five; the general-text loss is 3.552, 3.544, 3.578, 3.574 and 3.586. The line `the r = 8 run repeats Step 6.6` shows the same validation loss twice. The figure plots validation loss against the number of trainable parameters (with the full fine-tuning level as a dashed line) and style_rate against r.

*What it means.* Three honest readings. **(1) The style is learned even by rank 1** (36,864 numbers, style 1.00): copying one fixed opening and closing is an easy change. **(2) The validation loss falls steeply from r = 1 to r = 4** (0.525 to 0.299) and then flattens. **(3) r = 16 is *worse* than r = 8** (0.310 against 0.270) although it has twice the parameters, and no rank reaches the full fine-tuning level of 0.193. We have one seed and 8 validation examples, so a difference of 0.04 is within the noise of this experiment; what we can claim is the shape (a steep gain, then a plateau), not the exact winner. A bigger rank is not automatically better with this little data and this few steps.

### Step 6.8. Merge the adapter into the weights

A LoRA model has extra layers to run. For deployment you can remove them: because (x A) B scaled by α / r is the same as x multiplied by (A B scaled by α / r), you can **merge** the update into the weight, W' = W + (α / r) · A B, and use the original architecture with new numbers. The merged model is exactly the same function, with no extra computation at answer time.

### ✏️ Your turn: Merge the adapter

*Exercise 6.8:* Write `merged_weight(W, A, B, scale)` that returns the merged matrix. Then the cell below uses it to build a copy of GPT-2 whose `c_attn` weights are merged (the bias does not change) and compares it with the LoRA model. Why it matters: this is how an adapter becomes an ordinary model that any tool can load. The check tests your function on a toy example.

Write your answer in the starter cell below, then run the check cell under it.

In [ ]:
def merged_weight(W, A, B, scale):
    """The weight matrix of the merged layer: the frozen W plus the update that LoRA learned."""
    return None   # YOUR CODE: W + scale * (A @ B)   (W is (d_in, d_out), A is (d_in, r), B is (r, d_out))

In [ ]:
W_, A_, B_ = torch.randn(5, 7), torch.randn(5, 2), torch.randn(2, 7)
if merged_weight(W_, A_, B_, 2.0) is None:
    print("Fill in merged_weight above, then run this cell again.")
else:
    check("the merged matrix has the shape of W", tuple(merged_weight(W_, A_, B_, 2.0).shape) == (5, 7), "A @ B has shape (d_in, d_out)")
    check("x @ merged equals x @ W + 2 * (x @ A) @ B", torch.allclose(torch.ones(1, 5) @ merged_weight(W_, A_, B_, 2.0), torch.ones(1, 5) @ W_ + 2 * (torch.ones(1, 5) @ A_) @ B_, atol=1e-4), "merged = W + scale * A @ B")

<details><summary>Show a possible answer</summary>

```python
def merged_weight(W, A, B, scale):
    return W + scale * (A @ B)

merged_model = copy.deepcopy(base_model).to(device)
for blk_m, blk_l in zip(merged_model.transformer.h, lora8_model.transformer.h):
    lay = blk_l.attn.c_attn
    with torch.no_grad(): blk_m.attn.c_attn.weight.copy_(merged_weight(lay.base.weight, lay.A, lay.B, lay.scale))     # the bias is unchanged
merged_model.eval(); lora8_model.eval()
ids_ = tok(format_prompt("How many days are in a week?"), return_tensors="pt").to(device)
with torch.no_grad(): diff_logits = float((merged_model(**ids_).logits - lora8_model(**ids_).logits).abs().max())
same_text = [generate(merged_model, format_prompt(q), 30) == generate(lora8_model, format_prompt(q), 30) for q, _ in INSTR_VAL]
print("merged model: %d parameters, %d trainable (no extra layers)" % (count_params(merged_model), count_params(merged_model, True)))
print("largest difference between the logits of the merged and the LoRA model: %.2e" % diff_logits)
print("identical greedy answers on the 8 validation questions: %d of 8;  val loss merged %.3f, LoRA %.3f" % (sum(same_text), p6_val_loss(merged_model), p6_val_loss(lora8_model)))
assert diff_logits < 1e-3 and all(same_text)
RESULTS["p6_merge_diff"] = float("%.2e" % diff_logits); RESULTS["p6_merge_same"] = sum(same_text)
```

The result is a plain GPT-2 with changed numbers: it has 124,439,808 parameters and no extra layers. The logits of the two models differ only by float rounding, because the two ways of computing the same sum add the numbers in a different order.

</details>

Below the exercise we run our version of the code above, so that the next cells work even if you skipped the exercise, and then the check on it.

In [ ]:
def merged_weight(W, A, B, scale):
    return W + scale * (A @ B)

merged_model = copy.deepcopy(base_model).to(device)
for blk_m, blk_l in zip(merged_model.transformer.h, lora8_model.transformer.h):
    lay = blk_l.attn.c_attn
    with torch.no_grad(): blk_m.attn.c_attn.weight.copy_(merged_weight(lay.base.weight, lay.A, lay.B, lay.scale))     # the bias is unchanged
merged_model.eval(); lora8_model.eval()
ids_ = tok(format_prompt("How many days are in a week?"), return_tensors="pt").to(device)
with torch.no_grad(): diff_logits = float((merged_model(**ids_).logits - lora8_model(**ids_).logits).abs().max())
same_text = [generate(merged_model, format_prompt(q), 30) == generate(lora8_model, format_prompt(q), 30) for q, _ in INSTR_VAL]
print("merged model: %d parameters, %d trainable (no extra layers)" % (count_params(merged_model), count_params(merged_model, True)))
print("largest difference between the logits of the merged and the LoRA model: %.2e" % diff_logits)
print("identical greedy answers on the 8 validation questions: %d of 8;  val loss merged %.3f, LoRA %.3f" % (sum(same_text), p6_val_loss(merged_model), p6_val_loss(lora8_model)))
assert diff_logits < 1e-3 and all(same_text)
RESULTS["p6_merge_diff"] = float("%.2e" % diff_logits); RESULTS["p6_merge_same"] = sum(same_text)

In [ ]:
W_, A_, B_ = torch.randn(5, 7), torch.randn(5, 2), torch.randn(2, 7)
if merged_weight(W_, A_, B_, 2.0) is None:
    print("Fill in merged_weight above, then run this cell again.")
else:
    check("the merged matrix has the shape of W", tuple(merged_weight(W_, A_, B_, 2.0).shape) == (5, 7), "A @ B has shape (d_in, d_out)")
    check("x @ merged equals x @ W + 2 * (x @ A) @ B", torch.allclose(torch.ones(1, 5) @ merged_weight(W_, A_, B_, 2.0), torch.ones(1, 5) @ W_ + 2 * (torch.ones(1, 5) @ A_) @ B_, atol=1e-4), "merged = W + scale * A @ B")

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The check passes two ticks. The answer prints `merged model: 124439808 parameters, 124439808 trainable (no extra layers)`, then a largest logit difference of `0.000214`, and `identical greedy answers on the 8 validation questions: 8 of 8` with the same validation loss for both models; the assertion passes.

*What it means.* A difference of 0.000214 in the logits is rounding (the two ways of adding the numbers round differently), not a different model, and the answers are the same words. (The merged copy shows 124,439,808 as trainable because we created it fresh; nothing is frozen in it. It is a normal model.) You can now save one ordinary model, or keep the 1 MB adapter apart and switch tasks: the next steps show why that is useful. `RESULTS["p6_merge_diff"]` is 0.000214.

### Step 6.9. Cross-check with the peft library

In practice you will not write the layer yourself: the `peft` library (Parameter-Efficient Fine-Tuning) does it for any model. We check that our work agrees with it, in two ways. **(1)** `LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"], fan_in_fan_out=True)` builds a peft model whose trainable parameters we count (the argument `fan_in_fan_out=True` tells peft that GPT-2 stores weights as (in, out): the same Conv1D detail as in Step 6.3). **(2)** We copy **our trained** A and B into the peft model (peft stores them transposed) and compare the logits. In Colab run `!pip install peft` first (the setup cell does it); if the library is missing the cell tells you.

In [ ]:
try:
    from peft import LoraConfig, get_peft_model
    cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"], fan_in_fan_out=True, lora_dropout=0.0, bias="none")   # fan_in_fan_out=True because GPT-2 uses Conv1D
    peft_model = get_peft_model(copy.deepcopy(base_model).to(device), cfg)
    n_peft = count_params(peft_model, trainable_only=True)
    print("peft trainable parameters:", n_peft, "  ours:", RESULTS["p6_trainable"])
    assert n_peft == RESULTS["p6_trainable"]
    # copy OUR trained A and B into the peft model (peft stores A as (r, in) and B as (out, r): the transposes of ours)
    for blk_p, blk_o in zip(peft_model.base_model.model.transformer.h, lora8_model.transformer.h):
        with torch.no_grad():
            blk_p.attn.c_attn.lora_A["default"].weight.copy_(blk_o.attn.c_attn.A.T)
            blk_p.attn.c_attn.lora_B["default"].weight.copy_(blk_o.attn.c_attn.B.T)
    peft_model.eval()
    with torch.no_grad(): diff_peft = float((peft_model(**ids_).logits - lora8_model(**ids_).logits).abs().max())
    print("largest difference between peft (with our numbers) and our LoRA model: %.2e" % diff_peft)
    assert diff_peft < 1e-3
    RESULTS["p6_peft_params"] = n_peft; RESULTS["p6_peft_diff"] = float("%.2e" % diff_peft)
    del peft_model; gc.collect()
except ImportError:
    print("peft is not installed here: run  pip install peft  (in Colab:  !pip install peft ) and run this cell again.")

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The cell prints `peft trainable parameters: 294912   ours: 294912` and `largest difference between peft (with our numbers) and our LoRA model: 0.0`; both assertions pass.

*What it means.* The count is identical, 294,912, and with our numbers inside, peft computes the same logits as our layer (difference 0.0). So the library is no magic: it is the layer you wrote, plus conveniences (saving only the adapter, switching adapters, other target layers, dropout, other methods). Which one to use in a project? peft, because it is tested and maintained; but you now know what it does and how to read its errors.

### Step 6.10. Is the update really low rank? A look with the SVD

The idea behind LoRA is that the change a fine-tuning run makes is *low rank*: it needs few directions. The **singular value decomposition (SVD)** of a matrix lists its directions and their strengths (singular values, largest first); the **rank** is the number of non-zero ones, and the **energy** of a direction is its singular value squared. We look at the update ΔW of three layers (blocks 0, 5 and 11) in two models: LoRA's update (A B scaled, which has rank at most 8 by construction) and the update of full fine-tuning (trained weight minus original weight, which is free to have any rank). For each we report the largest singular value, the rank, the number of directions that hold 90 percent of the energy, and the share of the energy in the top 8 directions.

### ✏️ Your turn: How big is the update really?

*Exercise 6.10:* Write `energy_rank(s, frac=0.9)`, which receives singular values `s` (sorted from large to small) and returns the smallest number of them whose squares add up to at least `frac` of the total of the squares. Hint: `torch.cumsum` of `s ** 2`, divided by its sum. Why it matters: it turns "how low rank is this update?" into one number you can compare. The check tests three small cases (the second shows why the comparison must allow for rounding).

Write your answer in the starter cell below, then run the check cell under it.

In [ ]:
def energy_rank(s, frac=0.9):
    """The smallest number of singular values whose squares add up to at least `frac` of the sum of all the squares."""
    return None   # YOUR CODE: s is a 1-D tensor sorted from large to small; use torch.cumsum on s ** 2

In [ ]:
if energy_rank(torch.tensor([3.0, 1.0, 0.1])) is None:
    print("Fill in energy_rank above, then run this cell again.")
else:
    check("[3, 1, 0.1]: 9 of 10.01 is 89.9 percent, so two values are needed for 90 percent", energy_rank(torch.tensor([3.0, 1.0, 0.1])) == 2, "squares are 9, 1, 0.01; the first one alone gives 0.899")
    check("ten equal values need 9 for 90 percent", energy_rank(torch.ones(10)) == 9, "9 of 10 squares is exactly 90 percent: use >=, not >")
    check("one big value needs 1", energy_rank(torch.tensor([5.0, 0.0, 0.0])) == 1, "the first value already holds 100 percent")

<details><summary>Show a possible answer</summary>

```python
def energy_rank(s, frac=0.9):
    c = torch.cumsum(s ** 2, 0) / (s ** 2).sum()
    return int((c >= frac - 1e-9).nonzero()[0]) + 1

rows = []
print("%-7s %-20s %12s %12s %16s %18s" % ("layer", "update", "largest sv", "rank (>1e-4)", "90% energy at", "energy in top 8"))
for li in (0, 5, 11):
    lay = lora8_model.transformer.h[li].attn.c_attn
    dW_lora = (lay.scale * lay.A @ lay.B).detach()
    dW_full = (full_model.transformer.h[li].attn.c_attn.weight - base_model.transformer.h[li].attn.c_attn.weight).detach()
    for name, dW in (("LoRA r = 8", dW_lora), ("full fine-tuning", dW_full)):
        s = torch.linalg.svdvals(dW)
        top8 = float((s[:8] ** 2).sum() / (s ** 2).sum())
        rows.append((li, name, float(s[0]), int((s > 1e-4 * s[0]).sum()), energy_rank(s), top8))
        print("%-7d %-20s %12.3f %12d %16d %17.1f%%" % (li, name, float(s[0]), int((s > 1e-4 * s[0]).sum()), energy_rank(s), 100 * top8))
    if li == 5: s_lora5, s_full5 = torch.linalg.svdvals(dW_lora), torch.linalg.svdvals(dW_full)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.semilogy(range(1, 33), s_full5[:32], "o-", label="full fine-tuning, layer 5"); ax.semilogy(range(1, 33), s_lora5[:32].clamp_min(1e-9), "s-", label="LoRA r = 8, layer 5")
ax.set_xlabel("index of the singular value"); ax.set_ylabel("singular value"); ax.legend(); plt.tight_layout(); plt.show()
RESULTS["p6_svd"] = {"layer%d %s" % (r[0], r[1]): dict(rank=r[3], e90=r[4], top8=round(r[5], 3)) for r in rows}
assert all(r[3] <= 8 for r in rows if r[1] == "LoRA r = 8")
```

`(c >= frac - 1e-9)` finds the first position where the running share reaches `frac`; the tiny 1e-9 stops rounding errors from rejecting an exact 90 percent.

</details>

Below the exercise we run our version of the code above, so that the next cells work even if you skipped the exercise, and then the check on it.

In [ ]:
def energy_rank(s, frac=0.9):
    c = torch.cumsum(s ** 2, 0) / (s ** 2).sum()
    return int((c >= frac - 1e-9).nonzero()[0]) + 1

rows = []
print("%-7s %-20s %12s %12s %16s %18s" % ("layer", "update", "largest sv", "rank (>1e-4)", "90% energy at", "energy in top 8"))
for li in (0, 5, 11):
    lay = lora8_model.transformer.h[li].attn.c_attn
    dW_lora = (lay.scale * lay.A @ lay.B).detach()
    dW_full = (full_model.transformer.h[li].attn.c_attn.weight - base_model.transformer.h[li].attn.c_attn.weight).detach()
    for name, dW in (("LoRA r = 8", dW_lora), ("full fine-tuning", dW_full)):
        s = torch.linalg.svdvals(dW)
        top8 = float((s[:8] ** 2).sum() / (s ** 2).sum())
        rows.append((li, name, float(s[0]), int((s > 1e-4 * s[0]).sum()), energy_rank(s), top8))
        print("%-7d %-20s %12.3f %12d %16d %17.1f%%" % (li, name, float(s[0]), int((s > 1e-4 * s[0]).sum()), energy_rank(s), 100 * top8))
    if li == 5: s_lora5, s_full5 = torch.linalg.svdvals(dW_lora), torch.linalg.svdvals(dW_full)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.semilogy(range(1, 33), s_full5[:32], "o-", label="full fine-tuning, layer 5"); ax.semilogy(range(1, 33), s_lora5[:32].clamp_min(1e-9), "s-", label="LoRA r = 8, layer 5")
ax.set_xlabel("index of the singular value"); ax.set_ylabel("singular value"); ax.legend(); plt.tight_layout(); plt.show()
RESULTS["p6_svd"] = {"layer%d %s" % (r[0], r[1]): dict(rank=r[3], e90=r[4], top8=round(r[5], 3)) for r in rows}
assert all(r[3] <= 8 for r in rows if r[1] == "LoRA r = 8")

In [ ]:
if energy_rank(torch.tensor([3.0, 1.0, 0.1])) is None:
    print("Fill in energy_rank above, then run this cell again.")
else:
    check("[3, 1, 0.1]: 9 of 10.01 is 89.9 percent, so two values are needed for 90 percent", energy_rank(torch.tensor([3.0, 1.0, 0.1])) == 2, "squares are 9, 1, 0.01; the first one alone gives 0.899")
    check("ten equal values need 9 for 90 percent", energy_rank(torch.ones(10)) == 9, "9 of 10 squares is exactly 90 percent: use >=, not >")
    check("one big value needs 1", energy_rank(torch.tensor([5.0, 0.0, 0.0])) == 1, "the first value already holds 100 percent")

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The check passes three ticks. The table has six rows (layers 0, 5, 11; LoRA and full): for LoRA the rank is 8 in every layer and 90 percent of the energy sits in 4, 2 and 1 directions (layers 0, 5 and 11), with 100 percent in the top 8. For full fine-tuning the rank is 768 in every layer, 90 percent of the energy needs 89, 82 and 55 directions, and the top 8 hold only 54, 50 and 61 percent. The figure compares the first 32 singular values for layer 5 on a log scale.

*What it means.* LoRA's update is, by construction, concentrated: layer 11's has 90 percent of its energy in a *single* direction. The full fine-tuning update is **not** low rank in the strict sense (rank 768, and 55 to 89 directions needed for 90 percent), and the best 8-direction approximation would keep only about half of its energy. So the honest statement is not "fine-tuning updates are tiny": it is that a low-rank update can capture *enough* of what a task needs, as the style_rate of 1.00 and the validation loss of 0.270 (against 0.193) show, and that the full update spreads over many more directions (this experiment cannot tell how much of that is useful and how much is noise from only 24 examples). Hu et al. argued that the useful change has low *intrinsic* rank; this small experiment supports "enough", not "identical".

### Step 6.11. The size of an adapter

The last advantage is the file. To share what you taught the model you only need the adapter: the 24 tensors A and B (12 layers, two each). We save them with `torch.save` to a temporary file and measure its size, and do the same for the full model, with the arithmetic (4 bytes per float32 number) next to it.

In [ ]:
import tempfile
def file_mb(state):
    with tempfile.TemporaryDirectory() as d:
        f = os.path.join(d, "ckpt.pt"); torch.save(state, f); return os.path.getsize(f) / 1e6
adapter_state = {k: v for k, v in lora8_model.state_dict().items() if k.endswith(".A") or k.endswith(".B")}
mb_adapter = file_mb(adapter_state); mb_full = file_mb(full_model.state_dict())
print("adapter: %d tensors, %d numbers, file %.2f MB (computed 4 bytes per number: %.2f MB)" % (len(adapter_state), sum(v.numel() for v in adapter_state.values()), mb_adapter, 4 * 294912 / 1e6))
print("full checkpoint: file %.1f MB (computed %.1f MB)" % (mb_full, 4 * count_params(full_model) / 1e6))
print("the full checkpoint is %.0f times larger; ten adapters (ten tasks) take %.1f MB, ten full copies %.0f MB" % (mb_full / mb_adapter, 10 * mb_adapter, 10 * mb_full))
RESULTS["p6_adapter_mb"] = round(mb_adapter, 2); RESULTS["p6_full_mb"] = round(mb_full, 1); RESULTS["p6_size_ratio"] = round(mb_full / mb_adapter)

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The cell prints `adapter: 24 tensors, 294912 numbers, file 1.19 MB (computed 1.18 MB ...)`, `full checkpoint: file 497.8 MB (computed 497.8 MB)` and `the full checkpoint is 420 times larger; ten adapters (ten tasks) take about 12 MB, ten full copies about 4,978 MB`. The sizes match the arithmetic to within the small header of the file.

*What it means.* One school server could keep one 500 MB GPT-2 and a 1.2 MB adapter for each subject, the handbook help desk, the report-card tone, the language club, and swap them per request, instead of ten full copies. This is why LoRA, not full fine-tuning, is the default for sharing models. `RESULTS["p6_size_ratio"]` is 420. Part 7 asks you to use the numbers of this part for your own plan.

## Part 7. Plan your own project and reflect (20 min)

*Lesson sections: "Doing it well" and "Key takeaways".* You can now fine-tune, align and adapt a small language model. The hard part of a real project is the decision before the code. ==In this part you fill in a one-page design card for two scenarios from a school, decide for six situations whether a prompt, retrieval (RAG) or fine-tuning fits, build an honest table of what worked and what did not in your lab, and answer four reflection questions.== Nothing here trains a model: it is thinking, reading your own numbers and writing, and it links to your Capstone and to Week 15.

### ✏️ Imagine first

Think of something at your school that a language model could help with: answering the office's repeated questions, drafting comments, checking homework against a rubric. Before you read on, write in one sentence what the model should do and which of the three tools (a prompt, retrieval of documents, or fine-tuning) you would try first. At the end of the part you check whether the decision rule of Step 7.2 agrees.

Write your guesses in the starter cell below (replace each `None` or empty string), then run it. Do not look ahead: you compare with the real numbers later in this part.

In [ ]:
# Imagine first: the shape of your own project
my_project_task = ""      # one sentence: what should the model do for people at your school? (words)
my_project_method = ""    # prompt, RAG or fine-tune? (one of these words)

### Step 7.1. A design card for your own project

A good project starts on one page. The **design card** has six fields: the **task** (what the model does and for whom), the **data** (where it comes from, how many, who checked it), the **base model** (which one and why), the **method** (prompt, RAG, LoRA or full fine-tuning), the **metric** (a number you will measure on examples the model never saw, with the value that counts as good enough) and the **risks** (what could go wrong and who checks). Writing the card first often changes the plan: a metric you cannot measure, or data you do not have, is better discovered before you train.

### ✏️ Your turn: A design card

*Exercise 7.1:* Fill in the two cards, six fields each, with at least 8 words per field. **`card_handbook`:** a help desk that answers students' and parents' questions from the school handbook. **`card_reports`:** a tool that turns a teacher's rough notes into a report-card comment in the school's tone. Why it matters: the two scenarios look alike but need different methods. The check tests that every field is long enough, that the method fits the facts of the scenario (retrieval for the handbook, fine-tuning for the report tone) and that each metric contains a number.

Write your answer in the starter cell below, then run the check cell under it.

In [ ]:
CARD_FIELDS = ("task", "data", "base_model", "method", "metric", "risks")
card_help = {"task": "what the model does and for whom", "data": "where the examples or documents come from, how many, who checked them",
             "base_model": "which model you start from and why", "method": "prompt, RAG, LoRA or full fine-tuning, and why that one",
             "metric": "a number you will measure on held-out examples, with the value that counts as good enough", "risks": "what could go wrong, and who checks"}

card_handbook = {f: "" for f in CARD_FIELDS}     # YOUR TEXT: a help desk that answers students' questions from the school handbook
card_reports = {f: "" for f in CARD_FIELDS}      # YOUR TEXT: a tool that turns a teacher's rough notes into a report-card comment in the school's own tone

In [ ]:
def card_problems(card, must_method):
    bad = [f for f in CARD_FIELDS if len(card[f].split()) < 8]
    return bad, any(w in card["method"].lower() for w in must_method), any(ch.isdigit() for ch in card["metric"])

if all(v == "" for c in (card_handbook, card_reports) for v in c.values()):
    print("Fill in the two cards above (six fields each), then run this cell again.")
else:
    for nm, c, words in (("handbook", card_handbook, ("rag", "retriev")), ("reports", card_reports, ("lora", "fine-tun", "fine tun"))):
        bad, method_ok, metric_ok = card_problems(c, words)
        check("%s: every field has at least 8 words" % nm, not bad, "too short: " + ", ".join(bad))
        check("%s: the method is the one the facts point to (%s)" % (nm, " or ".join(words)), method_ok, "facts that change point to retrieval; a style or format that must always hold points to fine-tuning")
        check("%s: the metric contains a number" % nm, metric_ok, "for example 'at least 90 percent of 50 held-out questions answered correctly'")

<details><summary>Show a possible answer</summary>

```python
card_handbook = {
    "task": "Answer students' and parents' questions about the school handbook (uniform, timetable, rules, dates) in a polite way and always say which page the answer comes from.",
    "data": "The 60-page handbook of this year, split into paragraphs, plus 80 real questions from the office that two teachers answered, kept aside to test the tool.",
    "base_model": "A small open chat model that runs on the school server, because the questions contain names and the data must stay in the school.",
    "method": "RAG with a ready-made model: the handbook changes every year, so the facts must come from retrieval and not from the weights; a prompt fixes the tone and the rule to cite the page.",
    "metric": "Share of the 80 held-out questions answered correctly with the right page cited; good enough means at least 90 percent, and 100 percent of 'I do not know' when the handbook is silent.",
    "risks": "The tool may answer from memory when retrieval fails or quote an old rule; the office reads a sample of 20 answers each month and the tool never decides on discipline.",
}
card_reports = {
    "task": "Turn a teacher's rough notes about a student (three or four bullet points) into two polite, honest sentences in the school's own report-card tone.",
    "data": "About 600 past comments written by teachers, with the notes they started from, names removed and approved by the head of year; 100 kept aside for testing.",
    "base_model": "A small instruction-tuned model of about one billion parameters, because the task is the same every time and it must run on a school laptop.",
    "method": "LoRA fine-tuning on the 500 training pairs: the tone is a habit that should hold every time without a long prompt, and the adapter is a file of a few megabytes per subject.",
    "metric": "Two teachers rate 100 held-out comments blind (tone right, facts match the notes); good enough is at least 85 percent accepted without edits, with 0 invented facts.",
    "risks": "The model may invent praise or a fact that is not in the notes, or copy phrases from one real student; every comment is read and signed by the teacher before it is sent.",
}
for nm, c in (("handbook help desk", card_handbook), ("report-card comments", card_reports)):
    print("== %s ==" % nm)
    for f in CARD_FIELDS: print("  %-10s %s" % (f, c[f]))
```

Read the two cards side by side. The handbook changes every year and the answers must name a page, so the facts live in documents and are retrieved; a prompt sets the tone. The report tone is a *habit* that must hold every time, with hundreds of examples available, so it is a fine-tuning (LoRA) job, as in Part 6, and the adapter is a file of a few megabytes. Both cards keep a person in charge of what leaves the building.

</details>

Below the exercise we run our version of the code above, so that the next cells work even if you skipped the exercise, and then the check on it.

In [ ]:
card_handbook = {
    "task": "Answer students' and parents' questions about the school handbook (uniform, timetable, rules, dates) in a polite way and always say which page the answer comes from.",
    "data": "The 60-page handbook of this year, split into paragraphs, plus 80 real questions from the office that two teachers answered, kept aside to test the tool.",
    "base_model": "A small open chat model that runs on the school server, because the questions contain names and the data must stay in the school.",
    "method": "RAG with a ready-made model: the handbook changes every year, so the facts must come from retrieval and not from the weights; a prompt fixes the tone and the rule to cite the page.",
    "metric": "Share of the 80 held-out questions answered correctly with the right page cited; good enough means at least 90 percent, and 100 percent of 'I do not know' when the handbook is silent.",
    "risks": "The tool may answer from memory when retrieval fails or quote an old rule; the office reads a sample of 20 answers each month and the tool never decides on discipline.",
}
card_reports = {
    "task": "Turn a teacher's rough notes about a student (three or four bullet points) into two polite, honest sentences in the school's own report-card tone.",
    "data": "About 600 past comments written by teachers, with the notes they started from, names removed and approved by the head of year; 100 kept aside for testing.",
    "base_model": "A small instruction-tuned model of about one billion parameters, because the task is the same every time and it must run on a school laptop.",
    "method": "LoRA fine-tuning on the 500 training pairs: the tone is a habit that should hold every time without a long prompt, and the adapter is a file of a few megabytes per subject.",
    "metric": "Two teachers rate 100 held-out comments blind (tone right, facts match the notes); good enough is at least 85 percent accepted without edits, with 0 invented facts.",
    "risks": "The model may invent praise or a fact that is not in the notes, or copy phrases from one real student; every comment is read and signed by the teacher before it is sent.",
}
for nm, c in (("handbook help desk", card_handbook), ("report-card comments", card_reports)):
    print("== %s ==" % nm)
    for f in CARD_FIELDS: print("  %-10s %s" % (f, c[f]))

In [ ]:
def card_problems(card, must_method):
    bad = [f for f in CARD_FIELDS if len(card[f].split()) < 8]
    return bad, any(w in card["method"].lower() for w in must_method), any(ch.isdigit() for ch in card["metric"])

if all(v == "" for c in (card_handbook, card_reports) for v in c.values()):
    print("Fill in the two cards above (six fields each), then run this cell again.")
else:
    for nm, c, words in (("handbook", card_handbook, ("rag", "retriev")), ("reports", card_reports, ("lora", "fine-tun", "fine tun"))):
        bad, method_ok, metric_ok = card_problems(c, words)
        check("%s: every field has at least 8 words" % nm, not bad, "too short: " + ", ".join(bad))
        check("%s: the method is the one the facts point to (%s)" % (nm, " or ".join(words)), method_ok, "facts that change point to retrieval; a style or format that must always hold points to fine-tuning")
        check("%s: the metric contains a number" % nm, metric_ok, "for example 'at least 90 percent of 50 held-out questions answered correctly'")

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The check gives three ticks for each card. The answer prints the two cards.

*What it means.* A card is good when every field could be tested: the metric names a number and a held-out set, the data field says who checked it, the risks field names a person. Your own cards will differ; what matters is that they are specific to the scenario. The numbers on the cards (80 questions, 90 percent, 600 comments) are examples, not measurements from this course.

### Step 7.2. Prompt, RAG or fine-tune?

Three tools solve different problems. A **prompt** is the cheapest: put the instructions and a few examples in the context and try. **Retrieval (RAG)** puts the *facts* in the context at question time, so they can change without any training and the answer can cite its source. **Fine-tuning** changes the *behaviour* (style, format, a skill) so that it holds without a long prompt and in a small model. A rule that works in most cases: if the facts change, retrieve; if the behaviour must always hold and you have hundreds or thousands of good examples, fine-tune; if it is a one-off or you have little data, try a prompt first. Part 6 gives the costs of the last option: a 1 MB adapter, about 4 times less training time here, but you still need good data.

### ✏️ Your turn: Prompt, RAG or fine-tune?

*Exercise 7.2:* For each of the six situations `S1` to `S6` write in `choice` one of "prompt", "RAG" or "fine-tune". Why it matters: choosing wrongly is the most expensive mistake in a project (fine-tuning facts that change every week, or retrieving when the real need is a fixed format). The check compares your choice with our key and prints a hint for each difference; a defensible different choice is worth a sentence in your report.

Write your answer in the starter cell below, then run the check cell under it.

In [ ]:
SITUATIONS = {
    "S1": "Answer questions about this week's cafeteria menu and timetable changes, which the office updates every few days.",
    "S2": "Translate one letter to parents into three languages, once, tomorrow.",
    "S3": "Make a small model that must run on a phone always answer in the same JSON format with six fields; there are 5,000 labelled examples.",
    "S4": "Answer from 300 pages of school policies and say on which page the answer is.",
    "S5": "Reply in the warm voice of the school counsellors; there are 800 examples of their replies and the voice must hold on every answer.",
    "S6": "Check whether a student's answer matches a three-sentence rubric; there are only a few dozen answers a week.",
}
choice = {k: "" for k in SITUATIONS}      # YOUR CODE: for every situation write "prompt", "RAG" or "fine-tune"

In [ ]:
KEY = {"S1": "RAG", "S2": "prompt", "S3": "fine-tune", "S4": "RAG", "S5": "fine-tune", "S6": "prompt"}
WHY = {"S1": "the facts change every few days: put them in the context, do not bake them into weights", "S2": "one-off and no data: a good prompt is enough",
       "S3": "a fixed format learned from thousands of examples, in a small model: fine-tune", "S4": "a big document the answer must cite: retrieve the right pages",
       "S5": "a voice is a behaviour: many examples and it must always hold, so fine-tune (LoRA)", "S6": "a short rule you can paste into the prompt; try the prompt first"}
if all(v == "" for v in choice.values()):
    print("Fill in choice for the six situations above, then run this cell again.")
else:
    for k in SITUATIONS: check("%s: %s" % (k, choice[k] or "(empty)"), choice[k].strip().lower() == KEY[k].lower(), "think: do the facts change? is it a behaviour? is it one-off? " )

<details><summary>Show a possible answer</summary>

```python
choice = {"S1": "RAG", "S2": "prompt", "S3": "fine-tune", "S4": "RAG", "S5": "fine-tune", "S6": "prompt"}
for k in SITUATIONS: print("%s %-9s %s" % (k, choice[k], WHY[k]))
RESULTS["p7_choice"] = dict(choice)
```

Every row follows the rule of thumb above. Situations that really mix two needs (a counsellor's voice that also quotes the current timetable) combine the tools: fine-tune the voice and retrieve the facts. The point of the exercise is to name the reason (do the facts change? is it a behaviour? is it one-off?) before you pick.

</details>

Below the exercise we run our version of the code above, so that the next cells work even if you skipped the exercise, and then the check on it.

In [ ]:
choice = {"S1": "RAG", "S2": "prompt", "S3": "fine-tune", "S4": "RAG", "S5": "fine-tune", "S6": "prompt"}
for k in SITUATIONS: print("%s %-9s %s" % (k, choice[k], WHY[k]))
RESULTS["p7_choice"] = dict(choice)

In [ ]:
KEY = {"S1": "RAG", "S2": "prompt", "S3": "fine-tune", "S4": "RAG", "S5": "fine-tune", "S6": "prompt"}
WHY = {"S1": "the facts change every few days: put them in the context, do not bake them into weights", "S2": "one-off and no data: a good prompt is enough",
       "S3": "a fixed format learned from thousands of examples, in a small model: fine-tune", "S4": "a big document the answer must cite: retrieve the right pages",
       "S5": "a voice is a behaviour: many examples and it must always hold, so fine-tune (LoRA)", "S6": "a short rule you can paste into the prompt; try the prompt first"}
if all(v == "" for v in choice.values()):
    print("Fill in choice for the six situations above, then run this cell again.")
else:
    for k in SITUATIONS: check("%s: %s" % (k, choice[k] or "(empty)"), choice[k].strip().lower() == KEY[k].lower(), "think: do the facts change? is it a behaviour? is it one-off? " )

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

Six ticks if your choices match the key: S1 RAG, S2 prompt, S3 fine-tune, S4 RAG, S5 fine-tune, S6 prompt. The answer prints each choice with its reason.

*What it means.* If you disagree with a row, you may be right: S6, for example, could become a fine-tune if the rubric were long and thousands of answers had to be checked every day. A fair answer says what would change your choice. Compare with your Imagine-first guess at the top of this part: did you start with the cheapest tool?

### Step 7.3. Your results and the honest findings table

The most useful page of a project report is the one with the experiments that did not work. The first cell prints the Part 6 numbers that are in `RESULTS` in your session, so you can quote them (if you restarted the notebook, copy them from your report). In the exercise you write the findings table of your lab: at least four rows, each with what you **tried**, the **number** you measured and a **verdict**. You may add rows from Parts 1 to 5 (the masked loss, overfitting on 4 examples, forgetting and replay, the reward model, DPO on the toy).

In [ ]:
p6_keys = [k for k in RESULTS if k.startswith("p6_")]
if not p6_keys:
    print("RESULTS holds no Part 6 numbers in this session: run Part 6 first, or copy the numbers from your report.")
else:
    R6 = {k: RESULTS[k] for k in p6_keys}
    print("%-24s %s" % ("what", "your Part 6 number"))
    for k in ("p6_base_val", "p6_full_val", "p6_lora_val", "p6_full_sec", "p6_lora_sec", "p6_lora_share_pct", "p6_full_general", "p6_lora_general", "p6_size_ratio", "p6_merge_diff"):
        if k in R6: print("%-24s %s" % (k, R6[k]))

### ✏️ Your turn: The honest findings table

*Exercise 7.3:* Fill in `findings` with at least 4 dictionaries with the keys `tried`, `number` and `verdict`. Include at least one result that did **not** work or surprised you. Why it matters: a table of only successes teaches nothing and hides what you would do differently; the verdict column is where you practise saying "it did not". The check tests the structure, that each number field has a digit, and that at least one verdict is negative or surprised you.

Write your answer in the starter cell below, then run the check cell under it.

In [ ]:
findings = []       # YOUR CODE: at least 4 entries; each is a dictionary with the keys "tried", "number" and "verdict", for example
                    # {"tried": "LoRA r = 8 instead of full fine-tuning", "number": "val loss 0.30 against 0.25", "verdict": "worked: almost as good with 0.24 percent of the parameters"}

In [ ]:
if len(findings) == 0:
    print("Fill in findings above (at least 4 dictionaries), then run this cell again.")
else:
    check("at least 4 findings", len(findings) >= 4, "write one row for each experiment you want to remember")
    check("every row has tried, number and verdict", all(set(f) >= {"tried", "number", "verdict"} for f in findings), "three keys: tried, number, verdict")
    check("every number field contains a digit", all(any(ch.isdigit() for ch in str(f.get("number", ""))) for f in findings), "cite a measured value from your own run")
    check("at least one verdict says something did NOT work or surprised you", any(any(w in str(f.get("verdict", "")).lower() for w in ("not", "no ", "did not", "worse", "surpris", "failed", "same")) for f in findings), "an honest table has at least one negative or surprising result")

<details><summary>Show a possible answer</summary>

```python
R_ = RESULTS
findings = [
    {"tried": "LoRA r = 8 against full fine-tuning (Step 6.6)", "number": "val loss %s against %s; %s of %s numbers trained" % (R_["p6_lora_val"], R_["p6_full_val"], format(R_["p6_trainable"], ","), format(R_["p6_total"], ",")),
     "verdict": "partly: both reach style_rate 1.0, but LoRA did not match the validation loss of full fine-tuning (it is cheaper, not free)"},
    {"tried": "five ranks from 1 to 16 (Step 6.7)", "number": "val loss %s (r = 1), %s (r = 8), %s (r = 16)" % (R_["p6_sweep_val"][1], R_["p6_sweep_val"][8], R_["p6_sweep_val"][16]),
     "verdict": "surprising: r = 16 was not better than r = 8; with one seed and 8 validation answers the plateau is the finding, not the winner"},
    {"tried": "loss on the general paragraph after training (Step 6.6)", "number": "%s before, %s full, %s LoRA" % (R_["p6_base_general"], R_["p6_full_general"], R_["p6_lora_general"]),
     "verdict": "small forgetting in both; LoRA a little less, but 48 updates and one paragraph cannot prove it"},
    {"tried": "merging the adapter into the weights (Step 6.8)", "number": "largest logit difference %s; %d of 8 answers identical" % (R_["p6_merge_diff"], R_["p6_merge_same"]),
     "verdict": "worked: the merged model is the same function with no extra layers"},
    {"tried": "the SVD of the trained updates (Step 6.10)", "number": "LoRA update: 90 percent of the energy in 1 to 4 directions; full update: %d to %d directions" % (min(R_["p6_svd"]["layer%d full fine-tuning" % i]["e90"] for i in (0, 5, 11)), max(R_["p6_svd"]["layer%d full fine-tuning" % i]["e90"] for i in (0, 5, 11))),
     "verdict": "did not confirm that the full update is low rank; a low-rank update was enough for this task, not identical to the full one"},
]
print("%-52s | %-52s | %s" % ("what I tried", "what I measured", "verdict"))
for f in findings: print("%-52s | %-52s | %s" % (f["tried"], f["number"], f["verdict"]))
```

The example uses only the Part 6 numbers of your own session and keeps the verdict honest: LoRA did not match full fine-tuning on validation loss, rank 16 was not better than rank 8, forgetting differed by a small amount and the full update was not low rank. Notice that the successes (the merge, the identical peft result, the 420 times smaller file) are the ones that were *derived or checked*, and the doubtful ones are the ones that rest on one seed and eight validation examples.

</details>

Below the exercise we run our version of the code above, so that the next cells work even if you skipped the exercise, and then the check on it.

In [ ]:
R_ = RESULTS
findings = [
    {"tried": "LoRA r = 8 against full fine-tuning (Step 6.6)", "number": "val loss %s against %s; %s of %s numbers trained" % (R_["p6_lora_val"], R_["p6_full_val"], format(R_["p6_trainable"], ","), format(R_["p6_total"], ",")),
     "verdict": "partly: both reach style_rate 1.0, but LoRA did not match the validation loss of full fine-tuning (it is cheaper, not free)"},
    {"tried": "five ranks from 1 to 16 (Step 6.7)", "number": "val loss %s (r = 1), %s (r = 8), %s (r = 16)" % (R_["p6_sweep_val"][1], R_["p6_sweep_val"][8], R_["p6_sweep_val"][16]),
     "verdict": "surprising: r = 16 was not better than r = 8; with one seed and 8 validation answers the plateau is the finding, not the winner"},
    {"tried": "loss on the general paragraph after training (Step 6.6)", "number": "%s before, %s full, %s LoRA" % (R_["p6_base_general"], R_["p6_full_general"], R_["p6_lora_general"]),
     "verdict": "small forgetting in both; LoRA a little less, but 48 updates and one paragraph cannot prove it"},
    {"tried": "merging the adapter into the weights (Step 6.8)", "number": "largest logit difference %s; %d of 8 answers identical" % (R_["p6_merge_diff"], R_["p6_merge_same"]),
     "verdict": "worked: the merged model is the same function with no extra layers"},
    {"tried": "the SVD of the trained updates (Step 6.10)", "number": "LoRA update: 90 percent of the energy in 1 to 4 directions; full update: %d to %d directions" % (min(R_["p6_svd"]["layer%d full fine-tuning" % i]["e90"] for i in (0, 5, 11)), max(R_["p6_svd"]["layer%d full fine-tuning" % i]["e90"] for i in (0, 5, 11))),
     "verdict": "did not confirm that the full update is low rank; a low-rank update was enough for this task, not identical to the full one"},
]
print("%-52s | %-52s | %s" % ("what I tried", "what I measured", "verdict"))
for f in findings: print("%-52s | %-52s | %s" % (f["tried"], f["number"], f["verdict"]))

In [ ]:
if len(findings) == 0:
    print("Fill in findings above (at least 4 dictionaries), then run this cell again.")
else:
    check("at least 4 findings", len(findings) >= 4, "write one row for each experiment you want to remember")
    check("every row has tried, number and verdict", all(set(f) >= {"tried", "number", "verdict"} for f in findings), "three keys: tried, number, verdict")
    check("every number field contains a digit", all(any(ch.isdigit() for ch in str(f.get("number", ""))) for f in findings), "cite a measured value from your own run")
    check("at least one verdict says something did NOT work or surprised you", any(any(w in str(f.get("verdict", "")).lower() for w in ("not", "no ", "did not", "worse", "surpris", "failed", "same")) for f in findings), "an honest table has at least one negative or surprising result")

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The first cell lists ten Part 6 numbers, for example `p6_full_val 0.193`, `p6_lora_val 0.27`, `p6_lora_share_pct 0.237`, `p6_size_ratio 420` and `p6_merge_diff 0.000214`. The check passes four ticks, and the answer prints a five-row table.

*What it means.* Here is what a reader should take from the table: the things you can *prove* (the parameter count, the exact merge, the agreement with peft, the file size) are solid; the things that are *measured once on tiny data* (which rank is best, how much each method forgets) are suggestions. Say so in your report. This is also how a good model card is written.

### Step 7.4. Reflect, and what comes next

Last, four questions that connect the weeks. Answer each in two or three sentences, in your own words, with at least one number from this lab. They are the same questions as the written part of the assessed deliverable, in short form. **q1** What are the stages of building a chat model, and what does each one need? **q2** Why is LoRA cheap, and what does it cost you? **q3** For a project at your school, would you use a prompt, RAG or fine-tuning, and why? **q4** Name one thing in this lab that did not work as you expected and what you learned from it.

### ✏️ Your turn: Reflect

*Exercise 7.4:* Answer in `reflection` (four strings), 25 words or more each, each with at least one number from your own runs. Why it matters: a number in the answer shows that the claim comes from something you measured. The check tests the length and the presence of a digit, not the content.

Write your answer in the starter cell below, then run the check cell under it.

In [ ]:
reflection = {"q1": "", "q2": "", "q3": "", "q4": ""}    # YOUR TEXT: two or three sentences each, and at least one number from this lab in each answer

In [ ]:
if all(v.strip() == "" for v in reflection.values()):
    print("Write your four answers in reflection above, then run this cell again.")
else:
    for k, v in reflection.items():
        check("%s: at least 25 words and at least one number" % k, len(v.split()) >= 25 and any(ch.isdigit() for ch in v), "write two or three sentences and cite a number from the lab")

<details><summary>Show a possible answer</summary>

```python
reflection = {
    "q1": 'The stages are pre-training (next-token prediction on trillions of tokens, the most expensive stage), instruction tuning on pairs like our 24 examples (a base model with style_rate 0.0 reached 1.0 after two epochs), and preference tuning with comparisons. Each needs different data: raw text, demonstrations, and judgements.',
    "q2": 'LoRA trains only 294912 numbers (0.237 percent of GPT-2), keeps 2.36 MB of Adam state instead of 995.5 MB and saves a 1.19 MB file, but its validation loss was 0.27 against 0.193 for full fine-tuning.',
    "q3": 'For the handbook help desk I would use RAG with a prompt, because the handbook changes every year and answers must cite a page; for the report-card tone I would fine-tune with LoRA on about 500 past comments, because a tone must hold every time and the adapter is only about 1.19 MB.',
    "q4": 'I expected rank 16 to beat rank 8, but its validation loss was 0.31 against 0.27. I learned that with 24 examples, 8 validation answers and one seed, small differences are noise, and that I should report the shape of the curve and not the winner.',
}
for k, v in reflection.items(): print(k, len(v.split()), "words")
```

These are examples built from the numbers of Part 6; yours should use your own words and your own results from Parts 1 to 6.

</details>

Below the exercise we run our version of the code above, so that the next cells work even if you skipped the exercise, and then the check on it.

In [ ]:
reflection = {
    "q1": 'The stages are pre-training (next-token prediction on trillions of tokens, the most expensive stage), instruction tuning on pairs like our 24 examples (a base model with style_rate 0.0 reached 1.0 after two epochs), and preference tuning with comparisons. Each needs different data: raw text, demonstrations, and judgements.',
    "q2": 'LoRA trains only 294912 numbers (0.237 percent of GPT-2), keeps 2.36 MB of Adam state instead of 995.5 MB and saves a 1.19 MB file, but its validation loss was 0.27 against 0.193 for full fine-tuning.',
    "q3": 'For the handbook help desk I would use RAG with a prompt, because the handbook changes every year and answers must cite a page; for the report-card tone I would fine-tune with LoRA on about 500 past comments, because a tone must hold every time and the adapter is only about 1.19 MB.',
    "q4": 'I expected rank 16 to beat rank 8, but its validation loss was 0.31 against 0.27. I learned that with 24 examples, 8 validation answers and one seed, small differences are noise, and that I should report the shape of the curve and not the winner.',
}
for k, v in reflection.items(): print(k, len(v.split()), "words")

In [ ]:
if all(v.strip() == "" for v in reflection.values()):
    print("Write your four answers in reflection above, then run this cell again.")
else:
    for k, v in reflection.items():
        check("%s: at least 25 words and at least one number" % k, len(v.split()) >= 25 and any(ch.isdigit() for ch in v), "write two or three sentences and cite a number from the lab")

**What you should see** (all numbers for `SEED = 42`; training times differ between machines, and the last digit of a loss can differ a little):

The check passes four ticks, and the answer prints the word count of each example answer.

*What it means.* **What comes next.** Your design card and your findings table are the start of the **Capstone** (the Structural Inspection Assistant, in the Capstone page of the course app): ask for each part of the pipeline, "prompt, retrieval or fine-tune, and how will I measure it on photos and notes the model has never seen?", and remember that a licensed engineer makes every pass or fail decision. In **Week 15 (Pre-final exam and capstone proposal)** you review Weeks 11 to 14 and write the proposal for your own project: the design card of this part is the template for it. The numbers you need to remember from this week: 6ND for pre-training, the Bradley-Terry probability and loss, the KL-regularised optimum, the DPO loss, r(d_in + d_out) for LoRA, and the memory arithmetic of Step 6.5.